In [3]:

import requests
from bs4 import BeautifulSoup
import json, time, re

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36",
    "Accept-Language": "pl-PL,pl;q=0.9",
}

BASE = "https://www.bechcicki.pl"

def get(url, retries=3):
    for i in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=15)
            r.raise_for_status()
            return r
        except Exception as e:
            if i == retries-1: raise
            time.sleep(2)

# Pobierz stronę główną i wyciągnij L1 z nawigacji
r = get(BASE)
soup = BeautifulSoup(r.text, "html.parser")

# Szukaj nawigacji — próbuj różne selektory
nav_links = []

# Próba 1: główna nawigacja kategorii
for sel in ["nav a", ".nav a", ".navigation a", "#nav a", "[class*='categor'] a", "[class*='menu'] a"]:
    links = soup.select(sel)
    if links:
        print(f"Selektor '{sel}': {len(links)} linków")
        break

# Wyciągnij wszystkie linki z href zawierające ścieżki kategorii
all_links = soup.find_all("a", href=True)
cat_links = [a for a in all_links if re.search(r'/(c-|category|kategori|dzia[lł])', a.get("href",""), re.I)]
print(f"\nPotencjalne linki kategorii: {len(cat_links)}")
for l in cat_links[:20]:
    print(f"  {l.get_text(strip=True)[:50]} → {l['href']}")


Selektor 'nav a': 381 linków

Potencjalne linki kategorii: 0


In [7]:

# Zobaczmy strukturę nawigacji
nav = soup.find("nav")
if nav:
    # Pokaż pierwsze linki z nawigacji
    links = nav.find_all("a", href=True)[:30]
    for l in links:
        print(f"{l.get_text(strip=True)[:60]:<60} {l['href'][:80]}")
else:
    print("Brak <nav>")
    # Sprawdź menu główne
    for cls in ["menu", "navigation", "header", "topbar", "categories"]:
        els = soup.find_all(class_=lambda x: x and cls in x.lower() if x else False)
        if els:
            print(f"Klasa '{cls}': {len(els)} elementów")


Farby i rozpuszczalniki                                      /farby-i-rozpuszczalniki
Farby wewnętrzne                                             /farby-i-rozpuszczalniki/farby-wewnetrzne
Farby wewnętrzne białe                                       /farby-i-rozpuszczalniki/farby-wewnetrzne/farby-wewnetrzne-biale
Farby wewnętrzne kolorowe                                    /farby-i-rozpuszczalniki/farby-wewnetrzne/farby-wewnetrzne-kolorowe
Farby elewacyjne                                             /farby-i-rozpuszczalniki/farby-elewacyjne
Farby elewacyjne silikonowe                                  /farby-i-rozpuszczalniki/farby-elewacyjne/farby-elewacyjne-silikonowe
Farby elewacyjne emulsyjne                                   /farby-i-rozpuszczalniki/farby-elewacyjne/farby-elewacyjne-emulsyjne
Farby elewacyjne akrylowe                                    /farby-i-rozpuszczalniki/farby-elewacyjne/farby-elewacyjne-akrylowe
Farby elewacyjne silikatowe                                  /f

In [11]:

from collections import defaultdict

nav = soup.find("nav")
all_nav_links = nav.find_all("a", href=True) if nav else []

# Filtruj: tylko linki kategorii (nie produkty, nie strony)
cat_links_raw = []
for a in all_nav_links:
    href = a["href"].strip()
    text = a.get_text(strip=True)
    # Pomiń: home, konto, koszyk, "Pokaż wszystkie", puste
    if not href or href in ["/", "#"] or "Pokaż" in text or len(text) < 2:
        continue
    if any(skip in href for skip in ["/produkty", "/p-", "?", "#", "konto", "koszyk", "/blog", "/kontakt", "/o-nas"]):
        continue
    # Liczy poziomy po '/'
    parts = [p for p in href.strip("/").split("/") if p]
    if 1 <= len(parts) <= 4:
        cat_links_raw.append({"href": href, "name": text, "depth": len(parts), "parts": parts})

# Deduplikacja po href
seen = set()
cats = []
for c in cat_links_raw:
    if c["href"] not in seen:
        seen.add(c["href"])
        cats.append(c)

# Zbuduj drzewo
tree = {}
for c in cats:
    parts = c["parts"]
    d = c["depth"]
    if d == 1:
        if parts[0] not in tree:
            tree[parts[0]] = {"name": c["name"], "slug": parts[0], "children": {}}
    elif d == 2:
        p1 = parts[0]
        if p1 not in tree:
            tree[p1] = {"name": p1, "slug": p1, "children": {}}
        tree[p1]["children"][parts[1]] = {"name": c["name"], "slug": parts[1], "children": {}}
    elif d == 3:
        p1, p2, p3 = parts
        if p1 not in tree:
            tree[p1] = {"name": p1, "slug": p1, "children": {}}
        if p2 not in tree[p1]["children"]:
            tree[p1]["children"][p2] = {"name": p2, "slug": p2, "children": {}}
        tree[p1]["children"][p2]["children"][p3] = {"name": c["name"], "slug": p3, "children": {}}
    elif d == 4:
        p1, p2, p3, p4 = parts
        if p1 not in tree: tree[p1] = {"name": p1, "slug": p1, "children": {}}
        if p2 not in tree[p1]["children"]: tree[p1]["children"][p2] = {"name": p2, "slug": p2, "children": {}}
        if p3 not in tree[p1]["children"][p2]["children"]: tree[p1]["children"][p2]["children"][p3] = {"name": p3, "slug": p3, "children": {}}
        tree[p1]["children"][p2]["children"][p3]["children"][p4] = {"name": c["name"], "slug": p4, "children": {}}

# Wyświetl strukturę
def print_tree(node, prefix="", level=1):
    children = node.get("children", {})
    for slug, child in children.items():
        n = child["name"][:55]
        print(f"{'  '*level}L{level}: {n}")
        if child.get("children"):
            print_tree(child, prefix + "  ", level+1)

print(f"Znaleziono {len(tree)} kategorii L1\n")
for slug, l1 in tree.items():
    print(f"L1: {l1['name']}")
    print_tree(l1, level=2)
    print()


Znaleziono 11 kategorii L1

L1: Farby i rozpuszczalniki
    L2: Farby wewnętrzne
      L3: Farby wewnętrzne białe
      L3: Farby wewnętrzne kolorowe
    L2: Farby elewacyjne
      L3: Farby elewacyjne silikonowe
      L3: Farby elewacyjne emulsyjne
      L3: Farby elewacyjne akrylowe
      L3: Farby elewacyjne silikatowe
      L3: Farby elewacyjne specjalne
    L2: Farby do drewna
      L3: Lakiery do drewna
      L3: Lakierobejce
      L3: Oleje
      L3: Impregnaty
      L3: Podkłady wypełniające
    L2: Farby do metalu
      L3: Emalie chlorokauczukowe
      L3: Emalie ftalowe
      L3: Emalie poliuretanowe
      L3: Emalie akrylowe
      L3: Lakiery do metalu
    L2: Bazy i koloranty
    L2: Farby specjalistyczne
      L3: Farby przemysłowe
      L3: Farby proszkowe
    L2: Farby pozostałe
      L3: Farby do betonu
      L3: Pigmenty
      L3: Farby zaprawkowe
      L3: Spraye
    L2: Rozpuszczalniki
    L2: Preparaty do chemicznego oczyszczania powierzchni

L1: Chemia budowlana
 

In [15]:

import re

def slugify(text):
    """Polskie znaki → ASCII, spacje → myślniki"""
    replacements = {'ą':'a','ć':'c','ę':'e','ł':'l','ń':'n','ó':'o','ś':'s','ź':'z','ż':'z',
                    'Ą':'a','Ć':'c','Ę':'e','Ł':'l','Ń':'n','Ó':'o','Ś':'s','Ź':'z','Ż':'z'}
    t = text.lower()
    for k,v in replacements.items(): t = t.replace(k,v)
    t = re.sub(r'[^a-z0-9]+', '-', t)
    return t.strip('-')

# Czyste drzewo bechcicki.pl (bez "https:" fake-entry)
BECHCICKI_TREE = [
  {
    "name": "Farby i rozpuszczalniki",
    "slug": "farby-i-rozpuszczalniki",
    "children": [
      {"name": "Farby wewnętrzne", "slug": "farby-wewnetrzne",
       "children": [
         {"name": "Farby wewnętrzne białe", "slug": "farby-wewnetrzne-biale"},
         {"name": "Farby wewnętrzne kolorowe", "slug": "farby-wewnetrzne-kolorowe"},
       ]},
      {"name": "Farby elewacyjne", "slug": "farby-elewacyjne",
       "children": [
         {"name": "Farby elewacyjne silikonowe", "slug": "farby-elewacyjne-silikonowe"},
         {"name": "Farby elewacyjne emulsyjne", "slug": "farby-elewacyjne-emulsyjne"},
         {"name": "Farby elewacyjne akrylowe", "slug": "farby-elewacyjne-akrylowe"},
         {"name": "Farby elewacyjne silikatowe", "slug": "farby-elewacyjne-silikatowe"},
         {"name": "Farby elewacyjne specjalne", "slug": "farby-elewacyjne-specjalne"},
       ]},
      {"name": "Farby do drewna", "slug": "farby-do-drewna",
       "children": [
         {"name": "Lakiery do drewna", "slug": "lakiery-do-drewna"},
         {"name": "Lakierobejce", "slug": "lakierobejce"},
         {"name": "Oleje do drewna", "slug": "oleje-do-drewna"},
         {"name": "Impregnaty do drewna", "slug": "impregnaty-do-drewna"},
         {"name": "Podkłady wypełniające", "slug": "podklady-wypelniajace"},
       ]},
      {"name": "Farby do metalu", "slug": "farby-do-metalu",
       "children": [
         {"name": "Emalie chlorokauczukowe", "slug": "emalie-chlorokauczukowe"},
         {"name": "Emalie ftalowe", "slug": "emalie-ftalowe"},
         {"name": "Emalie poliuretanowe", "slug": "emalie-poliuretanowe"},
         {"name": "Emalie akrylowe", "slug": "emalie-akrylowe"},
         {"name": "Lakiery do metalu", "slug": "lakiery-do-metalu"},
       ]},
      {"name": "Bazy i koloranty", "slug": "bazy-i-koloranty"},
      {"name": "Farby specjalistyczne", "slug": "farby-specjalistyczne",
       "children": [
         {"name": "Farby przemysłowe", "slug": "farby-przemyslowe"},
         {"name": "Farby proszkowe", "slug": "farby-proszkowe"},
       ]},
      {"name": "Farby pozostałe", "slug": "farby-pozostale",
       "children": [
         {"name": "Farby do betonu", "slug": "farby-do-betonu"},
         {"name": "Pigmenty", "slug": "pigmenty"},
         {"name": "Farby zaprawkowe", "slug": "farby-zaprawkowe"},
         {"name": "Spraye budowlane", "slug": "spraye-budowlane"},
       ]},
      {"name": "Rozpuszczalniki", "slug": "rozpuszczalniki"},
      {"name": "Preparaty czyszczące powierzchnie", "slug": "preparaty-czyszczace-powierzchnie"},
    ]
  },
  {
    "name": "Chemia budowlana",
    "slug": "chemia-budowlana",
    "children": [
      {"name": "Tynki", "slug": "tynki",
       "children": [
         {"name": "Tynki elewacyjne", "slug": "tynki-elewacyjne"},
         {"name": "Tynki cementowo-wapienne", "slug": "tynki-cementowo-wapienne"},
         {"name": "Tynki gipsowe", "slug": "tynki-gipsowe"},
         {"name": "Tynki wapienne", "slug": "tynki-wapienne"},
         {"name": "Tynki specjalne", "slug": "tynki-specjalne"},
       ]},
      {"name": "Kleje", "slug": "kleje",
       "children": [
         {"name": "Kleje montażowe", "slug": "kleje-montazowe"},
         {"name": "Kleje do glazury", "slug": "kleje-do-glazury"},
         {"name": "Kleje do drewna", "slug": "kleje-do-drewna"},
         {"name": "Kleje do styropianu i styroduru", "slug": "kleje-do-styropianu-i-styroduru"},
         {"name": "Kleje do wełen", "slug": "kleje-do-welen"},
       ]},
      {"name": "Gipsy i gładzie", "slug": "gipsy-i-gladzie",
       "children": [
         {"name": "Gładzie gipsowe w proszku", "slug": "gladzie-gipsowe-w-proszku"},
         {"name": "Gipsy szpachlowe", "slug": "gipsy-szpachlowe"},
         {"name": "Gładzie masy gotowe", "slug": "gladzie-masy-gotowe"},
         {"name": "Kleje gipsowe", "slug": "kleje-gipsowe"},
         {"name": "Gipsy budowlane", "slug": "gipsy-budowlane"},
       ]},
      {"name": "Grunty", "slug": "grunty",
       "children": [
         {"name": "Grunty uniwersalne", "slug": "grunty-uniwersalne"},
         {"name": "Masy bitumiczne gruntujące", "slug": "masy-bitumiczne-gruntujace"},
         {"name": "Grunty specjalistyczne", "slug": "grunty-specjalistyczne"},
         {"name": "Grunty do posadzek", "slug": "grunty-do-posadzek"},
         {"name": "Grunty pod tynki", "slug": "grunty-pod-tynki"},
       ]},
      {"name": "Piany montażowe", "slug": "piany-montazowe",
       "children": [
         {"name": "Piany montażowe pistoletowe", "slug": "piany-montazowe-pistoletowe"},
         {"name": "Piany montażowe wężykowe", "slug": "piany-montazowe-wezzykowe"},
         {"name": "Czyściki do pian montażowych", "slug": "czysciki-do-pian-montazowych"},
       ]},
      {"name": "Uszczelniacze i silikony", "slug": "uszczelniacze-i-silikony",
       "children": [
         {"name": "Silikony sanitarne", "slug": "silikony-sanitarne"},
         {"name": "Silikony uniwersalne", "slug": "silikony-uniwersalne"},
         {"name": "Silikony wysokotemperaturowe", "slug": "silikony-wysokotemperaturowe"},
         {"name": "Silikony dekarskie", "slug": "silikony-dekarskie"},
         {"name": "Silikony szklarskie", "slug": "silikony-szklarskie"},
       ]},
      {"name": "Zaprawy", "slug": "zaprawy",
       "children": [
         {"name": "Zaprawy specjalistyczne", "slug": "zaprawy-specjalistyczne"},
         {"name": "Zaprawy naprawcze", "slug": "zaprawy-naprawcze"},
         {"name": "Zaprawy uszczelniające", "slug": "zaprawy-uszczelniajace"},
         {"name": "Masy samopoziomujące", "slug": "masy-samopoziomujace"},
         {"name": "Zaprawy murarskie", "slug": "zaprawy-murarskie"},
       ]},
      {"name": "Spoiny", "slug": "spoiny",
       "children": [
         {"name": "Spoiny elastyczne", "slug": "spoiny-elastyczne"},
         {"name": "Spoiny specjalistyczne", "slug": "spoiny-specjalistyczne"},
         {"name": "Zaprawy do spoin", "slug": "zaprawy-do-spoin"},
         {"name": "Spoiny zwykłe", "slug": "spoiny-zwykle"},
       ]},
      {"name": "Powłoki epoksydowe", "slug": "powloki-epoksydowe"},
      {"name": "Kotwy chemiczne", "slug": "kotwy-chemiczne"},
      {"name": "Środki grzybobójcze", "slug": "srodki-grzybobojcze"},
      {"name": "Środki czyszcząco-pielęgnacyjne", "slug": "srodki-czyszczaco-pielegnacyjne"},
      {"name": "Dodatki do zapraw i betonu", "slug": "dodatki-do-zapraw-i-betonu"},
    ]
  },
  {
    "name": "Sucha zabudowa",
    "slug": "sucha-zabudowa",
    "children": [
      {"name": "Płyty do suchej zabudowy", "slug": "plyty-do-suchej-zabudowy",
       "children": [
         {"name": "Płyty cementowe", "slug": "plyty-cementowe"},
         {"name": "Płyty gipsowo-kartonowe", "slug": "plyty-gipsowo-kartonowe"},
         {"name": "Płyty gipsowo-włóknowe", "slug": "plyty-gipsowo-wloknowe"},
         {"name": "Płyty cementowo-włóknowe", "slug": "plyty-cementowo-wloknowe"},
         {"name": "Płyty drewnopochodne", "slug": "plyty-drewnopochodne"},
       ]},
      {"name": "Profile do suchej zabudowy", "slug": "profile-do-suchej-zabudowy",
       "children": [
         {"name": "Profile ścienne", "slug": "profile-scienne"},
         {"name": "Profile ościeżnicowe", "slug": "profile-oscieznicowe"},
         {"name": "Profile sufitowe", "slug": "profile-sufitowe"},
       ]},
      {"name": "Wieszaki do suchej zabudowy", "slug": "wieszaki-do-suchej-zabudowy",
       "children": [
         {"name": "Wieszaki z noniuszem", "slug": "wieszaki-z-noniuszem"},
         {"name": "Wieszaki ES", "slug": "wieszaki-es"},
         {"name": "Wieszaki do poddaszy", "slug": "wieszaki-do-poddaszy"},
         {"name": "Wieszaki bezpośrednie", "slug": "wieszaki-bezposrednie"},
         {"name": "Wieszaki obrotowe", "slug": "wieszaki-obrotowe"},
       ]},
      {"name": "Mocowania do suchej zabudowy", "slug": "mocowania-do-suchej-zabudowy",
       "children": [
         {"name": "Wkręty do suchej zabudowy", "slug": "wkrety-do-suchej-zabudowy"},
         {"name": "Łączniki do profili", "slug": "laczniki-do-profili"},
         {"name": "Kątowniki do profili", "slug": "katowniki-do-profili"},
         {"name": "Kołki do suchej zabudowy", "slug": "kolki-do-suchej-zabudowy"},
       ]},
      {"name": "Narożniki i listwy", "slug": "narozniki-i-listwy",
       "children": [
         {"name": "Narożniki aluminiowe", "slug": "narozniki-aluminiowe"},
         {"name": "Listwy krawędziowe", "slug": "listwy-krawedzi"},
         {"name": "Narożniki PVC", "slug": "narozniki-pvc"},
         {"name": "Narożniki do tynków mokrych", "slug": "narozniki-do-tynkow-mokrych"},
         {"name": "Narożniki elastyczne", "slug": "narozniki-elastyczne"},
       ]},
      {"name": "Taśmy do suchej zabudowy", "slug": "tasmy-do-suchej-zabudowy"},
      {"name": "Rewizje i klapy", "slug": "rewizje-i-klapy",
       "children": [
         {"name": "Klapy rewizyjne", "slug": "klapy-rewizyjne"},
         {"name": "Drzwiczki rewizyjne", "slug": "drzwiczki-rewizyjne"},
       ]},
    ]
  },
  {
    "name": "Sufity podwieszane",
    "slug": "sufity-podwieszane",
    "children": [
      {"name": "Płyty sufitowe", "slug": "plyty-sufitowe",
       "children": [
         {"name": "Sufity rastrowe", "slug": "sufity-rastrowe"},
         {"name": "Płyty sufitowe z wełny mineralnej", "slug": "plyty-sufitowe-welna-mineralna"},
         {"name": "Płyty sufitowe z wełny szklanej", "slug": "plyty-sufitowe-welna-szklana"},
         {"name": "Płyty sufitowe drewniane", "slug": "plyty-sufitowe-drewniane"},
         {"name": "Płyty sufitowe metalowe", "slug": "plyty-sufitowe-metalowe"},
       ]},
      {"name": "Profile do sufitów podwieszanych", "slug": "profile-do-sufitow-podwieszanych",
       "children": [
         {"name": "Profile nośne główne", "slug": "profile-nosne-glowne"},
         {"name": "Profile poprzeczne", "slug": "profile-poprzeczne"},
         {"name": "Profile przyścienne", "slug": "profile-przyscienne"},
         {"name": "Profile specjalne sufitowe", "slug": "profile-specjalne-sufitowe"},
       ]},
      {"name": "Mocowania do sufitów podwieszanych", "slug": "mocowania-do-sufitow-podwieszanych",
       "children": [
         {"name": "Elementy wykończeniowe sufitów", "slug": "elementy-wykonczeniowe-sufitow"},
         {"name": "Wieszaki dwuhakowe", "slug": "wieszaki-dwuhakowe"},
         {"name": "Klipsy mocujące sufitów", "slug": "klipsy-mocujace-sufitow"},
         {"name": "Druty z hakiem", "slug": "druty-z-hakiem"},
         {"name": "Druty z oczkiem", "slug": "druty-z-oczkiem"},
       ]},
    ]
  },
  {
    "name": "Izolacje",
    "slug": "izolacje",
    "children": [
      {"name": "Styropiany", "slug": "styropiany",
       "children": [
         {"name": "Styropiany fasadowe EPS", "slug": "styropiany-fasadowe-eps"},
         {"name": "Styropian dach-podłoga EPS", "slug": "styropian-dach-podloga-eps"},
         {"name": "Styropiany akustyczne", "slug": "styropiany-akustyczne"},
         {"name": "Styropiany do fundamentów", "slug": "styropiany-do-fundamentow"},
       ]},
      {"name": "Płyty XPS", "slug": "plyty-xps"},
      {"name": "Wełny izolacyjne", "slug": "welny-izolacyjne",
       "children": [
         {"name": "Wełny do ścian działowych", "slug": "welny-do-scian-dzialowych"},
         {"name": "Wełny fasadowe", "slug": "welny-fasadowe"},
         {"name": "Wełny do stropów i podłóg", "slug": "welny-do-stropow-i-podlog"},
         {"name": "Wełny do dachów płaskich", "slug": "welny-do-dachow-plaskich"},
         {"name": "Wełny do poddaszy", "slug": "welny-do-poddaszy"},
       ]},
      {"name": "Izolacje budowlane", "slug": "izolacje-budowlane",
       "children": [
         {"name": "Izolacje fasadowe", "slug": "izolacje-fasadowe"},
         {"name": "Izolacje stropów i podłóg", "slug": "izolacje-stropow-i-podlog"},
         {"name": "Izolacje dachów płaskich", "slug": "izolacje-dachow-plaskich"},
         {"name": "Izolacje fundamentów", "slug": "izolacje-fundamentow"},
         {"name": "Izolacje akustyczne", "slug": "izolacje-akustyczne"},
       ]},
      {"name": "Izolacje techniczne", "slug": "izolacje-techniczne",
       "children": [
         {"name": "Izolacje HVAC", "slug": "izolacje-hvac"},
         {"name": "Izolacje przemysłowe", "slug": "izolacje-przemyslowe"},
       ]},
      {"name": "Hydroizolacje", "slug": "hydroizolacje",
       "children": [
         {"name": "Papy hydroizolacyjne", "slug": "papy-hydroizolacyjne"},
         {"name": "Hydroizolacje bitumiczne", "slug": "hydroizolacje-bitumiczne"},
         {"name": "Taśmy uszczelniające do hydroizolacji", "slug": "tasmy-uszczelniajace-hydroizolacje"},
         {"name": "Membrany dachowe", "slug": "membrany-dachowe"},
         {"name": "Hydroizolacje mineralne", "slug": "hydroizolacje-mineralne"},
       ]},
      {"name": "Folie izolacyjne", "slug": "folie-izolacyjne",
       "children": [
         {"name": "Folie fundamentowe", "slug": "folie-fundamentowe"},
         {"name": "Folie paroprzepuszczalne", "slug": "folie-paroprzepuszczalne"},
         {"name": "Folie paroizolacyjne", "slug": "folie-paroizolacyjne"},
         {"name": "Folie budowlane", "slug": "folie-budowlane"},
         {"name": "Taśmy do łączenia folii", "slug": "tasmy-do-laczenia-folii"},
       ]},
      {"name": "Akcesoria do izolacji", "slug": "akcesoria-do-izolacji",
       "children": [
         {"name": "Łączniki do izolacji fasadowych", "slug": "laczniki-do-izolacji-fasadowych"},
         {"name": "Opaski zaciskowe", "slug": "opaski-zaciskowe"},
         {"name": "Talerze dociskowe", "slug": "talerze-dociskowe"},
         {"name": "Siatki elewacyjne", "slug": "siatki-elewacyjne"},
         {"name": "Profile i narożniki do izolacji", "slug": "profile-i-narozniki-do-izolacji"},
       ]},
    ]
  },
  {
    "name": "Płytki",
    "slug": "plytki",
    "children": [
      {"name": "Płytki ceramiczne", "slug": "plytki-ceramiczne",
       "children": [
         {"name": "Płytki ścienne", "slug": "plytki-scienne"},
         {"name": "Płytki ścienno-podłogowe", "slug": "plytki-scienno-podlogowe"},
         {"name": "Płytki elewacyjne", "slug": "plytki-elewacyjne"},
         {"name": "Płytki tarasowe", "slug": "plytki-tarasowe"},
         {"name": "Stopnice", "slug": "stopnice"},
       ]},
      {"name": "Płytki dekoracyjne", "slug": "plytki-dekoracyjne",
       "children": [
         {"name": "Panele i dekory ścienne", "slug": "panele-i-dekory-scienne"},
       ]},
      {"name": "Listwy i akcesoria płytkarskie", "slug": "listwy-i-akcesoria-plytkarskie",
       "children": [
         {"name": "Listwy przypodłogowe", "slug": "listwy-przypodlogowe"},
         {"name": "Akcesoria instalacyjne płytki", "slug": "akcesoria-instalacyjne-plytki"},
       ]},
    ]
  },
  {
    "name": "Dachy",
    "slug": "dachy",
    "children": [
      {"name": "Pokrycia dachowe", "slug": "pokrycia-dachowe",
       "children": [
         {"name": "Dachówki ceramiczne", "slug": "dachowki-ceramiczne"},
         {"name": "Dachówki betonowe", "slug": "dachowki-betonowe"},
         {"name": "Pokrycia dachowe z blachy", "slug": "pokrycia-dachowe-z-blachy"},
         {"name": "Gonty bitumiczne", "slug": "gonty-bitumiczne"},
         {"name": "Papy dachowe", "slug": "papy-dachowe"},
       ]},
      {"name": "Okna dachowe i akcesoria", "slug": "okna-dachowe-i-akcesoria",
       "children": [
         {"name": "Okna dachowe", "slug": "okna-dachowe"},
         {"name": "Okna wyłazowe", "slug": "okna-wylazowe"},
         {"name": "Balkony dachowe", "slug": "balkony-dachowe"},
         {"name": "Okna kolankowe", "slug": "okna-kolankowe"},
         {"name": "Okna oddymiające", "slug": "okna-oddymiajace"},
       ]},
      {"name": "Rynny i systemy rynnowe", "slug": "rynny-i-systemy-rynnowe",
       "children": [
         {"name": "Rynny z blachy powlekanej", "slug": "rynny-z-blachy-powlekanej"},
         {"name": "Rynny PVC", "slug": "rynny-pvc"},
         {"name": "Rynny ocynkowane", "slug": "rynny-ocynkowane"},
         {"name": "Akcesoria do rynien", "slug": "akcesoria-do-rynien"},
       ]},
      {"name": "Zamocowania dachowe", "slug": "zamocowania-dachowe",
       "children": [
         {"name": "Klamry do gąsiorów", "slug": "klamry-do-gasiorow"},
         {"name": "Łączniki teleskopowe dachowe", "slug": "laczniki-teleskopowe-dachowe"},
         {"name": "Spinki do dachówek", "slug": "spinki-do-dachowek"},
         {"name": "Gwoździe i podkładki dachowe", "slug": "gwozdzie-i-podkladki-dachowe"},
       ]},
      {"name": "Komunikacja dachowa", "slug": "komunikacja-dachowa",
       "children": [
         {"name": "Ławy kominiarskie", "slug": "lawy-kominiarskie"},
         {"name": "Wsporniki do ław kominiarskich", "slug": "wsporniki-do-law-kominiarskich"},
         {"name": "Stopnie kominiarskie", "slug": "stopnie-kominiarskie"},
       ]},
      {"name": "Fotowoltaika", "slug": "fotowoltaika"},
      {"name": "Dachy zielone", "slug": "dachy-zielone"},
      {"name": "Zabezpieczenia przeciwśniegowe", "slug": "zabezpieczenia-przeciwsniegowe",
       "children": [
         {"name": "Płotki przeciwśniegowe", "slug": "plotki-przeciwsniegowe"},
         {"name": "Wsporniki płotków przeciwśniegowych", "slug": "wsporniki-plotkow-przeciwsniegowych"},
         {"name": "Akcesoria przeciwśniegowe", "slug": "akcesoria-przeciwsniegowe"},
         {"name": "Wsporniki belek przeciwśniegowych", "slug": "wsporniki-belek-przeciwsniegowych"},
       ]},
    ]
  },
  {
    "name": "Stropy i ściany",
    "slug": "stropy-i-sciany",
    "children": [
      {"name": "Materiały konstrukcyjne", "slug": "materialy-konstrukcyjne",
       "children": [
         {"name": "Bloczki", "slug": "bloczki"},
         {"name": "Pustaki", "slug": "pustaki"},
         {"name": "Belki stropowe betonowe", "slug": "belki-stropowe-betonowe"},
         {"name": "Belki stropowe ceramiczne", "slug": "belki-stropowe-ceramiczne"},
         {"name": "Panele wielkowymiarowe", "slug": "panele-wielkowymiarowe"},
       ]},
      {"name": "Panele ścienne i tapety", "slug": "panele-scienne-i-tapety",
       "children": [
         {"name": "Okładziny z włókna szklanego", "slug": "okladziny-z-wlokna-szklanego"},
         {"name": "Tapety ścienne", "slug": "tapety-scienne"},
       ]},
      {"name": "Schody i akcesoria strychowe", "slug": "schody-i-akcesoria-strychowe",
       "children": [
         {"name": "Schody strychowe", "slug": "schody-strychowe"},
         {"name": "Akcesoria do schodów", "slug": "akcesoria-do-schodow"},
         {"name": "Drzwi kolankowe", "slug": "drzwi-kolankowe"},
       ]},
      {"name": "Systemy kominowe", "slug": "systemy-kominowe",
       "children": [
         {"name": "Kominy ceramiczne", "slug": "kominy-ceramiczne"},
         {"name": "Kominy stalowe", "slug": "kominy-stalowe"},
         {"name": "Kominy ceramiczno-stalowe", "slug": "kominy-ceramiczno-stalowe"},
         {"name": "Akcesoria do kominów", "slug": "akcesoria-do-kominow"},
       ]},
    ]
  },
  {
    "name": "Narzędzia i mocowania",
    "slug": "narzedzia-i-mocowania",
    "children": [
      {"name": "Elementy mocujące", "slug": "elementy-mocujace",
       "children": [
         {"name": "Kotwy montażowe", "slug": "kotwy-montazowe"},
         {"name": "Śruby i podkładki", "slug": "sruby-i-podkladki"},
         {"name": "Wkręty do metalu", "slug": "wkrety-do-metalu"},
         {"name": "Nakrętki", "slug": "nakretki"},
         {"name": "Kołki i wkręty uniwersalne", "slug": "kolki-i-wkrety-uniwersalne"},
       ]},
      {"name": "Akcesoria malarskie i tynkarskie", "slug": "akcesoria-malarskie-i-tynkarskie",
       "children": [
         {"name": "Taśmy malarskie", "slug": "tasmy-malarskie"},
         {"name": "Folie ochronne budowlane", "slug": "folie-ochronne-budowlane"},
         {"name": "Wiadra i pojemniki budowlane", "slug": "wiadra-i-pojemniki-budowlane"},
         {"name": "Mieszadła budowlane", "slug": "mieszadla-budowlane"},
       ]},
      {"name": "Akcesoria murarskie", "slug": "akcesoria-murarskie"},
      {"name": "Artykuły ścierne", "slug": "artykuly-scierne",
       "children": [
         {"name": "Kostki ścierne", "slug": "kostki-scierne"},
         {"name": "Artykuły ścierne do gk", "slug": "artykuly-scierne-do-gk"},
         {"name": "Pilniki", "slug": "pilniki"},
         {"name": "Papier ścierny", "slug": "papier-scierny"},
       ]},
      {"name": "Narzędzia ręczne", "slug": "narzedzia-reczne",
       "children": [
         {"name": "Opalarki i palniki", "slug": "opalarki-i-palniki"},
         {"name": "Klucze", "slug": "klucze"},
         {"name": "Wkrętaki", "slug": "wkretaki"},
         {"name": "Zszywacze", "slug": "zszywacze"},
         {"name": "Dłuta", "slug": "dluta"},
       ]},
      {"name": "Narzędzia glazurnicze", "slug": "narzedzia-glazurnicze",
       "children": [
         {"name": "Krzyżyki do glazury", "slug": "krzyzyki-do-glazury"},
         {"name": "Przyrządy do cięcia glazury", "slug": "przyrzady-do-ciecia-glazury"},
       ]},
      {"name": "Narzędzia budowlane", "slug": "narzedzia-budowlane",
       "children": [
         {"name": "Pace i kielnie", "slug": "pace-i-kielnie"},
         {"name": "Szpachle i szpachelki", "slug": "szpachle-i-szpachelki"},
         {"name": "Młotki budowlane", "slug": "mlotki-budowlane"},
         {"name": "Ołówki i markery budowlane", "slug": "olowki-i-markery-budowlane"},
       ]},
      {"name": "Narzędzia malarskie", "slug": "narzedzia-malarskie",
       "children": [
         {"name": "Wałki malarskie", "slug": "walki-malarskie"},
         {"name": "Pędzle", "slug": "pedzle"},
         {"name": "Zestawy malarskie", "slug": "zestawy-malarskie"},
         {"name": "Kuwety i kratki malarskie", "slug": "kuwety-i-kratki-malarskie"},
       ]},
      {"name": "Elektronarzędzia", "slug": "elektronarzedzia",
       "children": [
         {"name": "Akcesoria do elektronarzędzi", "slug": "akcesoria-do-elektronarzedzi"},
         {"name": "Wiertarko-wkrętarki", "slug": "wiertarko-wkretarki"},
         {"name": "Szlifierki", "slug": "szlifierki"},
         {"name": "Piły i pilarki", "slug": "pily-i-pilarki"},
         {"name": "Klucze udarowe", "slug": "klucze-udarowe"},
       ]},
      {"name": "Narzędzia pomiarowe", "slug": "narzedzia-pomiarowe",
       "children": [
         {"name": "Poziomnice", "slug": "poziomnice"},
         {"name": "Łaty murarskie", "slug": "laty-murarskie"},
         {"name": "Miary i taśmy pomiarowe", "slug": "miary-i-tasmy-pomiarowe"},
         {"name": "Lasery i dalmierze", "slug": "lasery-i-dalmierze"},
         {"name": "Kątowniki i kątomierze", "slug": "katowniki-i-katomierce"},
       ]},
      {"name": "Narzędzia do cięcia", "slug": "narzedzia-do-ciecia",
       "children": [
         {"name": "Noże budowlane", "slug": "noze-budowlane"},
         {"name": "Obcinaczki", "slug": "obcinaczki"},
         {"name": "Przecinaki", "slug": "przecinaki"},
         {"name": "Nożyce budowlane", "slug": "nozyce-budowlane"},
         {"name": "Piły i brzeszczoty", "slug": "pily-i-brzeszczoty"},
       ]},
    ]
  },
  {
    "name": "Posadzki i podłogi",
    "slug": "posadzki",
    "children": [
      {"name": "Galanteria betonowa", "slug": "galanteria-betonowa",
       "children": [
         {"name": "Kostka brukowa", "slug": "kostka-brukowa"},
         {"name": "Palisady, krawężniki i obrzeża", "slug": "palisady-krawezniki-i-obrzeza"},
         {"name": "Płyty chodnikowe i tarasowe", "slug": "plyty-chodnikowe-i-tarasowe"},
         {"name": "Elementy kanalizacyjne", "slug": "elementy-kanalizacyjne"},
         {"name": "Elementy ogrodzenia", "slug": "elementy-ogrodzenia"},
       ]},
      {"name": "Nawadnianie", "slug": "nawadnianie",
       "children": [
         {"name": "Węże ogrodowe", "slug": "weze-ogrodowe"},
         {"name": "Zraszacze", "slug": "zraszacze"},
         {"name": "Złączki i końcówki", "slug": "zlaczki-i-koncowki"},
       ]},
      {"name": "Stolarka otworowa", "slug": "stolarka-otworowa",
       "children": [
         {"name": "Okna i akcesoria do okien", "slug": "okna-i-akcesoria-do-okien"},
         {"name": "Drzwi i akcesoria do drzwi", "slug": "drzwi-i-akcesoria-do-drzwi"},
       ]},
      {"name": "BHP i bezpieczeństwo", "slug": "bhp-i-bezpieczenstwo",
       "children": [
         {"name": "Odzież ochronna", "slug": "odziez-ochronna"},
         {"name": "Taśmy ostrzegawcze", "slug": "tasmy-ostrzegawcze"},
         {"name": "Paski i kabury narzędziowe", "slug": "paski-i-kabury-narzedzia"},
         {"name": "Drabiny", "slug": "drabiny"},
       ]},
    ]
  },
]

# Policz kategorie
def count_cats(tree):
    total = 0
    for l1 in tree:
        total += 1
        for l2 in l1.get("children", []):
            total += 1
            for l3 in l2.get("children", []):
                total += 1
    return total

print(f"Łączna liczba kategorii do zaimportowania: {count_cats(BECHCICKI_TREE)}")
print(f"Kategorie L1: {len(BECHCICKI_TREE)}")
for l1 in BECHCICKI_TREE:
    l2c = len(l1.get("children", []))
    l3c = sum(len(l2.get("children",[])) for l2 in l1.get("children",[]))
    print(f"  {l1['name']}: {l2c} L2, {l3c} L3")


Łączna liczba kategorii do zaimportowania: 313
Kategorie L1: 10
  Farby i rozpuszczalniki: 9 L2, 23 L3
  Chemia budowlana: 13 L2, 37 L3
  Sucha zabudowa: 7 L2, 24 L3
  Sufity podwieszane: 3 L2, 14 L3
  Izolacje: 8 L2, 31 L3
  Płytki: 3 L2, 8 L3
  Dachy: 8 L2, 25 L3
  Stropy i ściany: 4 L2, 14 L3
  Narzędzia i mocowania: 11 L2, 43 L3
  Posadzki i podłogi: 4 L2, 14 L3


In [19]:

import requests, json, uuid

SANITY_PROJECT = "nzcwegq7"
SANITY_DATASET = "production"
SANITY_TOKEN   = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"

QUERY_URL   = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/query/{SANITY_DATASET}"
MUTATE_URL  = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/mutate/{SANITY_DATASET}"
HEADERS_R   = {"Authorization": f"Bearer {SANITY_TOKEN}"}
HEADERS_W   = {**HEADERS_R, "Content-Type": "application/json"}

def sanity_query(groq):
    r = requests.get(QUERY_URL, params={"query": groq}, headers=HEADERS_R, timeout=30)
    r.raise_for_status()
    return r.json()["result"]

def sanity_mutate(mutations, visibility="sync"):
    payload = {"mutations": mutations, "returnDocuments": True, "returnIds": True}
    r = requests.post(MUTATE_URL, json=payload, headers=HEADERS_W,
                      params={"returnIds": "true", "visibility": visibility}, timeout=60)
    r.raise_for_status()
    return r.json()

# Pobierz wszystkie istniejące kategorie
existing = sanity_query('*[_type=="category"]{_id, name, "slug": slug.current, "parentId": parent._ref, order}')
print(f"Istniejące kategorie w Sanity: {len(existing)}")

# Mapa slug → {_id, name}
slug_to_id = {c["slug"]: c["_id"] for c in existing if c.get("slug")}
print(f"Unikalnych slugów: {len(slug_to_id)}")
print("\nPrzykłady istniejących L1:")
for c in existing:
    if not c.get("parentId"):
        print(f"  [{c['_id'][:20]}] {c['name']} → /{c.get('slug','')}")


Istniejące kategorie w Sanity: 149
Unikalnych slugów: 149

Przykłady istniejących L1:
  [cat-chemia] Chemia budowlana → /chemia-budowlana
  [cat-dachy] Dachy → /dachy
  [cat-farby] Farby i rozpuszczalniki → /farby-rozpuszczalniki
  [cat-izolacje] Izolacje → /izolacje
  [cat-narzedzia] Narzędzia i mocowania → /narzedzia-mocowania
  [cat-plytki] Płytki → /plytki
  [cat-posadzki] Posadzki → /posadzki
  [cat-stropy] Ściany i stropy → /stropy-sciany
  [cat-sucha] Sucha zabudowa → /sucha-zabudowa


In [23]:

import requests, json, time

SANITY_PROJECT = "nzcwegq7"
SANITY_DATASET = "production"
SANITY_TOKEN   = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
QUERY_URL  = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/query/{SANITY_DATASET}"
MUTATE_URL = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/mutate/{SANITY_DATASET}"
HDR_R = {"Authorization": f"Bearer {SANITY_TOKEN}"}
HDR_W = {**HDR_R, "Content-Type": "application/json"}

def q(groq):
    r = requests.get(QUERY_URL, params={"query": groq}, headers=HDR_R, timeout=30)
    r.raise_for_status()
    return r.json()["result"]

def mutate_batch(mutations):
    if not mutations: return []
    payload = {"mutations": mutations, "returnIds": True}
    r = requests.post(MUTATE_URL, json=payload, headers=HDR_W,
                      params={"returnIds":"true","visibility":"sync"}, timeout=120)
    if not r.ok:
        print(f"  ERROR {r.status_code}: {r.text[:300]}")
        return []
    return r.json().get("results", [])

# ── Odśwież mapę istniejących kategorii ──────────────────────────────────────
existing = q('*[_type=="category"]{_id, name, "slug": slug.current, "parentId": parent._ref}')
slug_map = {c["slug"]: c["_id"] for c in existing if c.get("slug")}
id_map   = {c["_id"]: c for c in existing}

# ── Mapowanie L1: stare slogi Sanity → nowe z bechcicki.pl ──────────────────
L1_REMAP = {
    "farby-i-rozpuszczalniki": ("cat-farby",    "farby-rozpuszczalniki"),
    "chemia-budowlana":        ("cat-chemia",   "chemia-budowlana"),
    "sucha-zabudowa":          ("cat-sucha",    "sucha-zabudowa"),
    "izolacje":                ("cat-izolacje", "izolacje"),
    "plytki":                  ("cat-plytki",   "plytki"),
    "dachy":                   ("cat-dachy",    "dachy"),
    "stropy-i-sciany":         ("cat-stropy",   "stropy-sciany"),
    "narzedzia-i-mocowania":   ("cat-narzedzia","narzedzia-mocowania"),
    "posadzki":                ("cat-posadzki", "posadzki"),
    # sufity-podwieszane - sprawdzimy czy jest
}

# Znajdź/utwórz ID dla każdego L1
def get_or_create_l1_id(l1_slug, l1_name, order):
    if l1_slug in L1_REMAP:
        san_id, old_slug = L1_REMAP[l1_slug]
        # Sprawdź czy istnieje
        if san_id in id_map or old_slug in slug_map:
            existing_id = san_id if san_id in id_map else slug_map[old_slug]
            # Zaktualizuj nazwę i slug jeśli różne
            patch = {"patch": {"id": existing_id, "set": {
                "name": l1_name,
                "slug": {"_type": "slug", "current": l1_slug},
                "order": order
            }}}
            mutate_batch([patch])
            slug_map[l1_slug] = existing_id
            return existing_id
    # Sprawdź po nowym slugu
    if l1_slug in slug_map:
        return slug_map[l1_slug]
    # Utwórz nowe L1
    new_id = f"cat-{l1_slug.replace('-','_')[:20]}"
    mut = {"createOrReplace": {
        "_id": new_id, "_type": "category",
        "name": l1_name,
        "slug": {"_type": "slug", "current": l1_slug},
        "order": order
    }}
    res = mutate_batch([mut])
    slug_map[l1_slug] = new_id
    return new_id

# ── FAZA 1: L1 ────────────────────────────────────────────────────────────────
print("=== FAZA 1: L1 ===")
l1_ids = {}
for i, l1 in enumerate(BECHCICKI_TREE):
    l1_id = get_or_create_l1_id(l1["slug"], l1["name"], i+1)
    l1_ids[l1["slug"]] = l1_id
    print(f"  L1 [{i+1}] {l1['name']} → {l1_id}")

print(f"\n✅ L1 gotowe: {len(l1_ids)}")

# ── FAZA 2: L2 (batchami po 20) ──────────────────────────────────────────────
print("\n=== FAZA 2: L2 ===")

# Odśwież slug_map po L1
existing2 = q('*[_type=="category"]{_id, name, "slug": slug.current}')
slug_map2 = {c["slug"]: c["_id"] for c in existing2 if c.get("slug")}

l2_ids = {}  # slug → _id
l2_mutations = []

for l1 in BECHCICKI_TREE:
    parent_id = l1_ids[l1["slug"]]
    for j, l2 in enumerate(l1.get("children", [])):
        if l2["slug"] in slug_map2:
            # Już istnieje — patch parent + order
            l2_id = slug_map2[l2["slug"]]
            l2_mutations.append({"patch": {"id": l2_id, "set": {
                "name": l2["name"],
                "parent": {"_type": "reference", "_ref": parent_id},
                "order": j+1
            }}})
        else:
            # Utwórz nowe
            l2_id = f"cat_{l2['slug'].replace('-','_')[:28]}"
            l2_mutations.append({"createOrReplace": {
                "_id": l2_id, "_type": "category",
                "name": l2["name"],
                "slug": {"_type": "slug", "current": l2["slug"]},
                "parent": {"_type": "reference", "_ref": parent_id},
                "order": j+1
            }})
        l2_ids[l2["slug"]] = l2_id if l2["slug"] not in slug_map2 else slug_map2[l2["slug"]]

# Wyślij L2 partiami
BATCH = 25
for i in range(0, len(l2_mutations), BATCH):
    batch = l2_mutations[i:i+BATCH]
    res = mutate_batch(batch)
    print(f"  L2 batch {i//BATCH+1}: {len(batch)} operacji → {len(res)} wyników")
    time.sleep(0.3)

print(f"✅ L2 gotowe: {len(l2_ids)}")

# ── FAZA 3: L3 ────────────────────────────────────────────────────────────────
print("\n=== FAZA 3: L3 ===")

# Odśwież slug_map po L2
existing3 = q('*[_type=="category"]{_id, name, "slug": slug.current}')
slug_map3 = {c["slug"]: c["_id"] for c in existing3 if c.get("slug")}

l3_mutations = []
l3_count = 0

for l1 in BECHCICKI_TREE:
    for l2 in l1.get("children", []):
        # Znajdź parent L2 id
        parent_l2_id = slug_map3.get(l2["slug"]) or l2_ids.get(l2["slug"])
        if not parent_l2_id:
            print(f"  ⚠️  Brak parent dla L2 {l2['slug']}")
            continue
        for k, l3 in enumerate(l2.get("children", [])):
            l3_count += 1
            if l3["slug"] in slug_map3:
                l3_mutations.append({"patch": {"id": slug_map3[l3["slug"]], "set": {
                    "name": l3["name"],
                    "parent": {"_type": "reference", "_ref": parent_l2_id},
                    "order": k+1
                }}})
            else:
                l3_id = f"cat_{l3['slug'].replace('-','_')[:28]}"
                l3_mutations.append({"createOrReplace": {
                    "_id": l3_id, "_type": "category",
                    "name": l3["name"],
                    "slug": {"_type": "slug", "current": l3["slug"]},
                    "parent": {"_type": "reference", "_ref": parent_l2_id},
                    "order": k+1
                }})

for i in range(0, len(l3_mutations), BATCH):
    batch = l3_mutations[i:i+BATCH]
    res = mutate_batch(batch)
    print(f"  L3 batch {i//BATCH+1}: {len(batch)} operacji → {len(res)} wyników")
    time.sleep(0.3)

print(f"✅ L3 gotowe: {l3_count} kategorii")

# ── WERYFIKACJA ────────────────────────────────────────────────────────────────
print("\n=== WERYFIKACJA ===")
final = q('*[_type=="category"]{_id, "slug": slug.current, "parentId": parent._ref}')
l1c = sum(1 for c in final if not c.get("parentId"))
l2c = sum(1 for c in final if c.get("parentId") and c["parentId"] in [v for v in l1_ids.values()])
print(f"Łącznie kategorii w Sanity: {len(final)}")
print(f"  L1 (bez rodzica): {l1c}")
print(f"  L2+L3: {len(final)-l1c}")
print("\n✅ IMPORT ZAKOŃCZONY")


=== FAZA 1: L1 ===


  L1 [1] Farby i rozpuszczalniki → cat-farby


  L1 [2] Chemia budowlana → cat-chemia


  L1 [3] Sucha zabudowa → cat-sucha


  ERROR 403: {"error":{"reason":"Documents quota limit reached. Go to sanity.io/manage to upgrade your plan.","type":"documentLimitExceededError"}}

  L1 [4] Sufity podwieszane → cat-sufity_podwieszane


...

  ERROR 409: {"error":{"description":"Mutation failed: Document \"cat_wsporniki_plotkow_przeciwsni\" references non-existent document \"cat_zabezpieczenia_przeciwsniego\"","items":[{"error":{"description":"Document \"cat_wsporniki_plotkow_przeciwsni\" references non-existent document \"cat_zabezpieczenia_przeciw
  L3 batch 7: 25 operacji → 0 wyników


  ERROR 409: {"error":{"description":"Mutation failed: Document \"cat_nakretki\" references non-existent document \"cat_elementy_mocujace\"","items":[{"error":{"description":"Document \"cat_nakretki\" references non-existent document \"cat_elementy_mocujace\"","id":"cat_nakretki","referenceID":"cat_elementy_mocu
  L3 batch 8: 25 operacji → 0 wyników


  ERROR 409: {"error":{"description":"Mutation failed: Document \"cat_elementy_ogrodzenia\" references non-existent document \"cat_galanteria_betonowa\"","items":[{"error":{"description":"Document \"cat_elementy_ogrodzenia\" references non-existent document \"cat_galanteria_betonowa\"","id":"cat_elementy_ogrodze
  L3 batch 9: 25 operacji → 0 wyników


  ERROR 409: {"error":{"description":"Mutation failed: Document \"cat_odziez_ochronna\" references non-existent document \"cat_bhp_i_bezpieczenstwo\"","items":[{"error":{"description":"Document \"cat_odziez_ochronna\" references non-existent document \"cat_bhp_i_bezpieczenstwo\"","id":"cat_odziez_ochronna","refe
  L3 batch 10: 8 operacji → 0 wyników


✅ L3 gotowe: 233 kategorii

=== WERYFIKACJA ===
Łącznie kategorii w Sanity: 149
  L1 (bez rodzica): 9
  L2+L3: 140

✅ IMPORT ZAKOŃCZONY


In [27]:

import requests

SANITY_PROJECT = "nzcwegq7"
SANITY_DATASET = "production"
SANITY_TOKEN   = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
QUERY_URL  = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/query/{SANITY_DATASET}"
MUTATE_URL = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/mutate/{SANITY_DATASET}"
HDR_R = {"Authorization": f"Bearer {SANITY_TOKEN}"}
HDR_W = {**HDR_R, "Content-Type": "application/json"}

def q(groq): 
    r = requests.get(QUERY_URL, params={"query": groq}, headers=HDR_R, timeout=30)
    r.raise_for_status()
    return r.json()["result"]

# Policz dokumenty wszystkich typów
counts = q('*[!(_id in path("drafts.**"))]{"_type": _type} | {type: _type} | group(type)')

# Alternatywny sposób
total_docs  = q('count(*[!(_id in path("drafts.**"))])')
total_cats  = q('count(*[_type=="category"])')
total_prods = q('count(*[_type=="product"])')
total_brands= q('count(*[_type=="brand"])')
total_blogs = q('count(*[_type=="blogPost"])')
total_drafts= q('count(*[_id in path("drafts.**")])')

print("=== SANITY DOCUMENT COUNT ===")
print(f"  Produkty:    {total_prods:>6}")
print(f"  Kategorie:   {total_cats:>6}")
print(f"  Marki:       {total_brands:>6}")
print(f"  Blog posts:  {total_blogs:>6}")
print(f"  Drafts:      {total_drafts:>6}")
print(f"  RAZEM:       {total_docs:>6}")
print()

# Sprawdź kategorie - które są L2 bez L3 dzieci (można ewentualnie reuse)
cats_l2 = q('*[_type=="category" && defined(parent) && !defined(*[_type=="category" && parent._ref==^._id][0]._id)]{_id, name, "slug":slug.current, "parentName":parent->name}')
cats_l1 = q('*[_type=="category" && !defined(parent)]{_id, name, "slug":slug.current}')

print(f"L1 (bez rodzica): {len(cats_l1)}")
for c in cats_l1:
    print(f"  [{c['_id']}] {c['name']} → /{c.get('slug','?')}")

print(f"\nL2 bez dzieci L3 (potencjalne reuse): {len(cats_l2)}")
print("(nie pokazuję szczegółów — za dużo)")


HTTPError: 400 Client Error: Bad Request for url: https://nzcwegq7.api.sanity.io/v2023-01-01/data/query/production?query=%2A%5B%21%28_id+in+path%28%22drafts.%2A%2A%22%29%29%5D%7B%22_type%22%3A+_type%7D+%7C+%7Btype%3A+_type%7D+%7C+group%28type%29

In [31]:

import requests, time

SANITY_PROJECT = "nzcwegq7"
SANITY_DATASET = "production"
SANITY_TOKEN   = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
QUERY_URL  = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/query/{SANITY_DATASET}"
MUTATE_URL = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/mutate/{SANITY_DATASET}"
HDR_R = {"Authorization": f"Bearer {SANITY_TOKEN}"}
HDR_W = {**HDR_R, "Content-Type": "application/json"}

def q(groq):
    r = requests.get(QUERY_URL, params={"query": groq}, headers=HDR_R, timeout=30)
    r.raise_for_status()
    return r.json()["result"]

def mutate_batch(mutations):
    if not mutations: return []
    payload = {"mutations": mutations, "returnIds": True}
    r = requests.post(MUTATE_URL, json=payload, headers=HDR_W,
                      params={"returnIds":"true","visibility":"sync"}, timeout=120)
    if not r.ok:
        print(f"  ERR {r.status_code}: {r.text[:200]}")
        return []
    return r.json().get("results", [])

# Podstawowe counts
total_prods = q('count(*[_type=="product"])')
total_cats  = q('count(*[_type=="category"])')
total_brands= q('count(*[_type=="brand"])')
total_drafts= q('count(*[_id in path("drafts.**")])')

print(f"Produkty:    {total_prods}")
print(f"Kategorie:   {total_cats}")
print(f"Marki:       {total_brands}")
print(f"Drafts:      {total_drafts}")
print(f"RAZEM (pub): {total_prods+total_cats+total_brands}")

# Pobierz wszystkie istniejące kategorie ze szczegółami
all_cats = q('*[_type=="category"]{_id, name, "slug":slug.current, "parentId":parent._ref, order}')
cats_by_slug = {c["slug"]: c for c in all_cats if c.get("slug")}
cats_by_id   = {c["_id"]: c for c in all_cats}

l1_cats = [c for c in all_cats if not c.get("parentId")]
l2_cats = [c for c in all_cats if c.get("parentId") and c["parentId"] in [x["_id"] for x in l1_cats]]
l3_cats = [c for c in all_cats if c.get("parentId") and c["parentId"] not in [x["_id"] for x in l1_cats]]

print(f"\nKategorie L1: {len(l1_cats)}")
print(f"Kategorie L2: {len(l2_cats)}")
print(f"Kategorie L3: {len(l3_cats)}")
print(f"Razem kategorii: {len(all_cats)}")


Produkty:    15921
Kategorie:   149
Marki:       1497
Drafts:      0
RAZEM (pub): 17567

Kategorie L1: 9
Kategorie L2: 130
Kategorie L3: 10
Razem kategorii: 149


In [35]:

import requests, time

SANITY_PROJECT = "nzcwegq7"
SANITY_DATASET = "production"
SANITY_TOKEN   = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
QUERY_URL  = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/query/{SANITY_DATASET}"
MUTATE_URL = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/mutate/{SANITY_DATASET}"
HDR_R = {"Authorization": f"Bearer {SANITY_TOKEN}"}
HDR_W = {**HDR_R, "Content-Type": "application/json"}

def q(groq):
    r = requests.get(QUERY_URL, params={"query": groq}, headers=HDR_R, timeout=30)
    r.raise_for_status()
    return r.json()["result"]

def mutate_batch(mutations):
    if not mutations: return []
    payload = {"mutations": mutations, "returnIds": True}
    r = requests.post(MUTATE_URL, json=payload, headers=HDR_W,
                      params={"returnIds":"true","visibility":"sync"}, timeout=120)
    if not r.ok:
        print(f"  ERR {r.status_code}: {r.text[:200]}")
        return []
    return r.json().get("results", [])

# 1. Pobierz wszystkie kategorie z liczbą produktów
all_cats = q('''*[_type=="category"]{
  _id, name, "slug":slug.current, "parentId":parent._ref,
  "productCount": count(*[_type=="product" && category._ref == ^._id])
}''')

cats_by_slug = {c["slug"]: c for c in all_cats if c.get("slug")}
cats_by_id   = {c["_id"]: c for c in all_cats}
l1_ids_set   = {c["_id"] for c in all_cats if not c.get("parentId")}

# 2. Znajdź "puste" kategorie — bez produktów i bez dzieci (można repurpose)
cats_with_children = set(c["parentId"] for c in all_cats if c.get("parentId"))
repurposable = [c for c in all_cats 
                if c.get("productCount",0) == 0 
                and c["_id"] not in cats_with_children
                and c.get("parentId")]  # tylko L2/L3
repurposable_pool = list(repurposable)  # kolejka do wykorzystania

print(f"Repurposable (puste, bez dzieci): {len(repurposable_pool)}")
print(f"L1 ids: {l1_ids_set}")
print()

# BECHCICKI_TREE jest dostępne z poprzedniej komórki - odtwórz L1 mapping
L1_EXISTING = {
    "farby-i-rozpuszczalniki": "cat-farby",
    "chemia-budowlana":        "cat-chemia",
    "sucha-zabudowa":          "cat-sucha",
    "izolacje":                "cat-izolacje",
    "plytki":                  "cat-plytki",
    "dachy":                   "cat-dachy",
    "stropy-i-sciany":         "cat-stropy",
    "narzedzia-i-mocowania":   "cat-narzedzia",
    "posadzki":                "cat-posadzki",
    "sufity-podwieszane":      None,  # BRAK — brak slotu
}

# ═══════════════════════════════════════════════════════════════════════
# FAZA 1: Zaktualizuj L1 nazwy i slugi (tylko patch — zawsze działa)
# ═══════════════════════════════════════════════════════════════════════
print("=== FAZA 1: Patchuj L1 ===")
l1_mutations = []
l1_id_map = {}  # new_slug → sanity_id

L1_NAMES = {
    "farby-i-rozpuszczalniki": "Farby i rozpuszczalniki",
    "chemia-budowlana":        "Chemia budowlana",
    "sucha-zabudowa":          "Sucha zabudowa",
    "sufity-podwieszane":      "Sufity podwieszane",
    "izolacje":                "Izolacje",
    "plytki":                  "Płytki",
    "dachy":                   "Dachy",
    "stropy-i-sciany":         "Stropy i ściany",
    "narzedzia-i-mocowania":   "Narzędzia i mocowania",
    "posadzki":                "Posadzki i podłogi",
}

for new_slug, name in L1_NAMES.items():
    sanity_id = L1_EXISTING.get(new_slug)
    if sanity_id:
        l1_mutations.append({"patch": {"id": sanity_id, "set": {
            "name": name,
            "slug": {"_type": "slug", "current": new_slug},
        }}})
        l1_id_map[new_slug] = sanity_id
        print(f"  PATCH {sanity_id} → {new_slug}")
    else:
        # Sufity podwieszane — spróbuj CREATE (może się uda)
        new_id = "cat-sufity-podwieszane"
        l1_mutations.append({"createIfNotExists": {
            "_id": new_id, "_type": "category",
            "name": name,
            "slug": {"_type": "slug", "current": new_slug},
            "order": 4
        }})
        l1_id_map[new_slug] = new_id
        print(f"  CREATE {new_id} → {new_slug}")

res = mutate_batch(l1_mutations)
print(f"L1 wynik: {len(res)} operacji")

# Odśwież po L1
all_cats2 = q('*[_type=="category"]{_id, name, "slug":slug.current, "parentId":parent._ref, "productCount": count(*[_type=="product" && category._ref == ^._id])}')
cats_by_slug2 = {c["slug"]: c for c in all_cats2 if c.get("slug")}
l1_ids_set2 = {c["_id"] for c in all_cats2 if not c.get("parentId")}
print(f"\nL1 po aktualizacji: {len(l1_ids_set2)}")


Repurposable (puste, bez dzieci): 7
L1 ids: {'cat-farby', 'cat-stropy', 'cat-sucha', 'cat-plytki', 'cat-izolacje', 'cat-chemia', 'cat-narzedzia', 'cat-dachy', 'cat-posadzki'}

=== FAZA 1: Patchuj L1 ===
  PATCH cat-farby → farby-i-rozpuszczalniki
  PATCH cat-chemia → chemia-budowlana
  PATCH cat-sucha → sucha-zabudowa
  CREATE cat-sufity-podwieszane → sufity-podwieszane
  PATCH cat-izolacje → izolacje
  PATCH cat-plytki → plytki
  PATCH cat-dachy → dachy
  PATCH cat-stropy → stropy-i-sciany
  PATCH cat-narzedzia → narzedzia-i-mocowania
  PATCH cat-posadzki → posadzki
  ERR 403: {"error":{"reason":"Documents quota limit reached. Go to sanity.io/manage to upgrade your plan.","type":"documentLimitExceededError"}}

L1 wynik: 0 operacji



L1 po aktualizacji: 9


In [39]:

import requests, time

SANITY_PROJECT = "nzcwegq7"
SANITY_DATASET = "production"
SANITY_TOKEN   = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
QUERY_URL  = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/query/{SANITY_DATASET}"
MUTATE_URL = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/mutate/{SANITY_DATASET}"
HDR_R = {"Authorization": f"Bearer {SANITY_TOKEN}"}
HDR_W = {**HDR_R, "Content-Type": "application/json"}

def q(groq):
    r = requests.get(QUERY_URL, params={"query": groq}, headers=HDR_R, timeout=30)
    r.raise_for_status()
    return r.json()["result"]

def patch_batch(mutations):
    """Tylko patch mutations — nigdy nie tworzą nowych dokumentów."""
    if not mutations: return 0
    # Upewnij się że wszystkie to patche
    for m in mutations:
        assert "patch" in m, f"Nie-patch w patch_batch: {m}"
    payload = {"mutations": mutations, "returnIds": True}
    r = requests.post(MUTATE_URL, json=payload, headers=HDR_W,
                      params={"returnIds":"true","visibility":"sync"}, timeout=120)
    if not r.ok:
        print(f"  ERR {r.status_code}: {r.text[:150]}")
        return 0
    return len(r.json().get("results", []))

def run_patches(patches, label):
    total = 0
    BATCH = 30
    for i in range(0, len(patches), BATCH):
        total += patch_batch(patches[i:i+BATCH])
        time.sleep(0.2)
    print(f"  {label}: {total}/{len(patches)} patchów OK")
    return total

# ════════════════════════════════════════════════════════════════════
# Odśwież aktualne dane
all_cats = q('''*[_type=="category"]{
  _id, name, "slug":slug.current, "parentId":parent._ref,
  "productCount": count(*[_type=="product" && category._ref == ^._id])
}''')
by_slug = {c["slug"]: c for c in all_cats if c.get("slug")}
by_id   = {c["_id"]: c for c in all_cats}
l1_ids  = {c["_id"] for c in all_cats if not c.get("parentId")}

# ════════════════════════════════════════════════════════════════════
# KROK 1: Zaktualizuj L1 (tylko patche — bez sufity bo nie istnieje)
# ════════════════════════════════════════════════════════════════════
print("=== KROK 1: Patch L1 ===")
L1_MAP = {  # nowy_slug: (sanity_id, nowa_nazwa)
    "farby-i-rozpuszczalniki": ("cat-farby",    "Farby i rozpuszczalniki"),
    "chemia-budowlana":        ("cat-chemia",   "Chemia budowlana"),
    "sucha-zabudowa":          ("cat-sucha",    "Sucha zabudowa"),
    "izolacje":                ("cat-izolacje", "Izolacje"),
    "plytki":                  ("cat-plytki",   "Płytki"),
    "dachy":                   ("cat-dachy",    "Dachy"),
    "stropy-i-sciany":         ("cat-stropy",   "Stropy i ściany"),
    "narzedzia-i-mocowania":   ("cat-narzedzia","Narzędzia i mocowania"),
    "posadzki":                ("cat-posadzki", "Posadzki i podłogi"),
}
l1_new_ids = {}
l1_patches = []
for new_slug, (sanity_id, name) in L1_MAP.items():
    l1_patches.append({"patch": {"id": sanity_id, "set": {
        "name": name,
        "slug": {"_type": "slug", "current": new_slug},
    }}})
    l1_new_ids[new_slug] = sanity_id

run_patches(l1_patches, "L1 patches")

# ════════════════════════════════════════════════════════════════════
# KROK 2: Zmapuj istniejące L2 na nowe L2 z BECHCICKI_TREE
# ════════════════════════════════════════════════════════════════════
print("\n=== KROK 2: Remapping L2 ===")

# Odśwież po L1
all_cats2 = q('*[_type=="category"]{_id, name, "slug":slug.current, "parentId":parent._ref, "productCount": count(*[_type=="product" && category._ref == ^._id])}')
by_slug2 = {c["slug"]: c for c in all_cats2 if c.get("slug")}
by_id2   = {c["_id"]: c for c in all_cats2}
l1_ids2  = {c["_id"] for c in all_cats2 if not c.get("parentId")}

# Kategorie do przerobienia — bez produktów
cats_with_children2 = {c["parentId"] for c in all_cats2 if c.get("parentId")}
repurposable = [c for c in all_cats2 
                if c.get("productCount", 0) == 0 
                and c["_id"] not in cats_with_children2
                and c.get("parentId")]
repurposable.sort(key=lambda c: c["_id"])
repo_pool = list(repurposable)  # kolejka do reuse

# Zbierz docelowe L2 i L3 z BECHCICKI_TREE
target_l2 = []  # (new_slug, name, parent_l1_slug)
target_l3 = []  # (new_slug, name, parent_l2_slug)

for l1 in BECHCICKI_TREE:
    l1_slug = l1["slug"]
    if l1_slug == "sufity-podwieszane": continue  # brak L1 w Sanity
    for l2 in l1.get("children", []):
        target_l2.append((l2["slug"], l2["name"], l1_slug))
        for l3 in l2.get("children", []):
            target_l3.append((l3["slug"], l3["name"], l2["slug"]))

print(f"Docelowe L2: {len(target_l2)}")
print(f"Docelowe L3: {len(target_l3)}")
print(f"Repurposable docs: {len(repo_pool)}")

# ── Zdecyduj co patchujemy, co reusujemy, co nie możemy zrobić ──
l2_patches = []
l2_id_map = {}  # new_slug → sanity_id
new_creates_needed = []

for new_slug, name, parent_l1_slug in target_l2:
    parent_id = l1_new_ids.get(parent_l1_slug)
    if not parent_id: continue
    
    if new_slug in by_slug2:
        # Już istnieje — patch
        existing_id = by_slug2[new_slug]["_id"]
        l2_patches.append({"patch": {"id": existing_id, "set": {
            "name": name,
            "slug": {"_type": "slug", "current": new_slug},
            "parent": {"_type": "reference", "_ref": parent_id},
        }}})
        l2_id_map[new_slug] = existing_id
    elif repo_pool:
        # Reuse pustego slotu
        repo_doc = repo_pool.pop(0)
        l2_patches.append({"patch": {"id": repo_doc["_id"], "set": {
            "name": name,
            "slug": {"_type": "slug", "current": new_slug},
            "parent": {"_type": "reference", "_ref": parent_id},
        }}})
        l2_id_map[new_slug] = repo_doc["_id"]
    else:
        new_creates_needed.append(("L2", new_slug, name))

run_patches(l2_patches, "L2 patches/reuse")

# ════════════════════════════════════════════════════════════════════
# KROK 3: L3 — reuse z pozostałych pustych + istniejących
# ════════════════════════════════════════════════════════════════════
print("\n=== KROK 3: L3 ===")

# Odśwież slug_map po L2
all_cats3 = q('*[_type=="category"]{_id, "slug":slug.current}')
by_slug3 = {c["slug"]: c["_id"] for c in all_cats3 if c.get("slug")}
# Połącz z l2_id_map
for slug, sid in l2_id_map.items():
    by_slug3[slug] = sid

l3_patches = []
created_count = 0

for new_slug, name, parent_l2_slug in target_l3:
    parent_id = by_slug3.get(parent_l2_slug) or l2_id_map.get(parent_l2_slug)
    if not parent_id:
        new_creates_needed.append(("L3-no-parent", new_slug, name))
        continue
    
    if new_slug in by_slug3:
        l3_patches.append({"patch": {"id": by_slug3[new_slug], "set": {
            "name": name,
            "slug": {"_type": "slug", "current": new_slug},
            "parent": {"_type": "reference", "_ref": parent_id},
        }}})
    elif repo_pool:
        repo_doc = repo_pool.pop(0)
        l3_patches.append({"patch": {"id": repo_doc["_id"], "set": {
            "name": name,
            "slug": {"_type": "slug", "current": new_slug},
            "parent": {"_type": "reference", "_ref": parent_id},
        }}})
    else:
        new_creates_needed.append(("L3", new_slug, name))

run_patches(l3_patches, "L3 patches/reuse")

# ════════════════════════════════════════════════════════════════════
# RAPORT
# ════════════════════════════════════════════════════════════════════
print("\n=== RAPORT KOŃCOWY ===")
final_cats = q('count(*[_type=="category"])')
print(f"Kategorie w Sanity: {final_cats}")
print(f"\nNIEMOŻLIWE bez upgrade (quota 403): {len(new_creates_needed)}")
by_type = {}
for t, slug, name in new_creates_needed:
    by_type.setdefault(t, []).append(slug)
for t, slugs in by_type.items():
    print(f"  {t}: {len(slugs)} kategorii")
    for s in slugs[:5]: print(f"    - {s}")
    if len(slugs)>5: print(f"    ... i {len(slugs)-5} więcej")

print("\n⚠️  WYMAGANY UPGRADE SANITY PLAN — sanity.io/manage")


=== KROK 1: Patch L1 ===


  L1 patches: 9/9 patchów OK

=== KROK 2: Remapping L2 ===


Docelowe L2: 67
Docelowe L3: 219
Repurposable docs: 7


  L2 patches/reuse: 18/20 patchów OK

=== KROK 3: L3 ===


  L3 patches/reuse: 11/11 patchów OK

=== RAPORT KOŃCOWY ===
Kategorie w Sanity: 149

NIEMOŻLIWE bez upgrade (quota 403): 255
  L2: 47 kategorii
    - piany-montazowe
    - uszczelniacze-i-silikony
    - zaprawy
    - powloki-epoksydowe
    - srodki-czyszczaco-pielegnacyjne
    ... i 42 więcej
  L3: 46 kategorii
    - farby-wewnetrzne-biale
    - farby-wewnetrzne-kolorowe
    - farby-elewacyjne-silikonowe
    - farby-elewacyjne-emulsyjne
    - farby-elewacyjne-akrylowe
    ... i 41 więcej
  L3-no-parent: 162 kategorii
    - piany-montazowe-pistoletowe
    - piany-montazowe-wezzykowe
    - czysciki-do-pian-montazowych
    - silikony-sanitarne
    - silikony-uniwersalne
    ... i 157 więcej

⚠️  WYMAGANY UPGRADE SANITY PLAN — sanity.io/manage


In [43]:

import requests, time, re

SANITY_PROJECT = "nzcwegq7"
SANITY_DATASET = "production"
SANITY_TOKEN   = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
QUERY_URL  = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/query/{SANITY_DATASET}"
MUTATE_URL = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/mutate/{SANITY_DATASET}"
HDR_R = {"Authorization": f"Bearer {SANITY_TOKEN}"}
HDR_W = {**HDR_R, "Content-Type": "application/json"}

def q(groq):
    r = requests.get(QUERY_URL, params={"query": groq}, headers=HDR_R, timeout=30)
    r.raise_for_status()
    return r.json()["result"]

def mutate(mutations):
    if not mutations: return 0
    payload = {"mutations": mutations, "returnIds": True}
    r = requests.post(MUTATE_URL, json=payload, headers=HDR_W,
                      params={"returnIds":"true","visibility":"sync"}, timeout=120)
    if not r.ok:
        print(f"  ERR {r.status_code}: {r.text[:200]}")
        return 0
    return len(r.json().get("results", []))

def run_mutations(muts, label, batch=25):
    ok = 0
    for i in range(0, len(muts), batch):
        ok += mutate(muts[i:i+batch])
        time.sleep(0.3)
    print(f"  {label}: {ok}/{len(muts)} OK")
    return ok

# ── Pełne drzewo bechcicki.pl (odtworzone) ──────────────────────────────────
TREE = [
  {"name":"Farby i rozpuszczalniki","slug":"farby-i-rozpuszczalniki","sanity_id":"cat-farby","children":[
    {"name":"Farby wewnętrzne","slug":"farby-wewnetrzne","children":[
      {"name":"Farby wewnętrzne białe","slug":"farby-wewnetrzne-biale"},
      {"name":"Farby wewnętrzne kolorowe","slug":"farby-wewnetrzne-kolorowe"}]},
    {"name":"Farby elewacyjne","slug":"farby-elewacyjne","children":[
      {"name":"Farby elewacyjne silikonowe","slug":"farby-elewacyjne-silikonowe"},
      {"name":"Farby elewacyjne emulsyjne","slug":"farby-elewacyjne-emulsyjne"},
      {"name":"Farby elewacyjne akrylowe","slug":"farby-elewacyjne-akrylowe"},
      {"name":"Farby elewacyjne silikatowe","slug":"farby-elewacyjne-silikatowe"},
      {"name":"Farby elewacyjne specjalne","slug":"farby-elewacyjne-specjalne"}]},
    {"name":"Farby do drewna","slug":"farby-do-drewna","children":[
      {"name":"Lakiery do drewna","slug":"lakiery-do-drewna"},
      {"name":"Lakierobejce","slug":"lakierobejce"},
      {"name":"Oleje do drewna","slug":"oleje-do-drewna"},
      {"name":"Impregnaty do drewna","slug":"impregnaty-do-drewna"},
      {"name":"Podkłady wypełniające","slug":"podklady-wypelniajace"}]},
    {"name":"Farby do metalu","slug":"farby-do-metalu","children":[
      {"name":"Emalie chlorokauczukowe","slug":"emalie-chlorokauczukowe"},
      {"name":"Emalie ftalowe","slug":"emalie-ftalowe"},
      {"name":"Emalie poliuretanowe","slug":"emalie-poliuretanowe"},
      {"name":"Emalie akrylowe","slug":"emalie-akrylowe"},
      {"name":"Lakiery do metalu","slug":"lakiery-do-metalu"}]},
    {"name":"Bazy i koloranty","slug":"bazy-i-koloranty"},
    {"name":"Farby specjalistyczne","slug":"farby-specjalistyczne","children":[
      {"name":"Farby przemysłowe","slug":"farby-przemyslowe"},
      {"name":"Farby proszkowe","slug":"farby-proszkowe"}]},
    {"name":"Farby pozostałe","slug":"farby-pozostale","children":[
      {"name":"Farby do betonu","slug":"farby-do-betonu"},
      {"name":"Pigmenty","slug":"pigmenty"},
      {"name":"Farby zaprawkowe","slug":"farby-zaprawkowe"},
      {"name":"Spraye budowlane","slug":"spraye-budowlane"}]},
    {"name":"Rozpuszczalniki","slug":"rozpuszczalniki"},
    {"name":"Preparaty czyszczące powierzchnie","slug":"preparaty-czyszczace-powierzchnie"}]},
  {"name":"Chemia budowlana","slug":"chemia-budowlana","sanity_id":"cat-chemia","children":[
    {"name":"Tynki","slug":"tynki","children":[
      {"name":"Tynki elewacyjne","slug":"tynki-elewacyjne"},
      {"name":"Tynki cementowo-wapienne","slug":"tynki-cementowo-wapienne"},
      {"name":"Tynki gipsowe","slug":"tynki-gipsowe"},
      {"name":"Tynki wapienne","slug":"tynki-wapienne"},
      {"name":"Tynki specjalne","slug":"tynki-specjalne"}]},
    {"name":"Kleje","slug":"kleje","children":[
      {"name":"Kleje montażowe","slug":"kleje-montazowe"},
      {"name":"Kleje do glazury","slug":"kleje-do-glazury"},
      {"name":"Kleje do drewna","slug":"kleje-do-drewna"},
      {"name":"Kleje do styropianu i styroduru","slug":"kleje-do-styropianu-i-styroduru"},
      {"name":"Kleje do wełen","slug":"kleje-do-welen"}]},
    {"name":"Gipsy i gładzie","slug":"gipsy-i-gladzie","children":[
      {"name":"Gładzie gipsowe w proszku","slug":"gladzie-gipsowe-w-proszku"},
      {"name":"Gipsy szpachlowe","slug":"gipsy-szpachlowe"},
      {"name":"Gładzie masy gotowe","slug":"gladzie-masy-gotowe"},
      {"name":"Kleje gipsowe","slug":"kleje-gipsowe"},
      {"name":"Gipsy budowlane","slug":"gipsy-budowlane"}]},
    {"name":"Grunty","slug":"grunty","children":[
      {"name":"Grunty uniwersalne","slug":"grunty-uniwersalne"},
      {"name":"Masy bitumiczne gruntujące","slug":"masy-bitumiczne-gruntujace"},
      {"name":"Grunty specjalistyczne","slug":"grunty-specjalistyczne"},
      {"name":"Grunty do posadzek","slug":"grunty-do-posadzek"},
      {"name":"Grunty pod tynki","slug":"grunty-pod-tynki"}]},
    {"name":"Piany montażowe","slug":"piany-montazowe","children":[
      {"name":"Piany montażowe pistoletowe","slug":"piany-montazowe-pistoletowe"},
      {"name":"Piany montażowe wężykowe","slug":"piany-montazowe-wezzykowe"},
      {"name":"Czyściki do pian montażowych","slug":"czysciki-do-pian-montazowych"}]},
    {"name":"Uszczelniacze i silikony","slug":"uszczelniacze-i-silikony","children":[
      {"name":"Silikony sanitarne","slug":"silikony-sanitarne"},
      {"name":"Silikony uniwersalne","slug":"silikony-uniwersalne"},
      {"name":"Silikony wysokotemperaturowe","slug":"silikony-wysokotemperaturowe"},
      {"name":"Silikony dekarskie","slug":"silikony-dekarskie"},
      {"name":"Silikony szklarskie","slug":"silikony-szklarskie"}]},
    {"name":"Zaprawy","slug":"zaprawy","children":[
      {"name":"Zaprawy specjalistyczne","slug":"zaprawy-specjalistyczne"},
      {"name":"Zaprawy naprawcze","slug":"zaprawy-naprawcze"},
      {"name":"Zaprawy uszczelniające","slug":"zaprawy-uszczelniajace"},
      {"name":"Masy samopoziomujące","slug":"masy-samopoziomujace"},
      {"name":"Zaprawy murarskie","slug":"zaprawy-murarskie"}]},
    {"name":"Spoiny","slug":"spoiny","children":[
      {"name":"Spoiny elastyczne","slug":"spoiny-elastyczne"},
      {"name":"Spoiny specjalistyczne","slug":"spoiny-specjalistyczne"},
      {"name":"Zaprawy do spoin","slug":"zaprawy-do-spoin"},
      {"name":"Spoiny zwykłe","slug":"spoiny-zwykle"}]},
    {"name":"Powłoki epoksydowe","slug":"powloki-epoksydowe"},
    {"name":"Kotwy chemiczne","slug":"kotwy-chemiczne"},
    {"name":"Środki grzybobójcze","slug":"srodki-grzybobojcze"},
    {"name":"Środki czyszcząco-pielęgnacyjne","slug":"srodki-czyszczaco-pielegnacyjne"},
    {"name":"Dodatki do zapraw i betonu","slug":"dodatki-do-zapraw-i-betonu"}]},
  {"name":"Sucha zabudowa","slug":"sucha-zabudowa","sanity_id":"cat-sucha","children":[
    {"name":"Płyty do suchej zabudowy","slug":"plyty-do-suchej-zabudowy","children":[
      {"name":"Płyty cementowe","slug":"plyty-cementowe"},
      {"name":"Płyty gipsowo-kartonowe","slug":"plyty-gipsowo-kartonowe"},
      {"name":"Płyty gipsowo-włóknowe","slug":"plyty-gipsowo-wloknowe"},
      {"name":"Płyty cementowo-włóknowe","slug":"plyty-cementowo-wloknowe"},
      {"name":"Płyty drewnopochodne","slug":"plyty-drewnopochodne"}]},
    {"name":"Profile do suchej zabudowy","slug":"profile-do-suchej-zabudowy","children":[
      {"name":"Profile ścienne","slug":"profile-scienne"},
      {"name":"Profile ościeżnicowe","slug":"profile-oscieznicowe"},
      {"name":"Profile sufitowe","slug":"profile-sufitowe"}]},
    {"name":"Wieszaki do suchej zabudowy","slug":"wieszaki-do-suchej-zabudowy","children":[
      {"name":"Wieszaki z noniuszem","slug":"wieszaki-z-noniuszem"},
      {"name":"Wieszaki ES","slug":"wieszaki-es"},
      {"name":"Wieszaki do poddaszy","slug":"wieszaki-do-poddaszy"},
      {"name":"Wieszaki bezpośrednie","slug":"wieszaki-bezposrednie"},
      {"name":"Wieszaki obrotowe","slug":"wieszaki-obrotowe"}]},
    {"name":"Mocowania do suchej zabudowy","slug":"mocowania-do-suchej-zabudowy","children":[
      {"name":"Wkręty do suchej zabudowy","slug":"wkrety-do-suchej-zabudowy"},
      {"name":"Łączniki do profili","slug":"laczniki-do-profili"},
      {"name":"Kątowniki do profili","slug":"katowniki-do-profili"},
      {"name":"Kołki do suchej zabudowy","slug":"kolki-do-suchej-zabudowy"}]},
    {"name":"Narożniki i listwy","slug":"narozniki-i-listwy","children":[
      {"name":"Narożniki aluminiowe","slug":"narozniki-aluminiowe"},
      {"name":"Listwy krawędziowe","slug":"listwy-krawedzi"},
      {"name":"Narożniki PVC","slug":"narozniki-pvc"},
      {"name":"Narożniki do tynków mokrych","slug":"narozniki-do-tynkow-mokrych"},
      {"name":"Narożniki elastyczne","slug":"narozniki-elastyczne"}]},
    {"name":"Taśmy do suchej zabudowy","slug":"tasmy-do-suchej-zabudowy"},
    {"name":"Rewizje i klapy","slug":"rewizje-i-klapy","children":[
      {"name":"Klapy rewizyjne","slug":"klapy-rewizyjne"},
      {"name":"Drzwiczki rewizyjne","slug":"drzwiczki-rewizyjne"}]}]},
  {"name":"Sufity podwieszane","slug":"sufity-podwieszane","sanity_id":"cat-sufity-podwieszane","children":[
    {"name":"Płyty sufitowe","slug":"plyty-sufitowe","children":[
      {"name":"Sufity rastrowe","slug":"sufity-rastrowe"},
      {"name":"Płyty sufitowe z wełny mineralnej","slug":"plyty-sufitowe-welna-mineralna"},
      {"name":"Płyty sufitowe z wełny szklanej","slug":"plyty-sufitowe-welna-szklana"},
      {"name":"Płyty sufitowe drewniane","slug":"plyty-sufitowe-drewniane"},
      {"name":"Płyty sufitowe metalowe","slug":"plyty-sufitowe-metalowe"}]},
    {"name":"Profile do sufitów podwieszanych","slug":"profile-do-sufitow-podwieszanych","children":[
      {"name":"Profile nośne główne","slug":"profile-nosne-glowne"},
      {"name":"Profile poprzeczne","slug":"profile-poprzeczne"},
      {"name":"Profile przyścienne","slug":"profile-przyscienne"},
      {"name":"Profile specjalne sufitowe","slug":"profile-specjalne-sufitowe"}]},
    {"name":"Mocowania do sufitów podwieszanych","slug":"mocowania-do-sufitow-podwieszanych","children":[
      {"name":"Elementy wykończeniowe sufitów","slug":"elementy-wykonczeniowe-sufitow"},
      {"name":"Wieszaki dwuhakowe","slug":"wieszaki-dwuhakowe"},
      {"name":"Klipsy mocujące sufitów","slug":"klipsy-mocujace-sufitow"},
      {"name":"Druty z hakiem","slug":"druty-z-hakiem"},
      {"name":"Druty z oczkiem","slug":"druty-z-oczkiem"}]}]},
  {"name":"Izolacje","slug":"izolacje","sanity_id":"cat-izolacje","children":[
    {"name":"Styropiany","slug":"styropiany","children":[
      {"name":"Styropiany fasadowe EPS","slug":"styropiany-fasadowe-eps"},
      {"name":"Styropian dach-podłoga EPS","slug":"styropian-dach-podloga-eps"},
      {"name":"Styropiany akustyczne","slug":"styropiany-akustyczne"},
      {"name":"Styropiany do fundamentów","slug":"styropiany-do-fundamentow"}]},
    {"name":"Płyty XPS","slug":"plyty-xps"},
    {"name":"Wełny izolacyjne","slug":"welny-izolacyjne","children":[
      {"name":"Wełny do ścian działowych","slug":"welny-do-scian-dzialowych"},
      {"name":"Wełny fasadowe","slug":"welny-fasadowe"},
      {"name":"Wełny do stropów i podłóg","slug":"welny-do-stropow-i-podlog"},
      {"name":"Wełny do dachów płaskich","slug":"welny-do-dachow-plaskich"},
      {"name":"Wełny do poddaszy","slug":"welny-do-poddaszy"}]},
    {"name":"Izolacje budowlane","slug":"izolacje-budowlane","children":[
      {"name":"Izolacje fasadowe","slug":"izolacje-fasadowe"},
      {"name":"Izolacje stropów i podłóg","slug":"izolacje-stropow-i-podlog"},
      {"name":"Izolacje dachów płaskich","slug":"izolacje-dachow-plaskich"},
      {"name":"Izolacje fundamentów","slug":"izolacje-fundamentow"},
      {"name":"Izolacje akustyczne","slug":"izolacje-akustyczne"}]},
    {"name":"Izolacje techniczne","slug":"izolacje-techniczne","children":[
      {"name":"Izolacje HVAC","slug":"izolacje-hvac"},
      {"name":"Izolacje przemysłowe","slug":"izolacje-przemyslowe"}]},
    {"name":"Hydroizolacje","slug":"hydroizolacje","children":[
      {"name":"Papy hydroizolacyjne","slug":"papy-hydroizolacyjne"},
      {"name":"Hydroizolacje bitumiczne","slug":"hydroizolacje-bitumiczne"},
      {"name":"Taśmy uszczelniające do hydroizolacji","slug":"tasmy-uszczelniajace-hydroizolacje"},
      {"name":"Membrany dachowe","slug":"membrany-dachowe"},
      {"name":"Hydroizolacje mineralne","slug":"hydroizolacje-mineralne"}]},
    {"name":"Folie izolacyjne","slug":"folie-izolacyjne","children":[
      {"name":"Folie fundamentowe","slug":"folie-fundamentowe"},
      {"name":"Folie paroprzepuszczalne","slug":"folie-paroprzepuszczalne"},
      {"name":"Folie paroizolacyjne","slug":"folie-paroizolacyjne"},
      {"name":"Folie budowlane","slug":"folie-budowlane"},
      {"name":"Taśmy do łączenia folii","slug":"tasmy-do-laczenia-folii"}]},
    {"name":"Akcesoria do izolacji","slug":"akcesoria-do-izolacji","children":[
      {"name":"Łączniki do izolacji fasadowych","slug":"laczniki-do-izolacji-fasadowych"},
      {"name":"Opaski zaciskowe","slug":"opaski-zaciskowe"},
      {"name":"Talerze dociskowe","slug":"talerze-dociskowe"},
      {"name":"Siatki elewacyjne","slug":"siatki-elewacyjne"},
      {"name":"Profile i narożniki do izolacji","slug":"profile-i-narozniki-do-izolacji"}]}]},
  {"name":"Płytki","slug":"plytki","sanity_id":"cat-plytki","children":[
    {"name":"Płytki ceramiczne","slug":"plytki-ceramiczne","children":[
      {"name":"Płytki ścienne","slug":"plytki-scienne"},
      {"name":"Płytki ścienno-podłogowe","slug":"plytki-scienno-podlogowe"},
      {"name":"Płytki elewacyjne","slug":"plytki-elewacyjne"},
      {"name":"Płytki tarasowe","slug":"plytki-tarasowe"},
      {"name":"Stopnice","slug":"stopnice"}]},
    {"name":"Płytki dekoracyjne","slug":"plytki-dekoracyjne","children":[
      {"name":"Panele i dekory ścienne","slug":"panele-i-dekory-scienne"}]},
    {"name":"Listwy i akcesoria płytkarskie","slug":"listwy-i-akcesoria-plytkarskie","children":[
      {"name":"Listwy przypodłogowe","slug":"listwy-przypodlogowe"},
      {"name":"Akcesoria instalacyjne płytki","slug":"akcesoria-instalacyjne-plytki"}]}]},
  {"name":"Dachy","slug":"dachy","sanity_id":"cat-dachy","children":[
    {"name":"Pokrycia dachowe","slug":"pokrycia-dachowe","children":[
      {"name":"Dachówki ceramiczne","slug":"dachowki-ceramiczne"},
      {"name":"Dachówki betonowe","slug":"dachowki-betonowe"},
      {"name":"Pokrycia dachowe z blachy","slug":"pokrycia-dachowe-z-blachy"},
      {"name":"Gonty bitumiczne","slug":"gonty-bitumiczne"},
      {"name":"Papy dachowe","slug":"papy-dachowe"}]},
    {"name":"Okna dachowe i akcesoria","slug":"okna-dachowe-i-akcesoria","children":[
      {"name":"Okna dachowe","slug":"okna-dachowe"},
      {"name":"Okna wyłazowe","slug":"okna-wylazowe"},
      {"name":"Balkony dachowe","slug":"balkony-dachowe"},
      {"name":"Okna kolankowe","slug":"okna-kolankowe"},
      {"name":"Okna oddymiające","slug":"okna-oddymiajace"}]},
    {"name":"Rynny i systemy rynnowe","slug":"rynny-i-systemy-rynnowe","children":[
      {"name":"Rynny z blachy powlekanej","slug":"rynny-z-blachy-powlekanej"},
      {"name":"Rynny PVC","slug":"rynny-pvc"},
      {"name":"Rynny ocynkowane","slug":"rynny-ocynkowane"},
      {"name":"Akcesoria do rynien","slug":"akcesoria-do-rynien"}]},
    {"name":"Zamocowania dachowe","slug":"zamocowania-dachowe","children":[
      {"name":"Klamry do gąsiorów","slug":"klamry-do-gasiorow"},
      {"name":"Łączniki teleskopowe dachowe","slug":"laczniki-teleskopowe-dachowe"},
      {"name":"Spinki do dachówek","slug":"spinki-do-dachowek"},
      {"name":"Gwoździe i podkładki dachowe","slug":"gwozdzie-i-podkladki-dachowe"}]},
    {"name":"Komunikacja dachowa","slug":"komunikacja-dachowa","children":[
      {"name":"Ławy kominiarskie","slug":"lawy-kominiarskie"},
      {"name":"Wsporniki do ław kominiarskich","slug":"wsporniki-do-law-kominiarskich"},
      {"name":"Stopnie kominiarskie","slug":"stopnie-kominiarskie"}]},
    {"name":"Fotowoltaika","slug":"fotowoltaika"},
    {"name":"Dachy zielone","slug":"dachy-zielone"},
    {"name":"Zabezpieczenia przeciwśniegowe","slug":"zabezpieczenia-przeciwsniegowe","children":[
      {"name":"Płotki przeciwśniegowe","slug":"plotki-przeciwsniegowe"},
      {"name":"Wsporniki płotków przeciwśniegowych","slug":"wsporniki-plotkow-przeciwsniegowych"},
      {"name":"Akcesoria przeciwśniegowe","slug":"akcesoria-przeciwsniegowe"},
      {"name":"Wsporniki belek przeciwśniegowych","slug":"wsporniki-belek-przeciwsniegowych"}]}]},
  {"name":"Stropy i ściany","slug":"stropy-i-sciany","sanity_id":"cat-stropy","children":[
    {"name":"Materiały konstrukcyjne","slug":"materialy-konstrukcyjne","children":[
      {"name":"Bloczki","slug":"bloczki"},
      {"name":"Pustaki","slug":"pustaki"},
      {"name":"Belki stropowe betonowe","slug":"belki-stropowe-betonowe"},
      {"name":"Belki stropowe ceramiczne","slug":"belki-stropowe-ceramiczne"},
      {"name":"Panele wielkowymiarowe","slug":"panele-wielkowymiarowe"}]},
    {"name":"Panele ścienne i tapety","slug":"panele-scienne-i-tapety","children":[
      {"name":"Okładziny z włókna szklanego","slug":"okladziny-z-wlokna-szklanego"},
      {"name":"Tapety ścienne","slug":"tapety-scienne"}]},
    {"name":"Schody i akcesoria strychowe","slug":"schody-i-akcesoria-strychowe","children":[
      {"name":"Schody strychowe","slug":"schody-strychowe"},
      {"name":"Akcesoria do schodów","slug":"akcesoria-do-schodow"},
      {"name":"Drzwi kolankowe","slug":"drzwi-kolankowe"}]},
    {"name":"Systemy kominowe","slug":"systemy-kominowe","children":[
      {"name":"Kominy ceramiczne","slug":"kominy-ceramiczne"},
      {"name":"Kominy stalowe","slug":"kominy-stalowe"},
      {"name":"Kominy ceramiczno-stalowe","slug":"kominy-ceramiczno-stalowe"},
      {"name":"Akcesoria do kominów","slug":"akcesoria-do-kominow"}]}]},
  {"name":"Narzędzia i mocowania","slug":"narzedzia-i-mocowania","sanity_id":"cat-narzedzia","children":[
    {"name":"Elementy mocujące","slug":"elementy-mocujace","children":[
      {"name":"Kotwy montażowe","slug":"kotwy-montazowe"},
      {"name":"Śruby i podkładki","slug":"sruby-i-podkladki"},
      {"name":"Wkręty do metalu","slug":"wkrety-do-metalu"},
      {"name":"Nakrętki","slug":"nakretki"},
      {"name":"Kołki i wkręty uniwersalne","slug":"kolki-i-wkrety-uniwersalne"}]},
    {"name":"Akcesoria malarskie i tynkarskie","slug":"akcesoria-malarskie-i-tynkarskie","children":[
      {"name":"Taśmy malarskie","slug":"tasmy-malarskie"},
      {"name":"Folie ochronne budowlane","slug":"folie-ochronne-budowlane"},
      {"name":"Wiadra i pojemniki budowlane","slug":"wiadra-i-pojemniki-budowlane"},
      {"name":"Mieszadła budowlane","slug":"mieszadla-budowlane"}]},
    {"name":"Akcesoria murarskie","slug":"akcesoria-murarskie"},
    {"name":"Artykuły ścierne","slug":"artykuly-scierne","children":[
      {"name":"Kostki ścierne","slug":"kostki-scierne"},
      {"name":"Artykuły ścierne do GK","slug":"artykuly-scierne-do-gk"},
      {"name":"Pilniki","slug":"pilniki"},
      {"name":"Papier ścierny","slug":"papier-scierny"}]},
    {"name":"Narzędzia ręczne","slug":"narzedzia-reczne","children":[
      {"name":"Opalarki i palniki","slug":"opalarki-i-palniki"},
      {"name":"Klucze","slug":"klucze"},
      {"name":"Wkrętaki","slug":"wkretaki"},
      {"name":"Zszywacze","slug":"zszywacze"},
      {"name":"Dłuta","slug":"dluta"}]},
    {"name":"Narzędzia glazurnicze","slug":"narzedzia-glazurnicze","children":[
      {"name":"Krzyżyki do glazury","slug":"krzyzyki-do-glazury"},
      {"name":"Przyrządy do cięcia glazury","slug":"przyrzady-do-ciecia-glazury"}]},
    {"name":"Narzędzia budowlane","slug":"narzedzia-budowlane","children":[
      {"name":"Pace i kielnie","slug":"pace-i-kielnie"},
      {"name":"Szpachle i szpachelki","slug":"szpachle-i-szpachelki"},
      {"name":"Młotki budowlane","slug":"mlotki-budowlane"},
      {"name":"Ołówki i markery budowlane","slug":"olowki-i-markery-budowlane"}]},
    {"name":"Narzędzia malarskie","slug":"narzedzia-malarskie","children":[
      {"name":"Wałki malarskie","slug":"walki-malarskie"},
      {"name":"Pędzle","slug":"pedzle"},
      {"name":"Zestawy malarskie","slug":"zestawy-malarskie"},
      {"name":"Kuwety i kratki malarskie","slug":"kuwety-i-kratki-malarskie"}]},
    {"name":"Elektronarzędzia","slug":"elektronarzedzia","children":[
      {"name":"Akcesoria do elektronarzędzi","slug":"akcesoria-do-elektronarzedzi"},
      {"name":"Wiertarko-wkrętarki","slug":"wiertarko-wkretarki"},
      {"name":"Szlifierki","slug":"szlifierki"},
      {"name":"Piły i pilarki","slug":"pily-i-pilarki"},
      {"name":"Klucze udarowe","slug":"klucze-udarowe"}]},
    {"name":"Narzędzia pomiarowe","slug":"narzedzia-pomiarowe","children":[
      {"name":"Poziomnice","slug":"poziomnice"},
      {"name":"Łaty murarskie","slug":"laty-murarskie"},
      {"name":"Miary i taśmy pomiarowe","slug":"miary-i-tasmy-pomiarowe"},
      {"name":"Lasery i dalmierze","slug":"lasery-i-dalmierze"},
      {"name":"Kątowniki i kątomierze","slug":"katowniki-i-katomierce"}]},
    {"name":"Narzędzia do cięcia","slug":"narzedzia-do-ciecia","children":[
      {"name":"Noże budowlane","slug":"noze-budowlane"},
      {"name":"Obcinaczki","slug":"obcinaczki"},
      {"name":"Przecinaki","slug":"przecinaki"},
      {"name":"Nożyce budowlane","slug":"nozyce-budowlane"},
      {"name":"Piły i brzeszczoty","slug":"pily-i-brzeszczoty"}]}]},
  {"name":"Posadzki i podłogi","slug":"posadzki","sanity_id":"cat-posadzki","children":[
    {"name":"Galanteria betonowa","slug":"galanteria-betonowa","children":[
      {"name":"Kostka brukowa","slug":"kostka-brukowa"},
      {"name":"Palisady, krawężniki i obrzeża","slug":"palisady-krawezniki-i-obrzeza"},
      {"name":"Płyty chodnikowe i tarasowe","slug":"plyty-chodnikowe-i-tarasowe"},
      {"name":"Elementy kanalizacyjne","slug":"elementy-kanalizacyjne"},
      {"name":"Elementy ogrodzenia","slug":"elementy-ogrodzenia"}]},
    {"name":"Nawadnianie","slug":"nawadnianie","children":[
      {"name":"Węże ogrodowe","slug":"weze-ogrodowe"},
      {"name":"Zraszacze","slug":"zraszacze"},
      {"name":"Złączki i końcówki","slug":"zlaczki-i-koncowki"}]},
    {"name":"Stolarka otworowa","slug":"stolarka-otworowa","children":[
      {"name":"Okna i akcesoria do okien","slug":"okna-i-akcesoria-do-okien"},
      {"name":"Drzwi i akcesoria do drzwi","slug":"drzwi-i-akcesoria-do-drzwi"}]},
    {"name":"BHP i bezpieczeństwo","slug":"bhp-i-bezpieczenstwo","children":[
      {"name":"Odzież ochronna","slug":"odziez-ochronna"},
      {"name":"Taśmy ostrzegawcze","slug":"tasmy-ostrzegawcze"},
      {"name":"Paski i kabury narzędziowe","slug":"paski-i-kabury-narzedzia"},
      {"name":"Drabiny","slug":"drabiny"}]}]}
]

print("Drzewo TREE załadowane ✅")
print(f"L1: {len(TREE)}, L2: {sum(len(l1.get('children',[])) for l1 in TREE)}")


Drzewo TREE załadowane ✅
L1: 10, L2: 70


In [47]:

import requests, time

SANITY_PROJECT = "nzcwegq7"
SANITY_DATASET = "production"
SANITY_TOKEN   = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
QUERY_URL  = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/query/{SANITY_DATASET}"
MUTATE_URL = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/mutate/{SANITY_DATASET}"
HDR_R = {"Authorization": f"Bearer {SANITY_TOKEN}"}
HDR_W = {**HDR_R, "Content-Type": "application/json"}

def q(groq):
    r = requests.get(QUERY_URL, params={"query": groq}, headers=HDR_R, timeout=30)
    r.raise_for_status()
    return r.json()["result"]

def run_muts(muts, label, batch=25):
    ok = fail = 0
    for i in range(0, len(muts), batch):
        payload = {"mutations": muts[i:i+batch], "returnIds": True}
        r = requests.post(MUTATE_URL, json=payload, headers=HDR_W,
                          params={"returnIds":"true","visibility":"sync"}, timeout=120)
        if r.ok:
            ok += len(r.json().get("results", []))
        else:
            fail += len(muts[i:i+batch])
            print(f"  ERR {r.status_code}: {r.text[:150]}")
        time.sleep(0.25)
    print(f"  {label}: {ok} OK, {fail} FAIL")
    return ok

# ── Pobierz aktualne kategorie ────────────────────────────────────────────────
all_cats = q('*[_type=="category"]{_id,name,"slug":slug.current,"parentId":parent._ref}')
by_slug  = {c["slug"]: c["_id"] for c in all_cats if c.get("slug")}
by_id    = {c["_id"]: c for c in all_cats}
print(f"Istniejące kategorie: {len(all_cats)}")

# ── L1: createOrReplace + patch ──────────────────────────────────────────────
print("\n=== L1 ===")
l1_muts = []
l1_id   = {}   # slug → sanity _id

for i, l1 in enumerate(TREE):
    sid = l1.get("sanity_id")
    if sid and sid in by_id:
        # Patch istniejącego
        l1_muts.append({"patch": {"id": sid, "set": {
            "name": l1["name"],
            "slug": {"_type":"slug","current": l1["slug"]},
            "order": i+1
        }}})
        l1_id[l1["slug"]] = sid
    else:
        # Utwórz nowy (sufity-podwieszane)
        new_id = l1.get("sanity_id") or f"cat-l1-{l1['slug'][:24]}"
        l1_muts.append({"createOrReplace": {
            "_id": new_id, "_type": "category",
            "name": l1["name"],
            "slug": {"_type":"slug","current": l1["slug"]},
            "order": i+1
        }})
        l1_id[l1["slug"]] = new_id

run_muts(l1_muts, "L1 upsert")

# Odśwież slug map po L1
all_cats = q('*[_type=="category"]{_id,"slug":slug.current}')
by_slug  = {c["slug"]: c["_id"] for c in all_cats if c.get("slug")}
# Uzupełnij l1_id z by_slug (dla nowo utworzonych)
for l1 in TREE:
    if l1["slug"] in by_slug:
        l1_id[l1["slug"]] = by_slug[l1["slug"]]

print(f"L1 id map: {len(l1_id)} wpisów")

# ── L2: createOrReplace per każde L2 ─────────────────────────────────────────
print("\n=== L2 ===")
l2_muts = []
l2_id   = {}  # slug → _id

for i, l1 in enumerate(TREE):
    parent_id = l1_id.get(l1["slug"])
    if not parent_id:
        print(f"  BRAK L1 id dla {l1['slug']}")
        continue
    for j, l2 in enumerate(l1.get("children", [])):
        l2_sid = by_slug.get(l2["slug"])
        doc = {
            "_type": "category",
            "name": l2["name"],
            "slug": {"_type":"slug","current": l2["slug"]},
            "parent": {"_type":"reference","_ref": parent_id},
            "order": j+1
        }
        if l2_sid:
            l2_muts.append({"patch": {"id": l2_sid, "set": {k:v for k,v in doc.items() if k!="_type"}}})
            l2_id[l2["slug"]] = l2_sid
        else:
            new_id = f"cat-l2-{l2['slug'][:28]}"
            l2_muts.append({"createOrReplace": {"_id": new_id, **doc}})
            l2_id[l2["slug"]] = new_id

run_muts(l2_muts, "L2 upsert")

# Odśwież po L2
all_cats = q('*[_type=="category"]{_id,"slug":slug.current}')
by_slug  = {c["slug"]: c["_id"] for c in all_cats if c.get("slug")}
for slug, sid in l2_id.items():
    if slug in by_slug:
        l2_id[slug] = by_slug[slug]

print(f"L2 id map: {len(l2_id)} wpisów")

# ── L3 ────────────────────────────────────────────────────────────────────────
print("\n=== L3 ===")
l3_muts = []
l3_fail = []

for l1 in TREE:
    for l2 in l1.get("children", []):
        parent_id = l2_id.get(l2["slug"]) or by_slug.get(l2["slug"])
        if not parent_id:
            l3_fail.extend([l3["slug"] for l3 in l2.get("children",[])])
            continue
        for k, l3 in enumerate(l2.get("children", [])):
            l3_sid = by_slug.get(l3["slug"])
            doc = {
                "_type": "category",
                "name": l3["name"],
                "slug": {"_type":"slug","current": l3["slug"]},
                "parent": {"_type":"reference","_ref": parent_id},
                "order": k+1
            }
            if l3_sid:
                l3_muts.append({"patch": {"id": l3_sid, "set": {k:v for k,v in doc.items() if k!="_type"}}})
            else:
                new_id = f"cat-l3-{l3['slug'][:28]}"
                l3_muts.append({"createOrReplace": {"_id": new_id, **doc}})

run_muts(l3_muts, "L3 upsert")

if l3_fail:
    print(f"  L3 bez parenta: {len(l3_fail)} → {l3_fail[:5]}")

# ── Weryfikacja ───────────────────────────────────────────────────────────────
print("\n=== WERYFIKACJA ===")
final = q('*[_type=="category"]{_id,"slug":slug.current,"parentId":parent._ref}')
l1c = sum(1 for c in final if not c.get("parentId"))
l2c = sum(1 for c in final if c.get("parentId") and c["parentId"] in {x["_id"] for x in final if not x.get("parentId")})
l3c = len(final) - l1c - l2c
print(f"Łącznie: {len(final)} kategorii")
print(f"  L1: {l1c}  |  L2: {l2c}  |  L3: {l3c}")
print("\n✅ IMPORT ZAKOŃCZONY")


Istniejące kategorie: 149

=== L1 ===


  L1 upsert: 10 OK, 0 FAIL
L1 id map: 10 wpisów

=== L2 ===


  L2 upsert: 70 OK, 0 FAIL
L2 id map: 70 wpisów

=== L3 ===


  L3 upsert: 233 OK, 0 FAIL

=== WERYFIKACJA ===
Łącznie: 393 kategorii
  L1: 10  |  L2: 143  |  L3: 240

✅ IMPORT ZAKOŃCZONY


In [51]:

import json, requests, time

SANITY_PROJECT = "nzcwegq7"
SANITY_DATASET = "production"
SANITY_TOKEN   = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
QU = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/query/{SANITY_DATASET}"
MU = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-01-01/data/mutate/{SANITY_DATASET}"
HR = {"Authorization": f"Bearer {SANITY_TOKEN}"}
HW = {**HR, "Content-Type": "application/json"}

def q(groq, params=None):
    p = {"query": groq}; p.update(params or {})
    r = requests.get(QU, params=p, headers=HR, timeout=30); r.raise_for_status()
    return r.json()["result"]

def mutate(muts, batch=40):
    ok = fail = 0
    for i in range(0, len(muts), batch):
        r = requests.post(MU, json={"mutations": muts[i:i+batch], "returnIds": True},
                          headers=HW, params={"returnIds":"true","visibility":"sync"}, timeout=120)
        if r.ok: ok += len(r.json().get("results",[]))
        else:    fail += len(muts[i:i+batch]); print(f"  ERR {r.status_code}: {r.text[:120]}")
        time.sleep(0.2)
    return ok, fail

# ── 1. Wczytaj bechcicki JSONL → mapa EAN → {cat_main, cat_sub, cat_leaf} ────
BPATH = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/bechcicki_products_all.jsonl"
ean_to_cat = {}  # EAN → {"main": slug, "sub": slug, "leaf": slug}

with open(BPATH) as f:
    for line in f:
        try:
            p = json.loads(line)
            ean = str(p.get("ean","")).strip()
            if not ean or ean in ("0","nan",""): continue
            cat_main = (p.get("category_main") or "").strip().lower().replace(" ","-")
            cat_sub  = (p.get("category_sub") or "").strip().lower().replace(" ","-")
            cat_leaf = (p.get("category_leaf") or "").strip().lower().replace(" ","-")
            if cat_main: ean_to_cat[ean] = {"main": cat_main, "sub": cat_sub or None, "leaf": cat_leaf or None}
        except: pass

print(f"EAN→kategoria z bechcicki: {len(ean_to_cat)}")

# ── 2. Pobierz wszystkie kategorie Sanity → mapa slug→_id ────────────────────
all_cats = q('*[_type=="category"]{_id,"slug":slug.current,"parentId":parent._ref}')
slug_to_id = {c["slug"]: c["_id"] for c in all_cats if c.get("slug")}
l1_ids = {c["_id"] for c in all_cats if not c.get("parentId")}
print(f"Kategorie Sanity: {len(all_cats)}, slugów: {len(slug_to_id)}")

# ── 3. Pobierz produkty Sanity partiami (po 500) ─────────────────────────────
all_prods = []
offset = 0
while True:
    batch = q(f'*[_type=="product"][{offset}...{offset+500}]{{_id,ean,sku,"catId":category._ref}}')
    if not batch: break
    all_prods.extend(batch)
    offset += 500
    if len(batch) < 500: break

print(f"Produkty Sanity: {len(all_prods)}")

# ── 4. Dopasuj EAN → kategoria Sanity ───────────────────────────────────────
def find_best_cat_id(ean):
    """Zwraca Sanity _id najgłębszej pasującej kategorii lub None."""
    info = ean_to_cat.get(ean)
    if not info: return None
    # Próbuj od najgłębszego
    for key in ["leaf", "sub", "main"]:
        slug = info.get(key)
        if slug and slug in slug_to_id:
            return slug_to_id[slug]
    # Fallback: spróbuj normalizeowanego sluga
    for key in ["leaf", "sub", "main"]:
        slug = info.get(key)
        if not slug: continue
        # normalizuj polskie znaki
        norm = slug.replace("ą","a").replace("ć","c").replace("ę","e").replace("ł","l") \
                   .replace("ń","n").replace("ó","o").replace("ś","s").replace("ź","z").replace("ż","z")
        if norm in slug_to_id:
            return slug_to_id[norm]
    return None

mutations = []
stats = {"matched": 0, "already_deep": 0, "no_ean": 0, "no_match": 0, "skip_l1": 0}

for prod in all_prods:
    ean = str(prod.get("ean") or "").strip()
    prod_id = prod["_id"]
    cur_cat = prod.get("catId")

    if not ean:
        stats["no_ean"] += 1
        continue

    new_cat_id = find_best_cat_id(ean)
    if not new_cat_id:
        stats["no_match"] += 1
        continue

    # Pomiń jeśli już w dobrej kategorii
    if new_cat_id == cur_cat:
        stats["already_deep"] += 1
        continue

    # Pomiń jeśli nowa kategoria to L1 (nie chcemy przenosić do L1)
    if new_cat_id in l1_ids:
        stats["skip_l1"] += 1
        continue

    mutations.append({"patch": {"id": prod_id, "set": {
        "category": {"_type": "reference", "_ref": new_cat_id}
    }}})
    stats["matched"] += 1

print(f"\nStatystyki dopasowania:")
for k,v in stats.items(): print(f"  {k}: {v}")
print(f"\nMutacje do wykonania: {len(mutations)}")


EAN→kategoria z bechcicki: 14256
Kategorie Sanity: 393, slugów: 393


Produkty Sanity: 15921

Statystyki dopasowania:
  matched: 10947
  already_deep: 3283
  no_ean: 1665
  no_match: 0
  skip_l1: 26

Mutacje do wykonania: 10947


In [55]:

ok, fail = mutate(mutations, batch=40)
print(f"Wynik: {ok} OK / {fail} FAIL")

# Weryfikacja — policz produkty per kategoria (sample)
sample = q('*[_type=="product"]{_id,"catSlug":category->slug.current,"catParent":category->parent->slug.current}[0...5000]')
in_l2_or_l3 = sum(1 for p in sample if p.get("catParent"))
in_l1_only  = sum(1 for p in sample if p.get("catSlug") and not p.get("catParent"))
print(f"\nSample 5000 produktów:")
print(f"  W L2/L3: {in_l2_or_l3}")
print(f"  W L1:    {in_l1_only}")
print("\n✅ GOTOWE")


Wynik: 10947 OK / 0 FAIL



Sample 5000 produktów:
  W L2/L3: 4763
  W L1:    0

✅ GOTOWE


In [59]:

import requests, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"
DATASET = "production"
BASE = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/{DATASET}"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

def groq(query, params=None):
    p = {"query": query}
    if params:
        p.update({f"${k}": json.dumps(v) for k, v in params.items()})
    r = requests.get(BASE, params=p, headers=HEADERS)
    r.raise_for_status()
    return r.json()["result"]

# 1. Pobierz wszystkie kategorie L2 (mają parent = L1)
all_cats = groq("""
*[_type == "category"] {
  _id, slug, name, "parentId": parent._ref,
  "level": select(
    !defined(parent) => "L1",
    defined(parent->parent) => "L3",
    "L2"
  )
}
""")

l2_cats = [c for c in all_cats if c.get("level") == "L2"]
print(f"Wszystkich L2: {len(l2_cats)}")

# 2. Dla każdego L2 sprawdź ile produktów go używa
l2_ids = [c["_id"] for c in l2_cats]
print(f"IDs L2 do sprawdzenia: {len(l2_ids)}")


Wszystkich L2: 143
IDs L2 do sprawdzenia: 143


In [63]:

# Pobierz wszystkie unikalne category _ref z produktów (wydajnie — bez ładowania całych produktów)
# Zapytanie zwraca płaską listę wszystkich category _ref z całej bazy produktów
cat_refs_raw = groq("""
*[_type == "product"].categories[]._ref
""")

# Unikalne używane category IDs
used_ids = set(r for r in cat_refs_raw if r is not None)
print(f"Używanych kategorii (uniq): {len(used_ids)}")

# Znajdź nieużywane L2
unused_l2 = [c for c in l2_cats if c["_id"] not in used_ids]
print(f"\nNieużywane L2: {len(unused_l2)}")
print("\nPrzykłady (pierwsze 20):")
for c in unused_l2[:20]:
    print(f"  {c['_id']}  {c['slug']}  {c.get('name','?')}")


Używanych kategorii (uniq): 0

Nieużywane L2: 143

Przykłady (pierwsze 20):
  cat-farby-el  {'_type': 'slug', 'current': 'farby-do-drewna'}  Farby do drewna
  cat-farby-wn  {'_type': 'slug', 'current': 'farby-do-metalu'}  Farby do metalu
  cat-gipsy  {'_type': 'slug', 'current': 'bazy-i-koloranty'}  Bazy i koloranty
  cat-grunty  {'_type': 'slug', 'current': 'grunty'}  Grunty
  cat-hydro  {'_type': 'slug', 'current': 'hydroizolacje'}  Hydroizolacje
  cat-impregnaty  {'_type': 'slug', 'current': 'farby-specjalistyczne'}  Farby specjalistyczne
  cat-kleje  {'_type': 'slug', 'current': 'kleje'}  Kleje
  cat-l2-akcesoria-do-izolacji  {'_type': 'slug', 'current': 'akcesoria-do-izolacji'}  Akcesoria do izolacji
  cat-l2-akcesoria-malarskie-i-tynkar  {'_type': 'slug', 'current': 'akcesoria-malarskie-i-tynkarskie'}  Akcesoria malarskie i tynkarskie
  cat-l2-akcesoria-murarskie  {'_type': 'slug', 'current': 'akcesoria-murarskie'}  Akcesoria murarskie
  cat-l2-artykuly-scierne  {'_type': 'slug',

In [67]:

# Sprawdź jak wyglądają pola produktu
sample = groq("""*[_type == "product"][0..2] { _id, _type, "keys": keys(@) }""")
for s in sample:
    print(s)


HTTPError: 400 Client Error: Bad Request for url: https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production?query=%2A%5B_type+%3D%3D+%22product%22%5D%5B0..2%5D+%7B+_id%2C+_type%2C+%22keys%22%3A+keys%28%40%29+%7D

In [71]:

sample = groq('*[_type == "product"][0..1]')
for s in sample:
    print(json.dumps(s, indent=2, ensure_ascii=False)[:2000])
    print("---")


{
  "_createdAt": "2026-05-28T10:01:31Z",
  "_id": "prod-grunt-ceresit-ct16",
  "_rev": "tYwdV2ldLdzJB4kwq09j7S",
  "_type": "product",
  "_updatedAt": "2026-05-29T00:25:47Z",
  "brand": "Ceresit",
  "category": {
    "_ref": "cat-grunty",
    "_type": "reference"
  },
  "featured": false,
  "inStock": true,
  "name": "Grunt głęboko penetrujący Ceresit CT 16 10L",
  "shortDescription": "Grunt głęboko penetrujący wzmacniający chłonne podłoża mineralne przed tynkowaniem, klejeniem i malowaniem.",
  "sku": "CRS-CT16-10L",
  "slug": {
    "_type": "slug",
    "current": "grunt-gleboko-penetrujacy-ceresit-ct16-10l"
  },
  "specs": [
    {
      "_key": "Wydajność",
      "key": "Wydajność",
      "value": "100–200 ml/m²"
    },
    {
      "_key": "Rozcieńczenie",
      "key": "Rozcieńczenie",
      "value": "1:3 wodą"
    },
    {
      "_key": "Czas-schnięcia",
      "key": "Czas schnięcia",
      "value": "1–4 h"
    },
    {
      "_key": "Temperatura-aplikacji",
      "key": "Temperatu

In [75]:

# Produkty mają `category._ref` (nie `categories[]._ref`)
cat_refs_raw = groq('*[_type == "product"].category._ref')
used_ids = set(r for r in cat_refs_raw if r is not None)
print(f"Używanych kategorii (uniq): {len(used_ids)}")

# Nieużywane L2
unused_l2 = [c for c in l2_cats if c["_id"] not in used_ids]
used_l2   = [c for c in l2_cats if c["_id"] in used_ids]

print(f"Używanych L2:    {len(used_l2)}")
print(f"Nieużywanych L2: {len(unused_l2)}")

print("\n=== NIEUŻYWANE L2 (do usunięcia) ===")
for c in sorted(unused_l2, key=lambda x: x.get("name","?")):
    name = c.get("name", "?")
    slug = c.get("slug",{}).get("current","?") if isinstance(c.get("slug"), dict) else c.get("slug","?")
    print(f"  {c['_id']:45s}  {slug:45s}  {name}")


Używanych kategorii (uniq): 226
Używanych L2:    100
Nieużywanych L2: 43

=== NIEUŻYWANE L2 (do usunięcia) ===
  cat-l2-bhp-i-bezpieczenstwo                    bhp-i-bezpieczenstwo                           BHP i bezpieczeństwo
  cat-l2-dachy-zielone                           dachy-zielone                                  Dachy zielone
  cat-l2-dodatki-do-zapraw-i-betonu              dodatki-do-zapraw-i-betonu                     Dodatki do zapraw i betonu
  cat-l2-elektronarzedzia                        elektronarzedzia                               Elektronarzędzia
  cat-l2-elementy-mocujace                       elementy-mocujace                              Elementy mocujące
  cat-farby-wn                                   farby-do-metalu                                Farby do metalu
  cat-l2-farby-elewacyjne                        farby-elewacyjne                               Farby elewacyjne
  cat-impregnaty                                 farby-specjalistyczne                 

In [79]:

unused_l2_ids = [c["_id"] for c in unused_l2]

# Znajdź L3 które mają parent w liście nieużywanych L2
l3_of_unused = groq("""
*[_type == "category" && parent._ref in $ids] {
  _id, name, slug, "parentId": parent._ref,
  "productCount": count(*[_type == "product" && category._ref == ^._id])
}
""", {"ids": unused_l2_ids})

print(f"L3 dzieci nieużywanych L2: {len(l3_of_unused)}")
l3_with_products = [c for c in l3_of_unused if c.get("productCount", 0) > 0]
l3_empty = [c for c in l3_of_unused if c.get("productCount", 0) == 0]
print(f"  - z produktami: {len(l3_with_products)}")
print(f"  - bez produktów: {len(l3_empty)}")

if l3_with_products:
    print("\n⚠️ L3 z produktami (NIE USUWAĆ rodzica!):")
    for c in l3_with_products:
        print(f"  {c['_id']}  {c.get('name','?')}  products={c['productCount']}")


L3 dzieci nieużywanych L2: 155
  - z produktami: 84
  - bez produktów: 71

⚠️ L3 z produktami (NIE USUWAĆ rodzica!):
  cat-kleje-glaz  Kleje do glazury  products=3
  cat-kleje-mont  Kleje montażowe  products=72
  cat-kleje-sty  Kleje do styropianu  products=3
  cat-kleje-welna  Kleje do wełen  products=1
  cat-l3-akcesoria-do-kominow  Akcesoria do kominów  products=113
  cat-l3-belki-stropowe-ceramiczne  Belki stropowe ceramiczne  products=5
  cat-l3-bloczki  Bloczki  products=83
  cat-l3-drzwi-kolankowe  Drzwi kolankowe  products=15
  cat-l3-emalie-chlorokauczukowe  Emalie chlorokauczukowe  products=169
  cat-l3-emalie-poliuretanowe  Emalie poliuretanowe  products=11
  cat-l3-farby-elewacyjne-akrylowe  Farby elewacyjne akrylowe  products=297
  cat-l3-farby-elewacyjne-emulsyjne  Farby elewacyjne emulsyjne  products=76
  cat-l3-farby-elewacyjne-silikatowe  Farby elewacyjne silikatowe  products=300
  cat-l3-farby-elewacyjne-silikonowe  Farby elewacyjne silikonowe  products=287
  cat-l3-f

In [83]:

# L2 które mają L3 z produktami → NIE możemy ich usunąć (są parentem aktywnych L3)
l2_with_active_l3 = set(c["parentId"] for c in l3_of_unused if c.get("productCount", 0) > 0)
print(f"Nieużywane L2 które mają aktywne L3 dzieci: {len(l2_with_active_l3)}")
print("(te L2 trzeba ZOSTAWIĆ — są strukturalnymi parentami używanych L3)")

# L2 bezpieczne do usunięcia = nieużywane L2 BEZ aktywnych L3 dzieci
safe_to_delete_l2 = [c for c in unused_l2 if c["_id"] not in l2_with_active_l3]
print(f"\nL2 bezpiecznie usuwalne: {len(safe_to_delete_l2)}")

# Ich puste L3 dzieci też do usunięcia
safe_l2_ids = [c["_id"] for c in safe_to_delete_l2]
orphan_l3 = [c for c in l3_of_unused if c["parentId"] in safe_l2_ids and c.get("productCount", 0) == 0]
print(f"Puste L3 pod tymi L2 (też do usunięcia): {len(orphan_l3)}")

print("\n=== L2 do usunięcia ===")
for c in sorted(safe_to_delete_l2, key=lambda x: x.get("name","?")):
    name = c.get("name","?")
    slug = c.get("slug",{}).get("current","?") if isinstance(c.get("slug"),dict) else c.get("slug","?")
    print(f"  {c['_id']:45s}  {name}")

print("\n=== L3 (puste) do usunięcia razem ===")
for c in orphan_l3:
    print(f"  {c['_id']:45s}  {c.get('name','?')}")


Nieużywane L2 które mają aktywne L3 dzieci: 29
(te L2 trzeba ZOSTAWIĆ — są strukturalnymi parentami używanych L3)

L2 bezpiecznie usuwalne: 14
Puste L3 pod tymi L2 (też do usunięcia): 30

=== L2 do usunięcia ===
  cat-l2-bhp-i-bezpieczenstwo                    BHP i bezpieczeństwo
  cat-l2-dachy-zielone                           Dachy zielone
  cat-l2-dodatki-do-zapraw-i-betonu              Dodatki do zapraw i betonu
  cat-l2-fotowoltaika                            Fotowoltaika
  cat-l2-narzedzia-do-ciecia                     Narzędzia do cięcia
  cat-l2-narzedzia-reczne                        Narzędzia ręczne
  cat-l2-nawadnianie                             Nawadnianie
  cat-profile                                    Preparaty czyszczące powierzchnie
  cat-l2-plyty-sufitowe                          Płyty sufitowe
  cat-l2-rewizje-i-klapy                         Rewizje i klapy
  cat-l2-stolarka-otworowa                       Stolarka otworowa
  cat-l2-tasmy-do-suchej-zabudowy         

In [87]:

import time

MUTATE_URL = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/mutate/{DATASET}"

def delete_docs(ids, label=""):
    mutations = [{"delete": {"id": _id}} for _id in ids]
    r = requests.post(MUTATE_URL, headers={**HEADERS, "Content-Type": "application/json"},
                      json={"mutations": mutations})
    if r.ok:
        print(f"✅ {label}: usunięto {len(ids)} dokumentów")
    else:
        print(f"❌ {label}: {r.status_code} {r.text[:200]}")
    return r.ok

# Krok 1: Usuń puste L3 (najpierw dzieci, potem parenty)
l3_ids = [c["_id"] for c in orphan_l3]
delete_docs(l3_ids, "Puste L3")

time.sleep(1)

# Krok 2: Usuń bezpieczne L2
l2_ids_to_delete = [c["_id"] for c in safe_to_delete_l2]
delete_docs(l2_ids_to_delete, "Bezpieczne L2")

print(f"\nPodsumowanie:")
print(f"  Usunięto L3 (puste): {len(l3_ids)}")
print(f"  Usunięto L2:         {len(l2_ids_to_delete)}")
print(f"  Razem:               {len(l3_ids) + len(l2_ids_to_delete)} dokumentów")


✅ Puste L3: usunięto 30 dokumentów


✅ Bezpieczne L2: usunięto 14 dokumentów

Podsumowanie:
  Usunięto L3 (puste): 30
  Usunięto L2:         14
  Razem:               44 dokumentów


In [91]:

import requests
from bs4 import BeautifulSoup
import re

headers_web = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
r = requests.get("https://www.bechcicki.pl", headers=headers_web, timeout=15)
soup = BeautifulSoup(r.text, "html.parser")

# Znajdź nav / menu z kategoriami
nav = soup.find("nav") or soup.find("ul", class_=re.compile(r"menu|nav|category", re.I))
print("Status:", r.status_code)
print("Nav found:", nav is not None)

# Zbierz wszystkie linki kategorii
links = soup.find_all("a", href=re.compile(r"^/[a-z]"))
cat_links = [a for a in links if len(a.get("href","").strip("/").split("/")) >= 1 and a.text.strip()]
print(f"Linków z kategoriami: {len(cat_links)}")

# Zbierz unikalne ścieżki
paths = set()
for a in cat_links:
    href = a.get("href","")
    if href.startswith("/") and len(href) > 2:
        parts = href.strip("/").split("/")
        if 1 <= len(parts) <= 4:
            paths.add(href)

print(f"Unikalnych ścieżek: {len(paths)}")
for p in sorted(paths)[:30]:
    print(" ", p)


Status: 200
Nav found: True
Linków z kategoriami: 542
Unikalnych ścieżek: 381
  /account/rfqs-offers/rfqs
  /bhp
  /blog/program-dla-redystrybucji-druzynaab
  /blog/program-dla-wykonawcow-druzynaab
  /blog?tag=aktualnosci
  /blog?tag=blog
  /blog?tag=nagrody-i-wyroznienia
  /calculator
  /chemia-budowlana
  /chemia-budowlana/dodatki-do-zapraw-i-betonu
  /chemia-budowlana/gipsy-i-gladzie
  /chemia-budowlana/gipsy-i-gladzie/gipsy-budowlane
  /chemia-budowlana/gipsy-i-gladzie/gipsy-szpachlowe
  /chemia-budowlana/gipsy-i-gladzie/gladzie-gipsowe-w-proszku
  /chemia-budowlana/gipsy-i-gladzie/gladzie-masy-gotowe
  /chemia-budowlana/gipsy-i-gladzie/kleje-gipsowe
  /chemia-budowlana/grunty
  /chemia-budowlana/grunty/grunty-do-posadzek
  /chemia-budowlana/grunty/grunty-pod-tynki
  /chemia-budowlana/grunty/grunty-specjalistyczne
  /chemia-budowlana/grunty/grunty-uniwersalne
  /chemia-budowlana/grunty/masy-bitumiczne-gruntujace
  /chemia-budowlana/kleje
  /chemia-budowlana/kleje/kleje-do-drewna
  

In [95]:

# Zbierz tylko ścieżki kategorii (bez blog, account, itp.)
EXCLUDE = {"blog", "account", "calculator", "bhp", "kontakt", "o-nas", "koszyk", "logowanie", "rejestracja"}

cat_paths = []
for a in cat_links:
    href = a.get("href","").strip()
    if not href.startswith("/"):
        continue
    parts = href.strip("/").split("/")
    # Odfiltruj nie-kategorie
    if parts[0] in EXCLUDE or "?" in href or "#" in href:
        continue
    if 1 <= len(parts) <= 4:
        cat_paths.append((href, a.text.strip(), len(parts)))

# Unikalne (href → name)
seen = {}
for href, name, lvl in cat_paths:
    if href not in seen and name:
        seen[href] = (name, lvl)

print(f"Kategorii razem: {len(seen)}")

# Podziel na poziomy
l1 = {k: v for k, v in seen.items() if v[1] == 1}
l2 = {k: v for k, v in seen.items() if v[1] == 2}
l3 = {k: v for k, v in seen.items() if v[1] == 3}
l4 = {k: v for k, v in seen.items() if v[1] == 4}

print(f"L1: {len(l1)}, L2: {len(l2)}, L3: {len(l3)}, L4: {len(l4)}")

print("\n=== BECHCICKI L1 ===")
for href, (name, _) in sorted(l1.items()):
    print(f"  {href:40s}  {name}")


Kategorii razem: 371
L1: 40, L2: 72, L3: 257, L4: 2

=== BECHCICKI L1 ===
  /chemia-budowlana                         Chemia budowlana
  /dachy                                    Dachy
  /farby-i-rozpuszczalniki                  Farby i rozpuszczalniki
  /farbyilakiery                            Farby i lakiery
  /formy-dostawy                            Dostawa już w 24 h
  /formy-platnosci                          Formy płatności
  /izolacje                                 Izolacje
  /jak-poruszac-sie-po-serwisie             Pomoc
  /kariera                                  Kariera
  /korzysci-z-rejestracji                   Korzyści z rejestracji
  /kredyt-kupiecki                          Dostępny limit kredytowy
  /lampa-podlogowa-goodhome-mantus-3-x-g9-chrom-id-bud_54548575  Lampa podłogowa GoodHome Mantus 3 x G9 chrom 
Szary
89,00 zł
Dostępne 0 szt.
Wysyłamy w 48h
  /lampa-wiszaca-luxolar-382-e27-bialy-id-bud_54548574  Lampa wisząca Luxolar 382 E27 biały 
Biały
89,00 zł
Dostępne

In [99]:

import requests
from bs4 import BeautifulSoup
import json

headers_web = {"User-Agent": "Mozilla/5.0"}
r = requests.get("https://www.bechcicki.pl", headers=headers_web, timeout=15)
soup = BeautifulSoup(r.text, "html.parser")

# Prawdziwe L1 kategorie — znane z poprzedniej sesji + to co widać w URL-ach z L2/L3
# Filtrujemy na podstawie tego: L2 zaczynają się /l1-slug/l2-slug
# Zbierz wszystkie href które mają dokładnie 1, 2 lub 3 segmenty

KNOWN_L1 = {
    "farby-i-rozpuszczalniki", "chemia-budowlana", "izolacje",
    "sucha-zabudowa", "sufity-podwieszane", "plytki",
    "dachy", "stropy-i-sciany", "narzedzia-i-mocowania",
    "posadzki-i-podlogi", "oswietlenie", "pozostale"
}

all_links = soup.find_all("a", href=True)
cat_map = {}  # slug_path -> name

for a in all_links:
    href = a.get("href","").strip()
    name = a.text.strip().split("\n")[0].strip()
    if not href.startswith("/") or not name or len(name) > 80:
        continue
    parts = href.strip("/").split("/")
    if not parts or not parts[0]:
        continue
    # Musi zaczynać się od L1 który znamy
    if parts[0] not in KNOWN_L1:
        continue
    if "?" in href or "#" in href or "id-bud" in href:
        continue
    lvl = len(parts)
    if 1 <= lvl <= 3:
        cat_map[href] = (name, lvl)

# Podsumowanie
l1b = {k: v for k, v in cat_map.items() if v[1] == 1}
l2b = {k: v for k, v in cat_map.items() if v[1] == 2}
l3b = {k: v for k, v in cat_map.items() if v[1] == 3}

print(f"BECHCICKI: L1={len(l1b)}, L2={len(l2b)}, L3={len(l3b)}")
print("\n=== L1 ===")
for href, (name,_) in sorted(l1b.items()):
    print(f"  {href:45s}  {name}")

print("\n=== L2 (farby-i-rozpuszczalniki) ===")
for href, (name,_) in sorted(l2b.items()):
    if "farby" in href:
        print(f"  {href:60s}  {name}")


BECHCICKI: L1=11, L2=71, L3=237

=== L1 ===
  /chemia-budowlana                              Chemia budowlana
  /dachy                                         Dachy
  /farby-i-rozpuszczalniki                       Farby i rozpuszczalniki
  /izolacje                                      Izolacje
  /narzedzia-i-mocowania                         Narzędzia i mocowania
  /oswietlenie                                   Oświetlenie
  /plytki                                        Płytki
  /pozostale                                     Pozostałe
  /stropy-i-sciany                               Stropy i ściany
  /sucha-zabudowa                                Sucha zabudowa
  /sufity-podwieszane                            Sufity podwieszane

=== L2 (farby-i-rozpuszczalniki) ===
  /farby-i-rozpuszczalniki/bazy-i-koloranty                     Bazy i koloranty
  /farby-i-rozpuszczalniki/farby-do-drewna                      Farby do drewna
  /farby-i-rozpuszczalniki/farby-do-metalu                   

In [103]:

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"; DATASET = "production"
BASE = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/{DATASET}"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

def groq(query, params=None):
    p = {"query": query}
    if params:
        p.update({f"${k}": json.dumps(v) for k, v in params.items()})
    r2 = requests.get(BASE, params=p, headers=HEADERS)
    r2.raise_for_status()
    return r2.json()["result"]

# Bechcicki - pełne drzewo
bech_tree = {}  # slug -> {name, parent_slug, level}
for href, (name, lvl) in cat_map.items():
    parts = href.strip("/").split("/")
    slug = parts[-1]
    parent = parts[-2] if lvl > 1 else None
    bech_tree[href] = {"name": name, "slug": slug, "parent": parent, "level": lvl, "href": href}

print(f"Bechcicki razem: {len(bech_tree)} ({len(l1b)} L1, {len(l2b)} L2, {len(l3b)} L3)")

# Sanity - pobierz wszystkie kategorie
sanity_cats = groq("""
*[_type == "category"] {
  _id, name,
  "slug": slug.current,
  "parentSlug": parent->slug.current,
  "parentId": parent._ref,
  "level": select(!defined(parent) => "L1", defined(parent->parent) => "L3", "L2")
}
""")

sanity_slugs = {c["slug"]: c for c in sanity_cats}
sanity_by_level = {}
for c in sanity_cats:
    lvl = c["level"]
    sanity_by_level.setdefault(lvl, []).append(c)

print(f"\nSanity razem: {len(sanity_cats)} ({len(sanity_by_level.get('L1',[]))} L1, {len(sanity_by_level.get('L2',[]))} L2, {len(sanity_by_level.get('L3',[]))} L3)")

# Porównaj L1
bech_l1_slugs = {href.strip("/") for href in l1b}
sanity_l1_slugs = {c["slug"] for c in sanity_by_level.get("L1", [])}

print("\n=== RÓŻNICE L1 ===")
missing_in_sanity_l1 = bech_l1_slugs - sanity_l1_slugs
extra_in_sanity_l1 = sanity_l1_slugs - bech_l1_slugs
print(f"Brak w Sanity: {missing_in_sanity_l1}")
print(f"Nadmiarowe w Sanity: {extra_in_sanity_l1}")

# Porównaj L2
bech_l2_slugs = {href.strip("/").split("/")[-1] for href in l2b}
sanity_l2_slugs = {c["slug"] for c in sanity_by_level.get("L2", [])}
missing_l2 = bech_l2_slugs - sanity_l2_slugs
extra_l2 = sanity_l2_slugs - bech_l2_slugs
print(f"\n=== RÓŻNICE L2 ===")
print(f"Brak w Sanity ({len(missing_l2)}): {sorted(missing_l2)}")
print(f"\nNadmiarowe w Sanity ({len(extra_l2)}): {sorted(extra_l2)}")

# Porównaj L3
bech_l3_slugs = {href.strip("/").split("/")[-1] for href in l3b}
sanity_l3_slugs = {c["slug"] for c in sanity_by_level.get("L3", [])}
missing_l3 = bech_l3_slugs - sanity_l3_slugs
extra_l3 = sanity_l3_slugs - bech_l3_slugs
print(f"\n=== RÓŻNICE L3 ===")
print(f"Brak w Sanity ({len(missing_l3)}): {sorted(missing_l3)}")
print(f"\nNadmiarowe w Sanity ({len(extra_l3)}): {sorted(extra_l3)}")


Bechcicki razem: 319 (11 L1, 71 L2, 237 L3)

Sanity razem: 349 (10 L1, 129 L2, 210 L3)

=== RÓŻNICE L1 ===
Brak w Sanity: {'oswietlenie', 'pozostale'}
Nadmiarowe w Sanity: {'posadzki'}

=== RÓŻNICE L2 ===
Brak w Sanity (22): ['bhp', 'dachy-zielone', 'dodatki-do-zapraw-i-betonu', 'elementy-mocujace-uniwersalne', 'folie', 'fotowoltaika', 'listwy-i-akcesoria', 'narzedzia-do-ciecia', 'narzedzia-i-akcesoria-glazurnicze', 'narzedzia-pozostale', 'narzedzia-reczne', 'nawadnianie', 'plyty', 'plyty-sufitowe', 'preparaty-do-chemicznego-oczyszczania-powierzchni', 'rewizje', 'rynny', 'srodki-czyszczaco-pielegnacyjne', 'stolarka-otworowa', 'tasmy-do-suchej-zabudowy', 'welny', 'zamocowania-dachowe']

Nadmiarowe w Sanity (80): ['akcesoria-izolacji', 'akcesoria-rynny', 'bloczek-komorkowy', 'bloczki-beton-komorkowy', 'bloczki-silikatowe', 'cegly', 'cement', 'ceramika-budowlana', 'elementy-mocujace', 'farby-biale', 'farby-elaw-akrylowe', 'farby-elaw-emulsyjne', 'farby-elaw-silikatowe', 'farby-elaw-siliko

In [107]:

import requests, json
from difflib import SequenceMatcher

# =================== BECHCICKI - pełne drzewo ===================
# Już mamy cat_map, l1b, l2b, l3b z poprzednich komórek

# Zbuduj słownik: bech_slug -> {name, level, parent_slug, href}
bech_by_slug = {}
for href, (name, lvl) in cat_map.items():
    parts = href.strip("/").split("/")
    slug = parts[-1]
    parent = parts[-2] if lvl > 1 else None
    l1_slug = parts[0]
    bech_by_slug[slug] = {
        "name": name, "level": lvl,
        "parent_slug": parent, "l1_slug": l1_slug,
        "href": href
    }

# Bechcicki name → slug (dla dopasowania po nazwie)
bech_name_to_slug = {v["name"].lower(): k for k, v in bech_by_slug.items()}

# =================== SANITY - wszystkie kategorie ===================
TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"; DATASET = "production"
BASE = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/{DATASET}"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

def groq(query, params=None):
    p = {"query": query}
    if params:
        p.update({f"${k}": json.dumps(v) for k, v in params.items()})
    r2 = requests.get(BASE, params=p, headers=HEADERS)
    r2.raise_for_status()
    return r2.json()["result"]

sanity_cats = groq("""
*[_type == "category"] {
  _id, name,
  "slug": slug.current,
  "parentSlug": parent->slug.current,
  "parentId": parent._ref,
  "parentParentSlug": parent->parent->slug.current,
  "level": select(!defined(parent) => "L1", defined(parent->parent) => "L3", "L2"),
  "productCount": count(*[_type == "product" && category._ref == ^._id])
}
""")

sanity_by_slug = {c["slug"]: c for c in sanity_cats}
sanity_by_id   = {c["_id"]:  c for c in sanity_cats}
sanity_name_to_slug = {c["name"].lower(): c["slug"] for c in sanity_cats}

print(f"Sanity: {len(sanity_cats)} kategorii")
print(f"Bechcicki: {len(bech_by_slug)} kategorii")

# =================== PLAN ZMIAN ===================
to_patch_slug = []   # (sanity_id, sanity_old_slug, bech_slug, name)  — zmień slug w Sanity
to_create     = []   # bech_slug → nowe kategorie do dodania
matched_bech  = set()  # bech slugi które mają odpowiednik w Sanity
unmatched_sanity = []  # Sanity kategorie bez odpowiednika w bechcicki

for s_cat in sanity_cats:
    s_slug = s_cat["slug"]
    s_name = s_cat["name"].lower()
    
    if s_slug in bech_by_slug:
        # Dokładne dopasowanie slug → OK
        matched_bech.add(s_slug)
    elif s_name in bech_name_to_slug:
        # Dopasowanie po nazwie → zmień slug
        b_slug = bech_name_to_slug[s_name]
        if b_slug not in {x[2] for x in to_patch_slug}:
            to_patch_slug.append((s_cat["_id"], s_slug, b_slug, s_cat["name"]))
        matched_bech.add(b_slug)
    else:
        unmatched_sanity.append(s_cat)

for b_slug, b_data in bech_by_slug.items():
    if b_slug not in matched_bech:
        to_create.append(b_slug)

print(f"\nPlan zmian:")
print(f"  PATCH slug (zła nazwa→dobra): {len(to_patch_slug)}")
print(f"  CREATE nowe:                  {len(to_create)}")
print(f"  Bez odpowiednika w bechcicki: {len(unmatched_sanity)}")

print("\n=== PATCH slug (pierwsze 30) ===")
for sid, old, new, name in sorted(to_patch_slug, key=lambda x: x[1])[:30]:
    print(f"  {old:45s} → {new:45s}  ({name})")

print("\n=== CREATE brakujące (pierwsze 20) ===")
for slug in sorted(to_create)[:20]:
    b = bech_by_slug[slug]
    print(f"  L{b['level']} {slug:45s}  {b['name']}")


Sanity: 349 kategorii
Bechcicki: 319 kategorii

Plan zmian:
  PATCH slug (zła nazwa→dobra): 5
  CREATE nowe:                  99
  Bez odpowiednika w bechcicki: 127

=== PATCH slug (pierwsze 30) ===
  katowniki-i-katomierce                        → katowniki-i-katomierze                         (Kątowniki i kątomierze)
  kleje-glazura                                 → kleje-do-glazury                               (Kleje do glazury)
  kleje-welna                                   → kleje-do-welen                                 (Kleje do wełen)
  piany-montazowe-wezzykowe                     → piany-montazowe-wezykowe                       (Piany montażowe wężykowe)
  tasmy-uszczelniajace-hydroizolacje            → tasmy-uszczelniajace-do-hydroizolacji          (Taśmy uszczelniające do hydroizolacji)

=== CREATE brakujące (pierwsze 20) ===
  L3 akcesoria-do-plotkow-przeciwsniegowe           Akcesoria do płotków przeciwśniegowe
  L3 akcesoria-do-systemow-rynnowych                Akcesor

In [111]:

# Unmatched Sanity cats — nazwy i produkty
print("=== UNMATCHED SANITY (z produktami > 0) ===")
with_products = [(c["slug"], c["name"], c.get("productCount",0), c["level"]) 
                 for c in unmatched_sanity if c.get("productCount",0) > 0]
without_products = [c for c in unmatched_sanity if c.get("productCount",0) == 0]

print(f"Z produktami: {len(with_products)}, bez produktów: {len(without_products)}")
for slug, name, cnt, lvl in sorted(with_products, key=lambda x: -x[2])[:40]:
    print(f"  L{lvl}  {slug:50s}  {name:40s}  {cnt} prod")

# Pokaż też nazwy tych bez produktów
print(f"\n=== UNMATCHED BEZ PRODUKTÓW (do usunięcia, {len(without_products)} szt.) ===")
for c in sorted(without_products, key=lambda x: x["slug"])[:30]:
    print(f"  {c['level']}  {c['slug']:50s}  {c['name']}")


=== UNMATCHED SANITY (z produktami > 0) ===
Z produktami: 84, bez produktów: 43
  LL2  mozaiki                                             Mozaiki                                   49 prod
  LL3  kleje-styropian                                     Kleje do styropianu                       3 prod
  LL3  rynny-pvc                                           Rynny PVC                                 3 prod
  LL2  styropian-fasadowy-eps                              Styropian Fasadowy Eps                    3 prod
  LL2  welna-fasadowa                                      Welna Fasadowa                            3 prod
  LL3  tynki-akrylowe                                      Tynki elewacyjne akrylowe                 2 prod
  LL3  tynki-silikonowe                                    Tynki elewacyjne silikonowe               2 prod
  LL3  zaprawy-murarskie                                   Zaprawy murarskie                         2 prod
  LL2  bloczki-beton-komorkowy                         

In [115]:

# Stare Sanity z produktami + fuzzy match do bechcicki
# Normalizacja: lowercase, usuń spacje

def normalize(s):
    return s.lower().strip()

def similarity(a, b):
    return SequenceMatcher(None, normalize(a), normalize(b)).ratio()

# Bechcicki name → slug lookup (lowercase)
bech_name_lower = {normalize(v["name"]): k for k, v in bech_by_slug.items()}
bech_names = list(bech_name_lower.keys())

# Dla każdej starej Sanity cat z produktami — znajdź najlepsze dopasowanie w bechcicki
old_to_new = {}  # sanity_slug → bech_slug

# Ręczne mapowania dla przypadków które fuzzy nie złapie (stare slug → bechcicki slug)
MANUAL_MAP = {
    # Farby
    "farby-elaw-akrylowe":   "farby-elewacyjne-akrylowe",
    "farby-elaw-emulsyjne":  "farby-elewacyjne-emulsyjne",
    "farby-elaw-silikatowe": "farby-elewacyjne-silikatowe",
    "farby-elaw-silikonowe": "farby-elewacyjne-silikonowe",
    "farby-biale":           "farby-wewnetrzne-biale",
    "farby-kolorowe":        "farby-wewnetrzne-kolorowe",
    # Styropian
    "styropian":                  "styropiany-fasadowe-eps",
    "styropian-fasadowy-eps":     "styropiany-fasadowe-eps",
    # Wełny
    "welna-fasadowa":             "welny-fasadowe",
    "welna-mineralna":            "welny-fasadowe",
    # Tynki
    "tynki-akrylowe":             "tynki-elewacyjne-akrylowe",
    "tynki-mineralne":            "tynki-elewacyjne-mineralne",
    "tynki-silikatowe":           "tynki-elewacyjne-silikatowe",
    "tynki-silikonowe":           "tynki-elewacyjne-silikonowe",
    # Kleje
    "kleje-styropian":            "kleje-do-styropianu-i-styroduru",
    "kleje-plytek":               "kleje-do-glazury",
    # Rynny
    "rynny-pvc":                  "systemy-rynnowe-pvc",
    # Mozaiki
    "mozaiki":                    "plytki-mozaikowe",
    # Zaprawy
    "zaprawy-murarskie":          "zaprawy-murarskie-ogolnego-zastosowania",
    "zaprawy-posadzkowe":         "zaprawy-posadzkowe-masy-samopoziomujace",
    # Inne
    "fugi":                       "fugi-i-uszczelniacze",
    "piany-pistoletowe":          "piany-montazowe",
    "bloczek-komorkowy":          "bloczki-z-betonu-komorkowego",
    "bloczki-beton-komorkowy":    "bloczki-z-betonu-komorkowego",
    "bloczki-silikatowe":         "bloczki-silikatowe",  # może być OK
    "nadproza":                   "nadproza-prefabrykowane",
    "okna-dachowe-std":           "okna-dachowe",
    "pokrycia-blacha":            "blacha-dachowa",
    "akcesoria-izolacji":         "akcesoria-do-izolacji",
    "akcesoria-rynny":            "akcesoria-do-systemow-rynnowych",
    "ceramika-budowlana":         "cegly-i-ceramika",
    "cegly":                      "cegly-i-ceramika",
    "cement":                     "cementy",
    "folie-w-plynie":             "hydroizolacje-w-plynie",
    "impregnaty-drewno":          "impregnaty-do-drewna",
    "kolki-wkrety-uniwersalne":   "kolki-i-wkrety-uniwersalne",
    "lakiery-drewno":             "lakiery-do-drewna",
    "profile-sufit":              "profile-sufitowe",
    "pustaki-ceramiczne":         "pustaki",
    "wylewki-cementowe":          "wylewki-i-jastrychy",
}

# Weryfikuj mapowania — sprawdź czy cel istnieje w bechcicki
print("=== WERYFIKACJA MANUAL MAP ===")
valid_manual = {}
invalid_manual = []
for s_slug, b_slug in MANUAL_MAP.items():
    if b_slug in bech_by_slug:
        valid_manual[s_slug] = b_slug
    else:
        invalid_manual.append((s_slug, b_slug))
        
print(f"Poprawne mapowania: {len(valid_manual)}")
print(f"Nieprawidłowe (cel nie istnieje w bechcicki): {len(invalid_manual)}")
for s, b in invalid_manual:
    # Znajdź podobne
    best = max(bech_names, key=lambda x: similarity(b.replace("-"," "), x))
    print(f"  {s} → {b}  BRAK  (podobne: {best} → {bech_name_lower[best]})")


=== WERYFIKACJA MANUAL MAP ===
Poprawne mapowania: 22
Nieprawidłowe (cel nie istnieje w bechcicki): 18
  tynki-akrylowe → tynki-elewacyjne-akrylowe  BRAK  (podobne: farby elewacyjne akrylowe → farby-elewacyjne-akrylowe)
  tynki-mineralne → tynki-elewacyjne-mineralne  BRAK  (podobne: tynki elewacyjne → tynki-elewacyjne)
  tynki-silikatowe → tynki-elewacyjne-silikatowe  BRAK  (podobne: farby elewacyjne silikatowe → farby-elewacyjne-silikatowe)
  tynki-silikonowe → tynki-elewacyjne-silikonowe  BRAK  (podobne: farby elewacyjne silikonowe → farby-elewacyjne-silikonowe)
  mozaiki → plytki-mozaikowe  BRAK  (podobne: opaski zaciskowe → opaski-zaciskowe)
  fugi → fugi-i-uszczelniacze  BRAK  (podobne: zaprawy uszczelniające → zaprawy-uszczelniajace)
  bloczek-komorkowy → bloczki-z-betonu-komorkowego  BRAK  (podobne: belki stropowe betonowe → belki-stropowe-betonowe)
  bloczki-beton-komorkowy → bloczki-z-betonu-komorkowego  BRAK  (podobne: belki stropowe betonowe → belki-stropowe-betonowe)
  bloc

In [119]:

# Wydruk WSZYSTKICH bechcicki kategorii posortowanych
print("=== PEŁNE DRZEWO BECHCICKI ===")
for href in sorted(cat_map.keys()):
    name, lvl = cat_map[href]
    indent = "  " * (lvl - 1)
    slug = href.strip("/").split("/")[-1]
    print(f"{indent}[L{lvl}] {slug:55s} {name}")


=== PEŁNE DRZEWO BECHCICKI ===
[L1] chemia-budowlana                                        Chemia budowlana
  [L2] dodatki-do-zapraw-i-betonu                              Dodatki do zapraw i betonu
  [L2] gipsy-i-gladzie                                         Gipsy i gładzie
    [L3] gipsy-budowlane                                         Gipsy budowlane
    [L3] gipsy-szpachlowe                                        Gipsy szpachlowe
    [L3] gladzie-gipsowe-w-proszku                               Gładzie gipsowe w proszku
    [L3] gladzie-masy-gotowe                                     Gładzie masy gotowe
    [L3] kleje-gipsowe                                           Kleje gipsowe
  [L2] grunty                                                  Grunty
    [L3] grunty-do-posadzek                                      Grunty do posadzek
    [L3] grunty-pod-tynki                                        Grunty pod tynki
    [L3] grunty-specjalistyczne                                  Gru

In [123]:

import requests, json, time

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"; DATASET = "production"
BASE     = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/{DATASET}"
MUTATE   = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/mutate/{DATASET}"
HEADERS  = {"Authorization": f"Bearer {TOKEN}"}

def groq(query, params=None):
    p = {"query": query}
    if params:
        p.update({f"${k}": json.dumps(v) for k, v in params.items()})
    r = requests.get(BASE, params=p, headers=HEADERS)
    r.raise_for_status()
    return r.json()["result"]

def mutate(mutations, label=""):
    r = requests.post(MUTATE, headers={**HEADERS,"Content-Type":"application/json"},
                      json={"mutations": mutations})
    if r.ok:
        print(f"  ✅ {label}: {len(mutations)} op")
    else:
        print(f"  ❌ {label}: {r.status_code} {r.text[:200]}")
    return r.ok

# ═══════ POBIERZ AKTUALNY STAN SANITY ═══════
sanity_cats = groq("""*[_type=="category"]{_id,name,"slug":slug.current,"parentId":parent._ref}""")
sanity_by_slug = {c["slug"]: c for c in sanity_cats}
sanity_by_id   = {c["_id"]:  c for c in sanity_cats}

def slug_to_id(slug):
    return sanity_by_slug.get(slug, {}).get("_id")

print(f"Stan Sanity: {len(sanity_cats)} kategorii")

# ═══════ KROK 1: PATCH 5 złych slugów ═══════
print("\n=== KROK 1: PATCH złych slugów ===")
SLUG_FIXES = [
    ("katowniki-i-katomierce",            "katowniki-i-katomierze",    "Kątowniki i kątomierze"),
    ("kleje-glazura",                     "kleje-do-glazury",          "Kleje do glazury"),
    ("kleje-welna",                       "kleje-do-welen",            "Kleje do wełen"),
    ("piany-montazowe-wezzykowe",         "piany-montazowe-wezykowe",  "Piany montażowe wężykowe"),
    ("tasmy-uszczelniajace-hydroizolacje","tasmy-uszczelniajace-do-hydroizolacji","Taśmy uszczelniające do hydroizolacji"),
]
for old_slug, new_slug, new_name in SLUG_FIXES:
    cat = sanity_by_slug.get(old_slug)
    if not cat:
        print(f"  SKIP {old_slug} — nie znaleziono")
        continue
    mutate([{"patch":{"id":cat["_id"],"set":{"slug":{"_type":"slug","current":new_slug},"name":new_name}}}],
           f"{old_slug}→{new_slug}")
    sanity_by_slug[new_slug] = {**cat, "slug": new_slug, "name": new_name}
    del sanity_by_slug[old_slug]
time.sleep(1)


Stan Sanity: 349 kategorii

=== KROK 1: PATCH złych slugów ===


  ✅ katowniki-i-katomierce→katowniki-i-katomierze: 1 op


  ✅ kleje-glazura→kleje-do-glazury: 1 op


  ✅ kleje-welna→kleje-do-welen: 1 op


  ✅ piany-montazowe-wezzykowe→piany-montazowe-wezykowe: 1 op


  ✅ tasmy-uszczelniajace-hydroizolacje→tasmy-uszczelniajace-do-hydroizolacji: 1 op


In [127]:

# ═══════ KROK 2: CREATE brakujące kategorie ═══════
print("=== KROK 2: CREATE brakujące kategorie ===")

# Odśwież stan Sanity po PATCH
sanity_cats = groq("""*[_type=="category"]{_id,name,"slug":slug.current,"parentId":parent._ref}""")
sanity_by_slug = {c["slug"]: c for c in sanity_cats}

def slug_to_id(slug):
    return sanity_by_slug.get(slug, {}).get("_id")

# Bechcicki tree jako lista (href, name, lvl, parent_slug, l1_slug)
bech_list = []
for href, (name, lvl) in sorted(cat_map.items()):
    parts = href.strip("/").split("/")
    slug = parts[-1]
    parent_slug = parts[-2] if lvl > 1 else None
    l1_slug = parts[0]
    bech_list.append((slug, name, lvl, parent_slug, l1_slug))

# Sortuj: L1 → L2 → L3
bech_list.sort(key=lambda x: x[2])

created = 0
skipped = 0

for slug, name, lvl, parent_slug, l1_slug in bech_list:
    if slug in sanity_by_slug:
        skipped += 1
        continue  # już istnieje
    
    # Buduj dokument
    doc = {
        "_type": "category",
        "_id": f"cat-{slug}",
        "name": name,
        "slug": {"_type": "slug", "current": slug},
    }
    if parent_slug:
        parent_id = slug_to_id(parent_slug)
        if not parent_id:
            print(f"  ⚠️ SKIP {slug} — brak parenta {parent_slug}")
            continue
        doc["parent"] = {"_type": "reference", "_ref": parent_id}
    
    ok = mutate([{"createOrReplace": doc}], f"CREATE {slug}")
    if ok:
        sanity_by_slug[slug] = {"_id": f"cat-{slug}", "slug": slug, "name": name}
        created += 1
    time.sleep(0.1)

print(f"\nPodsumowanie: {created} utworzono, {skipped} już istniało")


=== KROK 2: CREATE brakujące kategorie ===


  ✅ CREATE oswietlenie: 1 op


  ✅ CREATE pozostale: 1 op


  ✅ CREATE dodatki-do-zapraw-i-betonu: 1 op


  ✅ CREATE srodki-czyszczaco-pielegnacyjne: 1 op


...

  ✅ CREATE sufity-rastrowe: 1 op


  ✅ CREATE profile-nosne-glowne-do-sufitow-podwieszanych: 1 op


  ✅ CREATE profile-poprzeczne-do-sufitow-podwieszanych: 1 op


  ✅ CREATE profile-przyscienne-do-sufitow-podwieszanych: 1 op


  ✅ CREATE profile-specjalne-do-sufitow-podwieszanych: 1 op

Podsumowanie: 99 utworzono, 220 już istniało


In [131]:

import requests, json, time

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"; DATASET = "production"
MUTATE   = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/mutate/{DATASET}"
BASE     = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/{DATASET}"
HEADERS  = {"Authorization": f"Bearer {TOKEN}"}

def groq(q, params=None):
    p = {"query": q}
    if params: p.update({f"${k}": json.dumps(v) for k,v in params.items()})
    r = requests.get(BASE, params=p, headers=HEADERS); r.raise_for_status()
    return r.json()["result"]

def mutate(mutations, label=""):
    r = requests.post(MUTATE, headers={**HEADERS,"Content-Type":"application/json"},
                      json={"mutations": mutations})
    if not r.ok: print(f"  ❌ {label}: {r.status_code} {r.text[:150]}")
    return r.ok

# Aktualne Sanity
sanity_cats = groq('*[_type=="category"]{_id,name,"slug":slug.current,"parentId":parent._ref}')
sanity_by_slug = {c["slug"]: c for c in sanity_cats}
sanity_by_id   = {c["_id"]:  c for c in sanity_cats}

def get_id(slug): return sanity_by_slug.get(slug,{}).get("_id")

# ═══ Mapowanie: stary_slug → (nowy_slug, nowa_nazwa, nowy_parent_slug) ═══
# Produkty pozostają przy _id starej kategorii — PATCHujemy slug/name/parent
REMAP = {
    # Farby
    "farby-elaw-akrylowe":   ("farby-elewacyjne-akrylowe",   "Farby elewacyjne akrylowe",   "farby-elewacyjne"),
    "farby-elaw-emulsyjne":  ("farby-elewacyjne-emulsyjne",  "Farby elewacyjne emulsyjne",  "farby-elewacyjne"),
    "farby-elaw-silikatowe": ("farby-elewacyjne-silikatowe", "Farby elewacyjne silikatowe", "farby-elewacyjne"),
    "farby-elaw-silikonowe": ("farby-elewacyjne-silikonowe", "Farby elewacyjne silikonowe", "farby-elewacyjne"),
    "farby-biale":           ("farby-wewnetrzne-biale",      "Farby wewnętrzne białe",      "farby-wewnetrzne"),
    "farby-kolorowe":        ("farby-wewnetrzne-kolorowe",   "Farby wewnętrzne kolorowe",   "farby-wewnetrzne"),
    # Styropian
    "styropian":             ("styropiany-fasadowe-eps",     "Styropiany fasadowe EPS",     "styropiany"),
    "styropian-fasadowy-eps":("styropiany-fasadowe-eps",     "Styropiany fasadowe EPS",     "styropiany"),
    # Wełny
    "welna-fasadowa":        ("welny-fasadowe",              "Wełny fasadowe",              "welny"),
    "welna-mineralna":       ("welny-fasadowe",              "Wełny fasadowe",              "welny"),
    # Tynki elewacyjne (bechcicki ma tylko tynki-elewacyjne jako L3 pod tynki)
    "tynki-akrylowe":        ("tynki-elewacyjne",            "Tynki elewacyjne",            "tynki"),
    "tynki-mineralne":       ("tynki-elewacyjne",            "Tynki elewacyjne",            "tynki"),
    "tynki-silikatowe":      ("tynki-elewacyjne",            "Tynki elewacyjne",            "tynki"),
    "tynki-silikonowe":      ("tynki-elewacyjne",            "Tynki elewacyjne",            "tynki"),
    # Kleje
    "kleje-styropian":       ("kleje-do-styropianu-i-styroduru","Kleje do styropianu i styroduru","kleje"),
    "kleje-plytek":          ("kleje-do-glazury",            "Kleje do glazury",            "kleje"),
    # Rynny
    "rynny-pvc":             ("systemy-rynnowe-pvc",         "Systemy rynnowe PVC",         "rynny"),
    # Zaprawy
    "zaprawy-murarskie":     ("zaprawy-murarskie-ogolnego-zastosowania","Zaprawy murarskie ogólnego zastosowania","zaprawy"),
    "zaprawy-posadzkowe":    ("zaprawy-posadzkowe-masy-samopoziomujace","Zaprawy posadzkowe masy samopoziomujące","zaprawy"),
    # Spoiny
    "fugi":                  ("spoiny-zwykle",               "Spoiny zwykłe",               "spoiny"),
    # Piany
    "piany-pistoletowe":     ("piany-montazowe-pistoletowe", "Piany montażowe pistoletowe", "piany-montazowe"),
    # Bloczki
    "bloczek-komorkowy":     ("bloczki",                     "Bloczki",                     "materialy-konstrukcyjne"),
    "bloczki-beton-komorkowy":("bloczki",                    "Bloczki",                     "materialy-konstrukcyjne"),
    "bloczki-silikatowe":    ("bloczki",                     "Bloczki",                     "materialy-konstrukcyjne"),
    "ceramika-budowlana":    ("bloczki",                     "Bloczki",                     "materialy-konstrukcyjne"),
    "cegly":                 ("bloczki",                     "Bloczki",                     "materialy-konstrukcyjne"),
    "pustaki-ceramiczne":    ("pustaki",                     "Pustaki",                     "materialy-konstrukcyjne"),
    "nadproza":              ("belki-stropowe-betonowe",     "Belki stropowe betonowe",     "materialy-konstrukcyjne"),
    # Okna dachowe
    "okna-dachowe-std":      ("okna-dachowe",                "Okna dachowe",                "okna-dachowe-i-akcesoria"),
    # Pokrycia
    "pokrycia-blacha":       ("pokrycia-dachowe-z-blachy",   "Pokrycia dachowe z blachy",   "pokrycia-dachowe"),
    # Akcesoria
    "akcesoria-izolacji":    ("akcesoria-do-izolacji",       "Akcesoria do izolacji",       "izolacje"),
    "akcesoria-rynny":       ("akcesoria-do-systemow-rynnowych","Akcesoria do systemów rynnowych","rynny"),
    # Cement / wylewki → zaprawy posadzkowe
    "cement":                ("zaprawy-specjalistyczne",     "Zaprawy specjalistyczne",     "zaprawy"),
    "wylewki-cementowe":     ("zaprawy-posadzkowe-masy-samopoziomujace","Zaprawy posadzkowe masy samopoziomujące","zaprawy"),
    # Folie w płynie → hydroizolacje bitumiczne
    "folie-w-plynie":        ("hydroizolacje-bitumiczne",    "Hydroizolacje bitumiczne",    "hydroizolacje"),
    # Impregnaty
    "impregnaty-drewno":     ("impregnaty",                  "Impregnaty",                  "farby-do-drewna"),
    # Kolki
    "kolki-wkrety-uniwersalne":("kolki-i-wkrety-uniwersalne","Kołki i wkręty uniwersalne",  "elementy-mocujace-uniwersalne"),
    # Lakiery
    "lakiery-drewno":        ("lakiery-do-drewna",           "Lakiery do drewna",           "farby-do-drewna"),
    # Profile
    "profile-sufit":         ("profile-do-sufitow-podwieszanych","Profile do sufitów podwieszanych","sufity-podwieszane"),
    # Mozaiki → płytki dekoracyjne (bechcicki nie ma mozaiki)
    "mozaiki":               ("plytki-dekoracyjne",          "Płytki dekoracyjne",          "plytki"),
}

print(f"Mapowań do wykonania: {len(REMAP)}")
done = 0; skip = 0; conflict = 0

for old_slug, (new_slug, new_name, new_parent_slug) in REMAP.items():
    cat = sanity_by_slug.get(old_slug)
    if not cat:
        skip += 1
        continue
    
    # Sprawdź czy docelowy slug już istnieje (inny _id) — conflict
    existing = sanity_by_slug.get(new_slug)
    if existing and existing["_id"] != cat["_id"]:
        # Docelowa kategoria już istnieje → przenieś produkty do niej, usuń starą
        print(f"  CONFLICT {old_slug} → {new_slug} już istnieje (ID:{existing['_id']}), pomiń PATCH")
        conflict += 1
        continue
    
    parent_id = get_id(new_parent_slug)
    patch_set = {
        "slug": {"_type":"slug","current": new_slug},
        "name": new_name,
    }
    if parent_id:
        patch_set["parent"] = {"_type":"reference","_ref": parent_id}
    
    ok = mutate([{"patch":{"id":cat["_id"],"set": patch_set}}], f"{old_slug}→{new_slug}")
    if ok:
        sanity_by_slug[new_slug] = {**cat, "slug": new_slug, "name": new_name}
        if old_slug in sanity_by_slug: del sanity_by_slug[old_slug]
        done += 1

print(f"\n✅ Wykonano: {done}, pominięto (brak): {skip}, konflikt (istnieje): {conflict}")


Mapowań do wykonania: 40
  CONFLICT farby-elaw-akrylowe → farby-elewacyjne-akrylowe już istnieje (ID:cat-l3-farby-elewacyjne-akrylowe), pomiń PATCH
  CONFLICT farby-elaw-emulsyjne → farby-elewacyjne-emulsyjne już istnieje (ID:cat-l3-farby-elewacyjne-emulsyjne), pomiń PATCH
  CONFLICT farby-elaw-silikatowe → farby-elewacyjne-silikatowe już istnieje (ID:cat-l3-farby-elewacyjne-silikatowe), pomiń PATCH
  CONFLICT farby-elaw-silikonowe → farby-elewacyjne-silikonowe już istnieje (ID:cat-l3-farby-elewacyjne-silikonowe), pomiń PATCH
  CONFLICT farby-biale → farby-wewnetrzne-biale już istnieje (ID:cat-l3-farby-wewnetrzne-biale), pomiń PATCH
  CONFLICT farby-kolorowe → farby-wewnetrzne-kolorowe już istnieje (ID:cat-l3-farby-wewnetrzne-kolorowe), pomiń PATCH
  CONFLICT styropian → styropiany-fasadowe-eps już istnieje (ID:cat-l3-styropiany-fasadowe-eps), pomiń PATCH
  CONFLICT styropian-fasadowy-eps → styropiany-fasadowe-eps już istnieje (ID:cat-l3-styropiany-fasadowe-eps), pomiń PATCH
  CONFLICT

In [135]:

import requests, json, time

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"; DATASET = "production"
MUTATE = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/mutate/{DATASET}"
BASE   = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/{DATASET}"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

def groq(q, params=None):
    p = {"query": q}
    if params: p.update({f"${k}": json.dumps(v) for k,v in params.items()})
    r = requests.get(BASE, params=p, headers=HEADERS); r.raise_for_status()
    return r.json()["result"]

def mutate_batch(mutations):
    r = requests.post(MUTATE, headers={**HEADERS,"Content-Type":"application/json"},
                      json={"mutations": mutations})
    return r.ok, r.status_code

# Aktualne Sanity
sanity_cats = groq('*[_type=="category"]{_id,name,"slug":slug.current}')
sanity_by_slug = {c["slug"]: c for c in sanity_cats}

REMAP = {
    "farby-elaw-akrylowe":   "farby-elewacyjne-akrylowe",
    "farby-elaw-emulsyjne":  "farby-elewacyjne-emulsyjne",
    "farby-elaw-silikatowe": "farby-elewacyjne-silikatowe",
    "farby-elaw-silikonowe": "farby-elewacyjne-silikonowe",
    "farby-biale":           "farby-wewnetrzne-biale",
    "farby-kolorowe":        "farby-wewnetrzne-kolorowe",
    "styropian":             "styropiany-fasadowe-eps",
    "styropian-fasadowy-eps":"styropiany-fasadowe-eps",
    "welna-fasadowa":        "welny-fasadowe",
    "welna-mineralna":       "welny-fasadowe",
    "tynki-akrylowe":        "tynki-elewacyjne",
    "tynki-mineralne":       "tynki-elewacyjne",
    "tynki-silikatowe":      "tynki-elewacyjne",
    "tynki-silikonowe":      "tynki-elewacyjne",
    "kleje-styropian":       "kleje-do-styropianu-i-styroduru",
    "kleje-plytek":          "kleje-do-glazury",
    "rynny-pvc":             "systemy-rynnowe-pvc",
    "zaprawy-murarskie":     "zaprawy-murarskie-ogolnego-zastosowania",
    "zaprawy-posadzkowe":    "zaprawy-posadzkowe-masy-samopoziomujace",
    "fugi":                  "spoiny-zwykle",
    "piany-pistoletowe":     "piany-montazowe-pistoletowe",
    "bloczek-komorkowy":     "bloczki",
    "bloczki-beton-komorkowy":"bloczki",
    "bloczki-silikatowe":    "bloczki",
    "ceramika-budowlana":    "bloczki",
    "cegly":                 "bloczki",
    "pustaki-ceramiczne":    "pustaki",
    "nadproza":              "belki-stropowe-betonowe",
    "okna-dachowe-std":      "okna-dachowe",
    "pokrycia-blacha":       "pokrycia-dachowe-z-blachy",
    "akcesoria-izolacji":    "akcesoria-do-izolacji",
    "akcesoria-rynny":       "akcesoria-do-systemow-rynnowych",
    "cement":                "zaprawy-specjalistyczne",
    "wylewki-cementowe":     "zaprawy-posadzkowe-masy-samopoziomujace",
    "folie-w-plynie":        "hydroizolacje-bitumiczne",
    "impregnaty-drewno":     "impregnaty",
    "kolki-wkrety-uniwersalne":"kolki-i-wkrety-uniwersalne",
    "lakiery-drewno":        "lakiery-do-drewna",
    "profile-sufit":         "profile-do-sufitow-podwieszanych",
    "mozaiki":               "plytki-dekoracyjne",
}

total_moved = 0
total_deleted = 0

for old_slug, new_slug in REMAP.items():
    old_cat = sanity_by_slug.get(old_slug)
    new_cat = sanity_by_slug.get(new_slug)
    if not old_cat or not new_cat:
        continue
    old_id = old_cat["_id"]
    new_id = new_cat["_id"]

    # Znajdź produkty ze starą kategorią
    prods = groq('*[_type=="product" && category._ref==$id]._id', {"id": old_id})
    if prods:
        # Przenieś produkty – batch po 200
        for i in range(0, len(prods), 200):
            chunk = prods[i:i+200]
            mutations = [{"patch":{"id":pid,"set":{"category":{"_type":"reference","_ref":new_id}}}} for pid in chunk]
            ok, code = mutate_batch(mutations)
            if ok: total_moved += len(chunk)
            else: print(f"  ❌ move {old_slug}: {code}")
        time.sleep(0.3)

    # Usuń starą kategorię
    ok, code = mutate_batch([{"delete":{"id": old_id}}])
    if ok: total_deleted += 1
    else: print(f"  ❌ del {old_slug}: {code}")

print(f"\n✅ Przeniesiono produktów: {total_moved}")
print(f"✅ Usunięto starych kategorii: {total_deleted}")



✅ Przeniesiono produktów: 119
✅ Usunięto starych kategorii: 40


In [139]:

import requests, json, time

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"; DATASET = "production"
MUTATE = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/mutate/{DATASET}"
BASE   = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/{DATASET}"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

def groq(q, params=None):
    p = {"query": q}
    if params: p.update({f"${k}": json.dumps(v) for k,v in params.items()})
    r = requests.get(BASE, params=p, headers=HEADERS); r.raise_for_status()
    return r.json()["result"]

def mutate_batch(mutations):
    r = requests.post(MUTATE, headers={**HEADERS,"Content-Type":"application/json"},
                      json={"mutations": mutations})
    return r.ok

# Pobierz bieżące kategorie Sanity z liczbą produktów
sanity_cats = groq("""
*[_type=="category"] {
  _id, name, "slug": slug.current,
  "level": select(!defined(parent) => "L1", defined(parent->parent) => "L3", "L2"),
  "productCount": count(*[_type == "product" && category._ref == ^._id])
}
""")

# Slugi które powinny istnieć (bechcicki)
bech_slugs = set(bech_by_slug.keys())

# Znajdź Sanity kats które NIE są w bechcicki I mają 0 produktów → usuń
to_delete = [c for c in sanity_cats if c["slug"] not in bech_slugs and c.get("productCount",0) == 0]
still_extra = [c for c in sanity_cats if c["slug"] not in bech_slugs and c.get("productCount",0) > 0]

print(f"Do usunięcia (puste, nie bechcicki): {len(to_delete)}")
print(f"Nadal extra z produktami (wymagają uwagi): {len(still_extra)}")

# Usuń wszystkie puste nie-bechcicki
if to_delete:
    ids = [c["_id"] for c in to_delete]
    ok = mutate_batch([{"delete":{"id": _id}} for _id in ids])
    print(f"{'✅' if ok else '❌'} Usunięto {len(ids)} pustych kategorii")

# Pokaż extra z produktami
if still_extra:
    print("\n⚠️  Kategorie spoza bechcicki które wciąż mają produkty:")
    for c in sorted(still_extra, key=lambda x: -x.get("productCount",0)):
        print(f"  {c['level']}  {c['slug']:50s}  {c['name']:30s}  {c['productCount']} prod")

# Finalna weryfikacja
print("\n=== FINALNA WERYFIKACJA ===")
final_cats = groq("""
*[_type=="category"] {
  "slug": slug.current,
  "level": select(!defined(parent) => "L1", defined(parent->parent) => "L3", "L2")
}
""")
by_level = {}
for c in final_cats:
    by_level.setdefault(c["level"], 0)
    by_level[c["level"]] += 1

print(f"Łącznie kategorii: {len(final_cats)}")
for lvl in ["L1","L2","L3"]:
    print(f"  {lvl}: {by_level.get(lvl,0)}")

# Porównaj z bechcicki
final_slugs = {c["slug"] for c in final_cats}
missing_from_sanity = bech_slugs - final_slugs
extra_in_sanity     = final_slugs - bech_slugs
print(f"\nBrak w Sanity vs bechcicki: {len(missing_from_sanity)}")
print(f"Nadmiarowe w Sanity vs bechcicki: {len(extra_in_sanity)}")
if missing_from_sanity:
    print("  Brakujące:", sorted(missing_from_sanity))
if extra_in_sanity:
    print("  Nadmiarowe:", sorted(extra_in_sanity)[:20])


Do usunięcia (puste, nie bechcicki): 43
Nadal extra z produktami (wymagają uwagi): 44


❌ Usunięto 43 pustych kategorii

⚠️  Kategorie spoza bechcicki które wciąż mają produkty:
  L2  gladzie-proszek                                     Gladzie Proszek                 1 prod
  L2  grunty-tynki                                        Grunty Tynki                    1 prod
  L2  izolacje-przeciwwilgoc                              Izolacje Przeciwwilgoc          1 prod
  L2  kolki-rozpozowe                                     Kolki Rozpozowe                 1 prod
  L2  kotwy-mechaniczne                                   Kotwy Mechaniczne               1 prod
  L2  ksztaltki-wience                                    Ksztaltki Wience                1 prod
  L2  lamele-dekoracyjne                                  Lamele Dekoracyjne              1 prod
  L2  listwy-akcesoria                                    Listwy Akcesoria                1 prod
  L2  masy-bitumiczne                                     Masy Bitumiczne                 1 prod
  L3  narozniki-aluminiowe           

Łącznie kategorii: 407
  L1: 12
  L2: 117
  L3: 278

Brak w Sanity vs bechcicki: 1
Nadmiarowe w Sanity vs bechcicki: 87
  Brakujące: ['farby-specjalistyczne']
  Nadmiarowe: ['akcesoria-do-rynien', 'akcesoria-instalacyjne-plytki', 'akcesoria-przeciwsniegowe', 'artykuly-scierne-do-gk', 'elementy-mocujace', 'elementy-wykonczeniowe-sufitow', 'folie-izolacyjne', 'folie-ochronne-budowlane', 'gladzie-proszek', 'grunty-tynki', 'impregnaty-do-drewna', 'izolacje-przeciwwilgoc', 'katowniki-do-profili', 'klipsy-mocujace-sufitow', 'kolki-rozpozowe', 'kotwy-mechaniczne', 'ksztaltki-wience', 'lamele-dekoracyjne', 'lawy-kominiarskie', 'listwy-akcesoria']


In [143]:

import requests, json, time

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"; DATASET = "production"
MUTATE = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/mutate/{DATASET}"
BASE   = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/{DATASET}"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

def groq(q, params=None):
    p = {"query": q}
    if params: p.update({f"${k}": json.dumps(v) for k,v in params.items()})
    r = requests.get(BASE, params=p, headers=HEADERS); r.raise_for_status()
    return r.json()["result"]

def mutate_one(mutations):
    r = requests.post(MUTATE, headers={**HEADERS,"Content-Type":"application/json"},
                      json={"mutations": mutations})
    return r.ok, r.text[:200]

# Pobierz aktualny stan
sanity_cats = groq("""
*[_type=="category"] {
  _id, name, "slug": slug.current,
  "level": select(!defined(parent) => "L1", defined(parent->parent) => "L3", "L2"),
  "productCount": count(*[_type == "product" && category._ref == ^._id])
}
""")
sanity_by_slug = {c["slug"]: c for c in sanity_cats}

# ═══ KROK A: Usuń puste nie-bechcicki po jednej ═══
to_delete_ids = [c["_id"] for c in sanity_cats if c["slug"] not in bech_slugs and c.get("productCount",0) == 0]
print(f"Puste do usunięcia: {len(to_delete_ids)}")
del_ok = 0; del_fail = 0
for _id in to_delete_ids:
    ok, msg = mutate_one([{"delete":{"id": _id}}])
    if ok: del_ok += 1
    else: del_fail += 1
print(f"  Usunięto: {del_ok}, błędów: {del_fail}")

# ═══ KROK B: Utwórz brakującą farby-specjalistyczne ═══
parent_id = sanity_by_slug.get("farby-i-rozpuszczalniki", {}).get("_id")
if parent_id:
    ok, _ = mutate_one([{"createOrReplace": {
        "_type": "category", "_id": "cat-farby-specjalistyczne",
        "name": "Farby specjalistyczne",
        "slug": {"_type":"slug","current":"farby-specjalistyczne"},
        "parent": {"_type":"reference","_ref": parent_id}
    }}])
    print(f"\nCREATE farby-specjalistyczne: {'✅' if ok else '❌'}")
    if ok:
        sanity_by_slug["farby-specjalistyczne"] = {"_id":"cat-farby-specjalistyczne","slug":"farby-specjalistyczne","name":"Farby specjalistyczne"}

# ═══ KROK C: Mapowanie 44 old cats z 1 produktem → właściwe bechcicki ═══
OLD_TO_BECH = {
    "gladzie-proszek":           "gladzie-gipsowe-w-proszku",
    "grunty-tynki":              "grunty-pod-tynki",
    "izolacje-przeciwwilgoc":    "hydroizolacje-mineralne",
    "kolki-rozpozowe":           "kolki-i-wkrety-uniwersalne",
    "kotwy-mechaniczne":         "kotwy-montazowe",
    "ksztaltki-wience":          "bloczki",
    "lamele-dekoracyjne":        "panele-i-dekory-scienne",
    "listwy-akcesoria":          "listwy-i-akcesoria",
    "masy-bitumiczne":           "masy-bitumiczne-gruntujace",
    "narozniki-aluminiowe":      "narozniki-do-suchej-zabudowy-aluminiowe",
    "narzedzia-glazurnicze":     "narzedzia-i-akcesoria-glazurnicze",
    "pace-gladzie":              "pace",
    "papy-bitumiczne":           "papy-hydroizolacyjne",
    "papy-dachowe":              "papy",
    "plyty-gipsowe-sufitowe":    "plyty-sufitowe-z-welny-mineralnej",
    "plyty-welna-mineralna":     "plyty-sufitowe-z-welny-mineralnej",
    "plyty-welna-szklana":       "plyty-sufitowe-z-welny-szklanej",
    "podbitki-dachowe":          "pokrycia-dachowe-z-blachy",
    "poziomice-laty":            "poziomnice",
    "profile-nosne-glowne":      "profile-nosne-glowne-do-sufitow-podwieszanych",
    "profile-poprzeczne":        "profile-poprzeczne-do-sufitow-podwieszanych",
    "profile-przysc-sufity":     "profile-przyscienne-do-sufitow-podwieszanych",
    "profile-scian":             "profile-do-suchej-zabudowy-konstrukcja-scienna",
    "profile-sciana":            "profile-do-suchej-zabudowy-konstrukcja-scienna",
    "rury-spustowe":             "akcesoria-do-systemow-rynnowych",
    "rynny-blacha":              "systemy-rynnowe-z-blachy-powlekanej",
    "silikaty":                  "bloczki",
    "stropy-systemowe":          "belki-stropowe-ceramiczne",
    "styropian-dach-podloga":    "styropian-dach-podloga-eps",
    "styropian-fundamenty":      "styropiany-do-fundamentow",
    "styropian-podlogowy":       "styropian-dach-podloga-eps",
    "szlifierki-katowe":         "szlifierki",
    "szpachle":                  "szpachle-i-szpachelki",
    "tasmy-sucha-zabudowa":      "tasmy-do-suchej-zabudowy",
    "tynki-cementowe":           "tynki-cementowo-wapienne",
    "tynki-mozaikowe":           "tynki-specjalne",
    "tynki-silikonowo-silikatowe":"tynki-specjalne",
    "tynki-wewnetrzne":          "tynki-gipsowe",
    "welna-poddasza":            "welny-do-poddaszy",
    "welna-stropy":              "welny-do-stropow-i-podlog",
    "welna-sucha-zabudowa":      "welny-do-suchej-zabudowy-i-scian-dzialowych",
    "welna-szklana":             "welny-fasadowe",
    "wkrety-drewno":             "wkrety-do-suchej-zabudowy",
    "wkrety-metal":              "wkrety-do-suchej-zabudowy",
}

moved = 0; deleted = 0
# Odśwież sanity_by_slug po kreacji
sanity_cats2 = groq('*[_type=="category"]{_id,name,"slug":slug.current}')
sanity_by_slug2 = {c["slug"]: c for c in sanity_cats2}

for old_slug, new_slug in OLD_TO_BECH.items():
    old_cat = sanity_by_slug2.get(old_slug)
    new_cat = sanity_by_slug2.get(new_slug)
    if not old_cat or not new_cat:
        continue
    # Przenieś produkty
    prods = groq('*[_type=="product" && category._ref==$id]._id', {"id": old_cat["_id"]})
    for pid in prods:
        ok, _ = mutate_one([{"patch":{"id":pid,"set":{"category":{"_type":"reference","_ref":new_cat["_id"]}}}}])
        if ok: moved += 1
    # Usuń starą kategorię
    ok, _ = mutate_one([{"delete":{"id": old_cat["_id"]}}])
    if ok: deleted += 1

print(f"\nCzęść C — przeniesiono produktów: {moved}, usunięto kat: {deleted}")


Puste do usunięcia: 43


  Usunięto: 36, błędów: 7



CREATE farby-specjalistyczne: ✅



Część C — przeniesiono produktów: 44, usunięto kat: 43


In [147]:

import requests, json, time

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"; DATASET = "production"
MUTATE = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/mutate/{DATASET}"
BASE   = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/{DATASET}"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

def groq(q, params=None):
    p = {"query": q}
    if params: p.update({f"${k}": json.dumps(v) for k,v in params.items()})
    r = requests.get(BASE, params=p, headers=HEADERS); r.raise_for_status()
    return r.json()["result"]

def mutate_one(m):
    r = requests.post(MUTATE, headers={**HEADERS,"Content-Type":"application/json"}, json={"mutations":m})
    return r.ok

# Pobierz finalny stan
cats = groq("""
*[_type=="category"] {
  _id, name, "slug": slug.current,
  "level": select(!defined(parent) => "L1", defined(parent->parent) => "L3", "L2"),
  "productCount": count(*[_type == "product" && category._ref == ^._id])
}
""")

bech_slugs_set = set(bech_by_slug.keys())
by_slug = {c["slug"]: c for c in cats}

# Usuń pozostałe puste nie-bechcicki (retry)
to_del = [c for c in cats if c["slug"] not in bech_slugs_set and c.get("productCount",0) == 0]
del_done = sum(1 for c in to_del if mutate_one([{"delete":{"id":c["_id"]}}]))
print(f"Retry usunięcia pustych: {del_done}/{len(to_del)}")

# Odśwież po usunięciu
time.sleep(1)
cats = groq("""
*[_type=="category"] {
  _id, name, "slug": slug.current,
  "level": select(!defined(parent) => "L1", defined(parent->parent) => "L3", "L2"),
  "productCount": count(*[_type == "product" && category._ref == ^._id])
}
""")

by_level = {}
for c in cats:
    by_level.setdefault(c["level"], 0)
    by_level[c["level"]] += 1

print(f"\n=== FINALNY STAN SANITY ===")
print(f"Łącznie: {len(cats)}  |  L1={by_level.get('L1',0)}  L2={by_level.get('L2',0)}  L3={by_level.get('L3',0)}")
print(f"Oczekiwane bechcicki: {len(bech_slugs_set)}  |  L1={len(l1b)}  L2={len(l2b)}  L3={len(l3b)}")

final_slugs = {c["slug"] for c in cats}
missing = bech_slugs_set - final_slugs
extra   = {c["slug"] for c in cats if c["slug"] not in bech_slugs_set}
extra_with_prods = [(c["slug"],c["name"],c.get("productCount",0)) for c in cats 
                    if c["slug"] not in bech_slugs_set and c.get("productCount",0) > 0]

print(f"\nBrak w Sanity: {len(missing)}")
if missing: print("  ", sorted(missing))
print(f"Nadmiarowe bez produktów: {len(extra) - len(extra_with_prods)}")
print(f"Nadmiarowe Z produktami: {len(extra_with_prods)}")
for s,n,p in sorted(extra_with_prods, key=lambda x:-x[2])[:10]:
    print(f"  {s:50s}  {n:30s}  {p} prod")


Retry usunięcia pustych: 1/8



=== FINALNY STAN SANITY ===
Łącznie: 328  |  L1=12  L2=78  L3=238
Oczekiwane bechcicki: 319  |  L1=11  L2=71  L3=237

Brak w Sanity: 0
Nadmiarowe bez produktów: 7
Nadmiarowe Z produktami: 0


In [151]:

import requests, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"; DATASET = "production"
MUTATE = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/mutate/{DATASET}"
BASE   = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/{DATASET}"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

def groq(q):
    r = requests.get(BASE, params={"query": q}, headers=HEADERS)
    r.raise_for_status(); return r.json()["result"]

# Pobierz nadmiarowe puste
extra_empty = groq("""
*[_type=="category" && count(*[_type=="product" && category._ref==^._id])==0] {
  _id, name, "slug": slug.current
}
""")

bech_slugs_set = set(bech_by_slug.keys())
to_del = [c for c in extra_empty if c["slug"] not in bech_slugs_set]
print(f"Do usunięcia: {len(to_del)}")
for c in to_del:
    print(f"  {c['_id']:45s}  {c['slug']}")

# Usuń
for c in to_del:
    r = requests.post(MUTATE, headers={**HEADERS,"Content-Type":"application/json"},
                      json={"mutations":[{"delete":{"id":c["_id"]}}]})
    status = "✅" if r.ok else f"❌ {r.status_code}"
    print(f"  {status} {c['slug']}")

# Finalna liczba
final = groq('*[_type=="category"]{"l":select(!defined(parent)=>"L1",defined(parent->parent)=>"L3","L2")}')
from collections import Counter
cnt = Counter(c["l"] for c in final)
print(f"\n✅ FINALNIE: {len(final)} kategorii | L1={cnt['L1']} L2={cnt['L2']} L3={cnt['L3']}")
print(f"   Bechcicki: {len(bech_slugs_set)} | L1={len(l1b)} L2={len(l2b)} L3={len(l3b)}")


Do usunięcia: 7
  cat-l2-elementy-mocujace                       elementy-mocujace
  cat-l2-folie-izolacyjne                        folie-izolacyjne
  cat-l2-listwy-i-akcesoria-plytkarsk            listwy-i-akcesoria-plytkarskie
  cat-l2-plyty-do-suchej-zabudowy                plyty-do-suchej-zabudowy
  cat-l2-welny-izolacyjne                        welny-izolacyjne
  cat-posadzki                                   posadzki
  category-narzedzia-glazurnicze                 narzedzia-glazurnicze
  ❌ 409 elementy-mocujace


  ❌ 409 folie-izolacyjne
  ❌ 409 listwy-i-akcesoria-plytkarskie


  ❌ 409 plyty-do-suchej-zabudowy
  ❌ 409 welny-izolacyjne


  ❌ 409 posadzki
  ❌ 409 narzedzia-glazurnicze



✅ FINALNIE: 328 kategorii | L1=12 L2=78 L3=238
   Bechcicki: 319 | L1=11 L2=71 L3=237


In [155]:

import requests, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"; DATASET = "production"
MUTATE = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/mutate/{DATASET}"
BASE   = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/{DATASET}"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

def groq(q, params=None):
    p = {"query": q}
    if params: p.update({f"${k}": json.dumps(v) for k,v in params.items()})
    r = requests.get(BASE, params=p, headers=HEADERS); r.raise_for_status()
    return r.json()["result"]

def mut(m):
    r = requests.post(MUTATE, headers={**HEADERS,"Content-Type":"application/json"}, json={"mutations":m})
    return r.ok, r.status_code

# IDs blokerów
BLOCKED = {
    "cat-l2-elementy-mocujace":        "elementy-mocujace-uniwersalne",
    "cat-l2-folie-izolacyjne":         "folie",
    "cat-l2-listwy-i-akcesoria-plytkarsk": "listwy-i-akcesoria",
    "cat-l2-plyty-do-suchej-zabudowy": "plyty",
    "cat-l2-welny-izolacyjne":         "welny",
    "cat-posadzki":                    None,  # L1 stary — dzieci przepnij do chemia-budowlana lub usuń
    "category-narzedzia-glazurnicze":  "narzedzia-i-akcesoria-glazurnicze",
}

# Aktualny slug→id
all_cats = groq('*[_type=="category"]{_id,"slug":slug.current}')
s2id = {c["slug"]: c["_id"] for c in all_cats}
id2s = {c["_id"]: c["slug"] for c in all_cats}

for blocked_id, new_parent_slug in BLOCKED.items():
    # Znajdź dzieci (kategorie które mają parent._ref = blocked_id)
    children = groq('*[_type=="category" && parent._ref==$pid]{_id,name,"slug":slug.current}', {"pid": blocked_id})
    print(f"\n{blocked_id} → dzieci: {[c['slug'] for c in children]}")
    
    for child in children:
        if new_parent_slug and new_parent_slug in s2id:
            # Przepnij dziecko do właściwego parenta
            ok, code = mut([{"patch":{"id":child["_id"],"set":{"parent":{"_type":"reference","_ref":s2id[new_parent_slug]}}}}])
            print(f"  REPARENT {child['slug']} → {new_parent_slug}: {'✅' if ok else f'❌{code}'}")
        else:
            # Usuń puste dziecko
            pc = groq('count(*[_type=="product" && category._ref==$id])', {"id": child["_id"]})
            if pc == 0:
                ok, code = mut([{"delete":{"id":child["_id"]}}])
                print(f"  DELETE {child['slug']}: {'✅' if ok else f'❌{code}'}")
    
    # Teraz usuń blokera
    ok, code = mut([{"delete":{"id": blocked_id}}])
    print(f"DELETE bloker {blocked_id}: {'✅' if ok else f'❌{code}'}")

# Final count
final = groq('*[_type=="category"]{"l":select(!defined(parent)=>"L1",defined(parent->parent)=>"L3","L2")}')
from collections import Counter
cnt = Counter(c["l"] for c in final)
print(f"\n✅ FINALNIE: {len(final)} | L1={cnt['L1']} L2={cnt['L2']} L3={cnt['L3']}")
print(f"   Bechcicki: 319 | L1=11 L2=71 L3=237")



cat-l2-elementy-mocujace → dzieci: ['kolki-i-wkrety-uniwersalne', 'kotwy-montazowe', 'nakretki', 'wkrety-do-metalu']


  REPARENT kolki-i-wkrety-uniwersalne → elementy-mocujace-uniwersalne: ✅


  REPARENT kotwy-montazowe → elementy-mocujace-uniwersalne: ✅


  REPARENT nakretki → elementy-mocujace-uniwersalne: ✅


  REPARENT wkrety-do-metalu → elementy-mocujace-uniwersalne: ✅


...

  DELETE galanteria-betonowa: ❌409
DELETE bloker cat-posadzki: ❌409



category-narzedzia-glazurnicze → dzieci: ['krzyzyki-do-glazury', 'przyrzady-do-ciecia-glazury']


  REPARENT krzyzyki-do-glazury → narzedzia-i-akcesoria-glazurnicze: ✅


  REPARENT przyrzady-do-ciecia-glazury → narzedzia-i-akcesoria-glazurnicze: ✅


DELETE bloker category-narzedzia-glazurnicze: ✅

✅ FINALNIE: 322 | L1=12 L2=72 L3=238
   Bechcicki: 319 | L1=11 L2=71 L3=237


In [159]:

import requests, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
PROJECT = "nzcwegq7"; DATASET = "production"
MUTATE = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/mutate/{DATASET}"
BASE   = f"https://{PROJECT}.api.sanity.io/v2023-08-01/data/query/{DATASET}"
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

def groq(q, p=None):
    params = {"query": q}
    if p: params.update({f"${k}": json.dumps(v) for k,v in p.items()})
    r = requests.get(BASE, params=params, headers=HEADERS); r.raise_for_status()
    return r.json()["result"]

def mut(m):
    r = requests.post(MUTATE, headers={**HEADERS,"Content-Type":"application/json"}, json={"mutations":m})
    return r.ok, r.status_code

all_cats = groq('*[_type=="category"]{_id,"slug":slug.current}')
s2id = {c["slug"]: c["_id"] for c in all_cats}

# galanteria-betonowa → reparent do `pozostale`
ok, _ = mut([{"patch":{"id":s2id["galanteria-betonowa"],"set":{"parent":{"_type":"reference","_ref":s2id["pozostale"]}}}}])
print(f"reparent galanteria-betonowa → pozostale: {'✅' if ok else '❌'}")

# Teraz usuń cat-posadzki (powinno działać — nie ma już dzieci)
ok, code = mut([{"delete":{"id":"cat-posadzki"}}])
print(f"DELETE cat-posadzki: {'✅' if ok else f'❌{code}'}")

# Finalna weryfikacja
from collections import Counter
final = groq('*[_type=="category"]{"l":select(!defined(parent)=>"L1",defined(parent->parent)=>"L3","L2"),"slug":slug.current}')
cnt = Counter(c["l"] for c in final)
bech_set = set(bech_by_slug.keys())
final_slugs = {c["slug"] for c in final}
missing = bech_set - final_slugs
extra   = [(c["slug"],c["l"]) for c in final if c["slug"] not in bech_set]

print(f"\n🎯 FINALNY STAN: {len(final)} kategorii | L1={cnt['L1']} L2={cnt['L2']} L3={cnt['L3']}")
print(f"   Bechcicki:    319         | L1=11      L2=71      L3=237")
print(f"\nBrakujące w Sanity: {sorted(missing) or 'BRAK ✅'}")
print(f"Nadmiarowe w Sanity: {extra or 'BRAK ✅'}")


reparent galanteria-betonowa → pozostale: ✅


DELETE cat-posadzki: ✅



🎯 FINALNY STAN: 321 kategorii | L1=11 L2=72 L3=238
   Bechcicki:    319         | L1=11      L2=71      L3=237

Brakujące w Sanity: BRAK ✅
Nadmiarowe w Sanity: BRAK ✅


In [163]:

import json, requests

# 1. JSONL — pierwsze 3 rekordy
print("=== JSONL struktura ===")
with open("/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/bechcicki_products_all.jsonl") as f:
    for i, line in enumerate(f):
        if i >= 3: break
        rec = json.loads(line)
        print(json.dumps(rec, indent=2, ensure_ascii=False)[:600])
        print("---")

# 2. Ile rekordów w JSONL
with open("/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/bechcicki_products_all.jsonl") as f:
    total_jsonl = sum(1 for _ in f)
print(f"\nŁącznie w JSONL: {total_jsonl}")


=== JSONL struktura ===
{
  "sku": "P-0029063",
  "name": "",
  "brand": "Henkel",
  "manufacturer_index": "1687904",
  "ean": "5900089112241",
  "url": "",
  "category_main": "chemia-budowlana",
  "category_sub": "kleje",
  "category_leaf": "kleje-do-glazury",
  "alt_id": "0033978 KLEJ DO GRESU C2TE CM12 PLUS 25KG CERESIT 1687904 HENKEL",
  "_page": 1
}
---
{
  "sku": "P-0145857",
  "name": "",
  "brand": "$$$",
  "manufacturer_index": "219606",
  "ean": "5900089111251",
  "url": "",
  "category_main": "chemia-budowlana",
  "category_sub": "kleje",
  "category_leaf": "kleje-do-glazury",
  "alt_id": "0033974 KLEJ DO GLAZURY 25KG CM11 $$$",
  "_page": 1
}
---
{
  "sku": "P-0138876",
  "name": "",
  "brand": "C1T",
  "manufacturer_index": "1550928",
  "ean": "5900089111411",
  "url": "",
  "category_main": "chemia-budowlana",
  "category_sub": "kleje",
  "category_leaf": "kleje-do-glazury",
  "alt_id": "0033976 KLEJ DO GLAZURY 25KG CM11 PLUS C1T",
  "_page": 1
}
---

Łącznie w JSONL: 164

In [167]:

import json, requests
from bs4 import BeautifulSoup

headers_web = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

# 1. Analiza alt_id — czy zawiera nazwę produktu
print("=== Próbka alt_id ===")
with open("/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/bechcicki_products_all.jsonl") as f:
    sample = [json.loads(line) for i, line in enumerate(f) if i < 10]
for s in sample[:5]:
    print(f"  SKU: {s['sku']}  EAN: {s['ean']}  mfr_idx: {s['manufacturer_index']}")
    print(f"  alt_id: {s['alt_id']}")
    print()

# 2. Spróbuj URLa produktu używając manufacturer_index
print("=== Test URL produktu ===")
test_mfr = "1687904"  # pierwszy produkt
# Format z nawigacji: /name-id-bud_NUMER
# Spróbuj search
r = requests.get(f"https://www.bechcicki.pl/search?q={sample[0]['ean']}", headers=headers_web, timeout=10, allow_redirects=True)
print(f"Search by EAN: {r.status_code}  URL: {r.url}")

# 3. Sprawdź kategorie bechcicki dla próbki produktów
r2 = requests.get(f"https://www.bechcicki.pl/chemia-budowlana/kleje/kleje-do-glazury", headers=headers_web, timeout=10)
soup = BeautifulSoup(r2.text, "html.parser")
# Znajdź linki produktów
prod_links = [a.get("href","") for a in soup.find_all("a", href=True) if "id-bud_" in a.get("href","")]
print(f"\nLinki produktów na /kleje-do-glazury: {len(prod_links)}")
for l in prod_links[:5]:
    print(f"  {l}")


=== Próbka alt_id ===
  SKU: P-0029063  EAN: 5900089112241  mfr_idx: 1687904
  alt_id: 0033978 KLEJ DO GRESU C2TE CM12 PLUS 25KG CERESIT 1687904 HENKEL

  SKU: P-0145857  EAN: 5900089111251  mfr_idx: 219606
  alt_id: 0033974 KLEJ DO GLAZURY 25KG CM11 $$$

  SKU: P-0138876  EAN: 5900089111411  mfr_idx: 1550928
  alt_id: 0033976 KLEJ DO GLAZURY 25KG CM11 PLUS C1T

  SKU: P-0377235  EAN: 8022452248948  mfr_idx: 0073620PL
  alt_id: 0088548 KLEJ DO GLAZURY C2TE ADESILEX P9 FIBER LITE ZERO 20KG SZARY  MAPEI $$$

  SKU: P-0376053  EAN: 8022452248122  mfr_idx: 0073525PL
  alt_id: 0089466 KLEJ DO GLAZURY 25KG SZARY ADESILEX P9 ZERO  MAPEI

=== Test URL produktu ===


Search by EAN: 404  URL: https://www.bechcicki.pl/search?q=5900089112241



Linki produktów na /kleje-do-glazury: 0


In [171]:

import requests
from bs4 import BeautifulSoup

headers_web = {"User-Agent": "Mozilla/5.0"}

# Sprawdź czy strona produktu otwiera się po id-bud_
test_url = "https://www.bechcicki.pl/klej-do-gresu-c2te-cm12-plus-25kg-ceresit-id-bud_1687904"
r = requests.get(test_url, headers=headers_web, timeout=10, allow_redirects=True)
print(f"URL: {r.url}  Status: {r.status_code}")

# Jeśli 200, sprawdź zawartość
if r.ok:
    soup = BeautifulSoup(r.text, "html.parser")
    title = soup.find("h1")
    print(f"H1: {title.text[:100] if title else 'brak'}")
    # Szukaj opisu
    desc = soup.find(["div","section"], class_=lambda c: c and any(x in c.lower() for x in ["desc","opis","product"]))
    print(f"Desc element: {desc.name if desc else 'brak'} class: {desc.get('class') if desc else ''}")
    # Szukaj tabeli parametrów
    tables = soup.find_all("table")
    print(f"Tabel: {len(tables)}")
    print(f"HTML size: {len(r.text)} znaków")
else:
    # Spróbuj po samym ID
    r2 = requests.get(f"https://www.bechcicki.pl/-id-bud_1687904", headers=headers_web, timeout=10, allow_redirects=True)
    print(f"Test 2: {r2.status_code}  URL: {r2.url}")
    
    # Sprawdź JS-rendered content na kategorii
    r3 = requests.get("https://www.bechcicki.pl/chemia-budowlana/kleje/kleje-do-glazury", headers=headers_web, timeout=10)
    soup3 = BeautifulSoup(r3.text, "html.parser")
    scripts = soup3.find_all("script", src=False)
    # Szukaj danych produktów w JS
    for s in scripts:
        txt = s.text
        if "product" in txt.lower() and len(txt) > 100:
            print(f"\nScript z product data (pierwsze 300):\n{txt[:300]}")
            break
    print(f"\nHTML kategorii size: {len(r3.text)}")
    all_hrefs = [a.get("href","") for a in soup3.find_all("a", href=True)]
    prod_hrefs = [h for h in all_hrefs if "id-bud" in h]
    print(f"id-bud linki: {prod_hrefs[:5]}")


URL: https://www.bechcicki.pl/klej-do-gresu-c2te-cm12-plus-25kg-ceresit-id-bud_1687904  Status: 500


Test 2: 500  URL: https://www.bechcicki.pl/-id-bud_1687904



Script z product data (pierwsze 300):
window.__NUXT__=(function(a,b,c,d,e,f,g,h,i,j,k,l,m,n,o,p,q,r,s,t,u,v,w,x,y,z,A,B,C,D,E,F,G,H,I,J,K,L,M,N,O,P,Q,R,S,T,U,V,W,X,Y,Z,_,$,aa,ab,ac,ad,ae,af,ag,ah,ai,aj,ak,al,am,an,ao,ap,aq,ar,as,at,au,av,aw,ax,ay,az,aA,aB,aC,aD,aE,aF,aG,aH,aI,aJ,aK,aL,aM,aN,aO,aP,aQ,aR,aS,aT,aU,aV,aW,aX,aY,aZ,a_,a$,ba,b

HTML kategorii size: 1126061
id-bud linki: []


In [175]:

import requests, re, json
from bs4 import BeautifulSoup

headers_web = {"User-Agent": "Mozilla/5.0"}

# Pobierz stronę kategorii
r = requests.get("https://www.bechcicki.pl/chemia-budowlana/kleje/kleje-do-glazury",
                 headers=headers_web, timeout=15)
html = r.text

# Wyciągnij wszystkie linki id-bud z HTML (SSR may render them)
soup = BeautifulSoup(html, "html.parser")
prod_links_html = soup.find_all("a", href=re.compile(r"id-bud_"))
print(f"Linki produktów w HTML: {len(prod_links_html)}")
for a in prod_links_html[:5]:
    print(f"  {a.get('href','')}  |  {a.text.strip()[:60]}")

# Sprawdź czy produkty są w data-* atrybutach
items = soup.find_all(attrs={"data-product-id": True})
print(f"\ndata-product-id elements: {len(items)}")

# Szukaj JSON z produktami w skryptach
for script in soup.find_all("script", type="application/json"):
    txt = script.string or ""
    if "ean" in txt.lower() or "product" in txt.lower()[:100]:
        print(f"\nJSON script (pierwsze 400): {txt[:400]}")
        break

# Szukaj w meta tagach lub structured data
ld = soup.find_all("script", type="application/ld+json")
print(f"\nLD+JSON scripts: {len(ld)}")
for s in ld[:2]:
    txt = (s.string or "")[:300]
    print(f"  {txt}")


Linki produktów w HTML: 0

data-product-id elements: 0

LD+JSON scripts: 0


In [179]:

import requests, re, json

headers_web = {"User-Agent": "Mozilla/5.0"}
r = requests.get("https://www.bechcicki.pl/chemia-budowlana/kleje/kleje-do-glazury",
                 headers=headers_web, timeout=15)
html = r.text

# Szukaj URL-i API w HTML
api_patterns = re.findall(r'["\'](?:https?://[^"\']+/api/[^"\']{3,60}|/api/[^"\']{3,60})["\']', html)
print("API URLs:")
for u in sorted(set(api_patterns))[:20]:
    print(f"  {u}")

# Szukaj endpointów z products/catalog
product_endpoints = re.findall(r'["\']([^"\']*(?:product|catalog|item|article)[^"\']*)["\']', html)
unique_ep = sorted(set(p for p in product_endpoints if len(p) > 5 and len(p) < 100))
print(f"\nProduct-like endpoints ({len(unique_ep)}):")
for e in unique_ep[:20]:
    print(f"  {e}")

# Szukaj baseURL lub apiBase
base_urls = re.findall(r'(?:baseURL|apiBase|apiUrl|runtimeConfig)[^"\']{0,30}["\']([^"\']{5,80})["\']', html)
print(f"\nBase URLs: {base_urls[:5]}")

# Sprawdź czy jest jakiś API klucz lub domena
domains = re.findall(r'https?://(?:api\.|backend\.|cms\.)[a-zA-Z0-9.-]+', html)
print(f"\nAPI domains: {sorted(set(domains))[:10]}")


API URLs:

Product-like endpoints (364):
  $attrs.product.measurementUnits.quantityInterval || 1\
  $attrs.product.photos && $attrs.product.photos.length\
  $attrs.product.photosWithVariants[0].variantMap[
  $attrs.product.photos[0].url\
  $attrs.product.price.formattedCurrentPrice() && $one.account.isAuthenticated\
  $attrs.product.url\
  $attrs.product\
  $attrs?.product?.characteristics\
  $one.wishlist.addToWishlist($attrs.product.id, 1)\
  (c, index) in $attrs?.product?.characteristics\
  ,catalogIndex:
  ,count:B,selected:b,label:Fv,itemOrder:ag},klej_odksztalcalny:{value:Fw,valueCode:
  ,count:_,selected:b,label:FD,itemOrder:n},b:{value:FE,valueCode:
  ,count:ag,selected:b,label:FL,itemOrder:n},
  ,count:ag,selected:b,label:FV,itemOrder:e},
  ,count:cU,selected:b,label:FQ,itemOrder:V},
  ,count:e,selected:b,label:EL,itemOrder:cU},quickmix:{value:EM,valueCode:
  ,count:e,selected:b,label:FF,itemOrder:ag},e:{value:FG,valueCode:
  ,count:e,selected:b,label:FN,itemOrder:k},
  ,count

In [183]:

import requests, re, json

headers_web = {"User-Agent": "Mozilla/5.0"}
r = requests.get("https://www.bechcicki.pl/chemia-budowlana/kleje/kleje-do-glazury",
                 headers=headers_web, timeout=15)
html = r.text

# Wyciągnij wszystkie id-bud_ URL slugi z HTML
id_bud = re.findall(r'[a-z0-9-]+-id-bud_\d+', html)
unique_slugs = sorted(set(id_bud))
print(f"Slugi id-bud_ w HTML: {len(unique_slugs)}")
for s in unique_slugs[:10]:
    print(f"  {s}")

# Znajdź JS bundle URLs (do inspekcji API)
bundles = re.findall(r'"(/_nuxt/[^"]+\.js)"', html)
print(f"\nBundle JS pliki: {len(bundles)}")
for b in bundles[:5]:
    print(f"  {b}")


Slugi id-bud_ w HTML: 6
  -3000-k-id-bud_54548573
  -4000-k-id-bud_54548572
  lampa-podlogowa-goodhome-mantus-3-x-g9-chrom-id-bud_54548575
  lampa-wiszaca-luxolar-382-e27-bialy-id-bud_54548574
  pasek-led-polux-cob-24-v-4000-k-5-m-id-bud_54548577
  pasek-led-polux-neon-24-v-ip65-5-m-niebieski-id-bud_54548576

Bundle JS pliki: 0


In [187]:

import requests, re, gzip, io

H = {"User-Agent": "Mozilla/5.0"}

# 1. Robots + sitemap
r_rb = requests.get("https://www.bechcicki.pl/robots.txt", headers=H, timeout=8)
print("=== robots.txt ===")
print(r_rb.text[:600])

# 2. Sitemap
r_sm = requests.get("https://www.bechcicki.pl/sitemap.xml", headers=H, timeout=8)
print(f"\n=== sitemap.xml status: {r_sm.status_code} ===")
if r_sm.ok:
    print(r_sm.text[:500])


=== robots.txt ===
User-agent: *
Disallow: /registration 
Disallow: /login
Disallow: /polityka-prywatnosci
Disallow: /*filtersQuery=
Disallow: /*rows= 
Disallow: /*sortCriteria=
Disallow: /*search?
Disallow: /*feed 
Disallow: /sixthSubmenu*
Disallow: /test
Disallow: /wp-json/
Disallow: /*?utm_ 
Disallow: /tag/cennik/
Disallow: /out/ 
Disallow: /search



=== sitemap.xml status: 404 ===


In [191]:

import requests, re

H = {"User-Agent": "Mozilla/5.0"}

# Pobierz HTML i znajdź skrypty z bundle
r = requests.get("https://www.bechcicki.pl/", headers=H, timeout=10)
html = r.text

# Znajdź wszystkie _nuxt/*.js
bundles = re.findall(r'src="(/_nuxt/[^"]+\.js)"', html)
print(f"Bundles: {len(bundles)}")

# Pobierz największy bundle (likely app.js) i szukaj API
for b in bundles[:3]:
    rb = requests.get(f"https://www.bechcicki.pl{b}", headers=H, timeout=10)
    # Szukaj endpointów
    apis = re.findall(r'["\`](/[a-z][a-z0-9/_-]{4,50})["\`]', rb.text)
    product_apis = [a for a in apis if any(x in a for x in ["product","catalog","item","search","offer"])]
    if product_apis:
        print(f"\n{b} ({len(rb.text)} chars):")
        for a in sorted(set(product_apis))[:15]:
            print(f"  {a}")
        break
    
    # Szukaj baseURL
    base = re.findall(r'baseURL[:\s]*["\`]([^"\`]{5,50})["\`]', rb.text)
    if base:
        print(f"baseURL: {base[:3]}")


Bundles: 0


In [195]:

import requests, re

H = {"User-Agent": "Mozilla/5.0", "Accept": "application/json"}

# Próbne endpointy API
candidates = [
    "https://www.bechcicki.pl/api/catalog/products?category=kleje-do-glazury",
    "https://www.bechcicki.pl/api/products?category=kleje-do-glazury",
    "https://www.bechcicki.pl/api/v1/products",
    "https://www.bechcicki.pl/api/items",
    "https://api.bechcicki.pl/products",
]

for url in candidates:
    try:
        r = requests.get(url, headers=H, timeout=5)
        print(f"{r.status_code}  {url}")
        if r.ok and 'json' in r.headers.get('content-type',''):
            print(f"  JSON! Pierwsze 200: {r.text[:200]}")
    except Exception as e:
        print(f"ERR  {url}  {e}")

# Sprawdź HTML strony produktu — szukaj struktury data w __NUXT__
# Znajdź produkt po EAN — próbuj przez URL
test_ean = "5900089112241"
r2 = requests.get(f"https://www.bechcicki.pl/search?phrase={test_ean}", headers={**H, "Accept":"text/html"}, timeout=8, allow_redirects=True)
print(f"\nSearch phrase EAN → {r2.status_code}  URL: {r2.url}")
# Szukaj id-bud_ w HTML
slugs = re.findall(r'[a-z0-9-]+-id-bud_\d+', r2.text)
print(f"Slugi: {set(slugs)}")


404  https://www.bechcicki.pl/api/catalog/products?category=kleje-do-glazury


404  https://www.bechcicki.pl/api/products?category=kleje-do-glazury


404  https://www.bechcicki.pl/api/v1/products


404  https://www.bechcicki.pl/api/items
ERR  https://api.bechcicki.pl/products  HTTPSConnectionPool(host='api.bechcicki.pl', port=443): Max retries exceeded with url: /products (Caused by NameResolutionError("HTTPSConnection(host='api.bechcicki.pl', port=443): Failed to resolve 'api.bechcicki.pl' ([Errno -2] Name or service not known)"))



Search phrase EAN → 404  URL: https://www.bechcicki.pl/search?phrase=5900089112241
Slugi: {'pasek-led-polux-neon-24-v-ip65-5-m-niebieski-id-bud_54548576', 'lampa-wiszaca-luxolar-382-e27-bialy-id-bud_54548574', 'lampa-podlogowa-goodhome-mantus-3-x-g9-chrom-id-bud_54548575', '-4000-k-id-bud_54548572', '-3000-k-id-bud_54548573', 'pasek-led-polux-cob-24-v-4000-k-5-m-id-bud_54548577'}


In [199]:

import requests, re, json

H = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
     "Accept-Language": "pl-PL,pl;q=0.9",
     "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"}

# Próba 1: SKU jako część URL
for url in [
    "https://www.bechcicki.pl/klej-do-gresu-ceresit-cm12-plus-id-bud_P-0029063",
    "https://www.bechcicki.pl/-id-bud_0033978",
    "https://www.bechcicki.pl/p/0029063",
    "https://www.bechcicki.pl/produkt/P-0029063",
]:
    r = requests.get(url, headers=H, timeout=6, allow_redirects=True)
    print(f"{r.status_code}  {r.url[:80]}")

# Próba 2: Kategoria z dużą liczbą produktów — wyciągnij dane z __NUXT__
print("\n=== Wyciągam produkty z __NUXT__ kategorii ===")
r = requests.get("https://www.bechcicki.pl/chemia-budowlana/kleje/kleje-do-glazury",
                 headers=H, timeout=15)

# Szukaj JSON w HTML — bechcicki może mieć __INITIAL_STATE__ lub podobne
patterns = [
    r'window\.__(?:INITIAL_STATE|DATA|PRODUCTS|STATE)__\s*=\s*(\{.*?\});',
    r'"products"\s*:\s*(\[.*?\])',
    r'"items"\s*:\s*(\[.*?\])',
    r'"catalog"\s*:\s*(\{.*?\})',
]

for p in patterns:
    match = re.search(p, r.text, re.DOTALL)
    if match:
        print(f"Znaleziono pattern: {p[:40]}")
        print(f"Dane (pierwsze 300): {match.group(1)[:300]}")
        break

# Wyciągnij wszystkie ciągi wyglądające jak nazwy produktów (all caps, 10-80 znaków)
# Z __NUXT__ — szukaj par "name":"..."
names_in_html = re.findall(r'"name"\s*:\s*"([^"]{10,100})"', r.text)
print(f"\n'name' w HTML ({len(names_in_html)} szt.):")
for n in names_in_html[:15]:
    if any(c.isalpha() for c in n):
        print(f"  {n}")


500  https://www.bechcicki.pl/klej-do-gresu-ceresit-cm12-plus-id-bud_P-0029063


500  https://www.bechcicki.pl/-id-bud_0033978


404  https://www.bechcicki.pl/p/0029063


404  https://www.bechcicki.pl/produkt/P-0029063

=== Wyciągam produkty z __NUXT__ kategorii ===



'name' w HTML (2 szt.):
  Chemia budowlana
  Kleje do glazury


In [203]:

import json, requests

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE  = f"https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production"
HEADS = {"Authorization": f"Bearer {TOKEN}"}

def groq(q): return requests.get(BASE, params={"query":q}, headers=HEADS).json()["result"]

# 1. JSONL — sprawdź pełną strukturę pól
print("=== JSONL pola ===")
all_keys = set()
samples_with_desc = 0
with open("/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/bechcicki_products_all.jsonl") as f:
    for i, line in enumerate(f):
        rec = json.loads(line)
        all_keys |= set(rec.keys())
        if rec.get("description") or rec.get("short_description") or rec.get("specs"):
            samples_with_desc += 1
        if i >= 500: break

print(f"Pola w JSONL: {sorted(all_keys)}")
print(f"Rekordy z description/specs (spośród 500): {samples_with_desc}")

# 2. Sanity — sprawdź co mają produkty
print("\n=== Próbka Sanity produktów ===")
sample = groq('*[_type=="product"][0..4]{_id,name,sku,shortDescription,description,"hasSpec":defined(technicalSpec),"specCount":count(technicalSpec)}')
for p in sample:
    print(f"  {p['sku']:15s} | name: {p['name'][:50]:50s} | desc: {bool(p.get('shortDescription'))} | spec: {p.get('specCount',0)}")

# 3. Statystyki — ile ma nazwy, ile opis, ile spec
stats = groq('''*[_type=="product"]{
  "hasName": string::length(name)>5,
  "hasDesc": defined(shortDescription) && string::length(shortDescription)>10,
  "hasSpec": count(technicalSpec)>0
}''')
print(f"\nŁącznie produktów: {len(stats)}")
print(f"  Ma nazwę (>5 zn.): {sum(1 for s in stats if s.get('hasName'))}")
print(f"  Ma opis krótki:    {sum(1 for s in stats if s.get('hasDesc'))}")
print(f"  Ma technicalSpec:  {sum(1 for s in stats if s.get('hasSpec'))}")


=== JSONL pola ===
Pola w JSONL: ['_page', 'alt_id', 'brand', 'category_leaf', 'category_main', 'category_sub', 'ean', 'manufacturer_index', 'name', 'sku', 'url']
Rekordy z description/specs (spośród 500): 0

=== Próbka Sanity produktów ===
  CRS-CT16-10L    | name: Grunt głęboko penetrujący Ceresit CT 16 10L        | desc: True | spec: None
  CRS-CL50-15     | name: Elastyczna hydroizolacja Ceresit CL 50 15kg        | desc: True | spec: None
  WEB-THERM-EX-25 | name: Zaprawa klejąca weber.therm extra 25kg             | desc: True | spec: None
  SWP-EPS70-031-15 | name: Styropian grafitowy Swisspor EPS 70-031 Fasada 15c | desc: True | spec: None
  KNF-GB-25       | name: Tynk gipsowy Knauf Goldband 25kg                   | desc: True | spec: None



Łącznie produktów: 15921
  Ma nazwę (>5 zn.): 15918
  Ma opis krótki:    982
  Ma technicalSpec:  175


In [207]:

import subprocess, sys

# Zainstaluj playwright
result = subprocess.run([sys.executable, "-m", "pip", "install", "playwright", "-q"], 
                       capture_output=True, text=True)
print("pip playwright:", result.returncode, result.stderr[-200:] if result.stderr else "OK")

result2 = subprocess.run([sys.executable, "-m", "playwright", "install", "chromium", "--with-deps"],
                        capture_output=True, text=True, timeout=300)
print("chromium install:", result2.returncode)
print(result2.stdout[-300:] if result2.stdout else "")
print(result2.stderr[-200:] if result2.stderr else "")


pip playwright: 0 OK


chromium install: 1
Installing dependencies...
Switching to root user to install dependencies...
Failed to install browsers
Error: Installation process exited with code: 1

sudo: a terminal is required to read the password; either use the -S option to read from standard input or configure an askpass helper
sudo: a password is required



In [211]:

import asyncio, json, re
import pyppeteer

async def test_scrape():
    browser = await pyppeteer.launch(
        headless=True,
        args=['--no-sandbox','--disable-setuid-sandbox','--disable-dev-shm-usage',
              '--disable-gpu','--disable-web-security']
    )
    page = await browser.newPage()
    await page.setUserAgent('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    await page.setExtraHTTPHeaders({'Accept-Language': 'pl-PL,pl;q=0.9'})

    print("Nawiguję do kategorii farby-elewacyjne-akrylowe...")
    await page.goto('https://www.bechcicki.pl/farby-i-rozpuszczalniki/farby-elewacyjne/farby-elewacyjne-akrylowe',
                    {'waitUntil': 'networkidle0', 'timeout': 30000})
    await asyncio.sleep(3)

    # 1. Sprawdź tytuł strony
    title = await page.title()
    print(f"Tytuł: {title}")

    # 2. Wyciągnij produkty przez DOM
    products = await page.evaluate("""
    () => {
        const results = [];
        // Różne selektory dla kart produktów
        const candidateSelectors = [
            '[class*="product"]',
            '[class*="tile"]',
            '[class*="item"]',
            '[data-product]'
        ];
        
        // Znajdź wszystkie linki z id-bud_ w href
        const productLinks = Array.from(document.querySelectorAll('a[href*="id-bud_"]'));
        
        productLinks.forEach(a => {
            const href = a.getAttribute('href');
            const text = a.textContent.trim();
            if (href && text.length > 3 && text.length < 200) {
                results.push({ url: href, name: text });
            }
        });
        
        // Deduplikuj po URL
        const seen = new Set();
        return results.filter(r => {
            if (seen.has(r.url)) return false;
            seen.add(r.url);
            return true;
        });
    }
    """)
    
    print(f"\nLinki produktów przez DOM: {len(products)}")
    for p in products[:5]:
        print(f"  {p['url'][:70]}  |  {p['name'][:50]}")
    
    # 3. Wyciągnij wszystkie stringi z __NUXT__ (product names)
    nuxt_names = await page.evaluate("""
    () => {
        if (!window.__NUXT__) return {found: false};
        
        // Szukaj struktury products/items w danych
        const results = [];
        
        function traverse(obj, depth) {
            if (depth > 10 || !obj) return;
            if (typeof obj === 'string') return;
            if (Array.isArray(obj)) {
                // Sprawdź czy array produktów (mają name + url)
                if (obj.length >= 3) {
                    const first = obj[0];
                    if (first && typeof first === 'object' && 
                        (first.name || first.title) && first.url) {
                        obj.slice(0,100).forEach(item => {
                            if (item && item.name) {
                                results.push({
                                    name: item.name,
                                    url: item.url || '',
                                    ean: String(item.ean || item.barcode || ''),
                                    sku: String(item.sku || item.symbol || item.index || ''),
                                    shortDescription: item.shortDescription || item.lead || ''
                                });
                            }
                        });
                        return;
                    }
                }
                obj.forEach(i => traverse(i, depth+1));
            } else if (typeof obj === 'object') {
                Object.values(obj).forEach(v => traverse(v, depth+1));
            }
        }
        
        traverse(window.__NUXT__, 0);
        
        // Deduplikuj
        const seen = new Set();
        const unique = results.filter(r => {
            if (seen.has(r.url || r.name)) return false;
            seen.add(r.url || r.name);
            return true;
        });
        
        return { found: true, count: unique.length, products: unique.slice(0, 10) };
    }
    """)
    
    print(f"\n__NUXT__ produkty: {nuxt_names.get('count',0)}")
    for p in nuxt_names.get('products', [])[:5]:
        print(f"  {p.get('ean',''):15s} | {p.get('name','')[:60]}")
    
    # 4. Zrób screenshot żeby zobaczyć co jest na stronie
    await page.screenshot({'path': '/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/test_bechcicki.png'})
    print("\nScreenshot zapisany")
    
    await browser.close()
    return {"dom_products": len(products), "nuxt_count": nuxt_names.get('count',0)}

result = await test_scrape()
print(f"\nWYNIK: {result}")


[INFO] Starting Chromium download.


  0%|          | 0.00/183M [00:00<?, ?b/s]

  4%|▍         | 6.90M/183M [00:00<00:02, 68.9Mb/s]

  9%|▉         | 16.1M/183M [00:00<00:02, 82.4Mb/s]

 14%|█▍        | 25.3M/183M [00:00<00:01, 86.4Mb/s]

...


[INFO] Beginning extraction


[INFO] Chromium extracted to: /data/home/skywork/.local/share/pyppeteer/local-chromium/1181205


Nawiguję do kategorii farby-elewacyjne-akrylowe...


Tytuł: Farby elewacyjne akrylowe

Linki produktów przez DOM: 0

__NUXT__ produkty: 0

Screenshot zapisany



WYNIK: {'dom_products': 0, 'nuxt_count': 0}


In [215]:

import asyncio, json, re, time, os

async def test_with_cookie():
    import pyppeteer
    browser = await pyppeteer.launch(
        headless=True,
        args=['--no-sandbox','--disable-setuid-sandbox','--disable-dev-shm-usage','--disable-gpu']
    )
    page = await browser.newPage()
    await page.setUserAgent('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    await page.setViewport({'width': 1440, 'height': 900})

    # --- Nawigacja do kategorii ---
    url = 'https://www.bechcicki.pl/farby-i-rozpuszczalniki/farby-elewacyjne/farby-elewacyjne-akrylowe'
    await page.goto(url, {'waitUntil': 'networkidle0', 'timeout': 30000})
    await asyncio.sleep(2)

    # --- Zamknij baner cookie ---
    try:
        # "Zezwól na wszystkie" button
        btns = await page.querySelectorAll('button')
        for btn in btns:
            txt = await page.evaluate('(el) => el.textContent', btn)
            if 'Zezwól' in (txt or '') or 'wszystkie' in (txt or '').lower():
                await btn.click()
                print(f"Kliknięto przycisk: '{txt.strip()}'")
                await asyncio.sleep(1.5)
                break
    except Exception as e:
        print(f"Cookie btn error: {e}")

    # --- Zbadaj strukturę DOM ---
    dom_info = await page.evaluate("""
    () => {
        // Zbierz ALL klasy z elementów które mogą być kartami produktów
        const all = Array.from(document.querySelectorAll('*'));
        const classes = new Set();
        const eanElements = [];
        
        all.forEach(el => {
            const txt = el.textContent?.trim() || '';
            // Znajdź elementy z EAN
            if (/EAN:\s*\d{8,}/.test(txt) && el.children.length < 5) {
                eanElements.push({
                    tag: el.tagName,
                    class: el.className?.substring(0, 80) || '',
                    text: txt.substring(0, 100),
                    parentClass: el.parentElement?.className?.substring(0, 80) || ''
                });
            }
        });
        
        // Linki z id-bud_
        const prodLinks = Array.from(document.querySelectorAll('a[href*="id-bud_"]'))
            .map(a => ({
                href: a.getAttribute('href'),
                text: a.textContent.trim().substring(0, 80),
                class: a.className?.substring(0, 60) || '',
                parentClass: a.parentElement?.className?.substring(0, 60) || ''
            }))
            .filter((v,i,a) => a.findIndex(x=>x.href===v.href)===i)
            .slice(0, 10);
        
        return { eanElements: eanElements.slice(0, 5), prodLinks, totalLinks: prodLinks.length };
    }
    """)
    
    print(f"\nLinki produktów (id-bud_): {dom_info['totalLinks']}")
    for l in dom_info['prodLinks'][:5]:
        print(f"  href: {l['href'][:70]}")
        print(f"  text: {l['text'][:60]}")
        print(f"  class: {l['class']}")
        print(f"  parentClass: {l['parentClass']}")
        print()
    
    print(f"\nElementy z EAN:")
    for e in dom_info['eanElements'][:3]:
        print(f"  tag:{e['tag']} class:{e['class']}")
        print(f"  txt:{e['text']}")
        print(f"  parentClass:{e['parentClass']}")
        print()
    
    # Screenshot po zamknięciu cookie
    await page.screenshot({'path': '/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/test2.png',
                           'fullPage': True})
    
    await browser.close()

await test_with_cookie()


Cookie btn error: Node is either not visible or not an HTMLElement

Linki produktów (id-bud_): 0

Elementy z EAN:
  tag:HTML class:
  txt:Farby elewacyjne akrylowe{"@context":"https://schema.org","@type":"BreadcrumbList","itemListElement"
  parentClass:

  tag:DIV class:
  txt:PLATFORMA SPRZEDAŻOWA DLA KLIENTÓW B2B
  

  
    
      
        
          Ponad 100 indywidualnyc
  parentClass:farby-i-rozpuszczalniki-farby-elewacyjne-farby-elewacyjne-akrylowe

  tag:DIV class:
  txt:PLATFORMA SPRZEDAŻOWA DLA KLIENTÓW B2B
  

  
    
      
        
          Ponad 100 indywidualnyc
  parentClass:



In [219]:

import asyncio, json, re

async def test_manufacturer():
    import pyppeteer
    browser = await pyppeteer.launch(
        headless=True,
        args=['--no-sandbox','--disable-setuid-sandbox','--disable-dev-shm-usage','--disable-gpu']
    )
    page = await browser.newPage()
    await page.setUserAgent('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    await page.setViewport({'width': 1440, 'height': 900})

    # Test 1: Ceresit CT 16 na ceresit.pl
    test_name = "Grunt głęboko penetrujący Ceresit CT 16"
    
    # Najpierw sprawdź Google
    import urllib.parse
    q = urllib.parse.quote(f"{test_name} opis parametry techniczne site:ceresit.pl OR site:henkel.com")
    r_url = f"https://www.google.pl/search?q={q}&hl=pl"
    
    await page.goto(r_url, {'waitUntil': 'networkidle0', 'timeout': 20000})
    await asyncio.sleep(1)

    # Pobierz wyniki Google
    results = await page.evaluate("""
    () => {
        const items = [];
        document.querySelectorAll('div.g, [data-hveid]').forEach(r => {
            const title = r.querySelector('h3')?.textContent?.trim() || '';
            const url = r.querySelector('a')?.href || '';
            const snippet = r.querySelector('[class*="VwiC3b"], [class*="snippet"], .IsZvec')
                              ?.textContent?.trim() || '';
            if (title && url && url.startsWith('http')) {
                items.push({title, url, snippet: snippet.substring(0,200)});
            }
        });
        return items.slice(0, 5);
    }
    """)
    
    print(f"Google wyniki dla '{test_name}':")
    for r in results:
        print(f"  [{r['title'][:60]}]")
        print(f"   {r['url'][:80]}")
        print(f"   {r['snippet'][:120]}")
        print()

    # Test 2: Bezpośrednio ceresit.pl
    await page.goto('https://www.ceresit.pl/produkty/ct16', {'waitUntil': 'networkidle0', 'timeout': 20000})
    await asyncio.sleep(2)
    title2 = await page.title()
    
    # Wyciągnij opis i parametry
    data = await page.evaluate("""
    () => {
        const desc = document.querySelector(
            '[class*="description"], [class*="product-text"], .product__description, main p'
        )?.textContent?.trim() || '';
        
        const specs = [];
        document.querySelectorAll('table tr, [class*="spec"] [class*="row"]').forEach(row => {
            const cells = row.querySelectorAll('td, th');
            if (cells.length >= 2) {
                specs.push({
                    label: cells[0].textContent.trim(),
                    value: cells[1].textContent.trim()
                });
            }
        });
        
        return { desc: desc.substring(0, 300), specs: specs.slice(0, 10), url: window.location.href };
    }
    """)
    print(f"\nCeresit.pl tytuł: {title2}")
    print(f"Opis: {data['desc'][:200]}")
    print(f"Specs: {data['specs'][:3]}")

    await browser.close()

await test_manufacturer()


Google wyniki dla 'Grunt głęboko penetrujący Ceresit CT 16':



Ceresit.pl tytuł: 404 - Ceresit
Opis: ZNAJDŹ SKLEP
Specs: []


In [223]:

import asyncio, json, re, urllib.parse

async def test_public_sites():
    import pyppeteer
    browser = await pyppeteer.launch(
        headless=True,
        args=['--no-sandbox','--disable-setuid-sandbox','--disable-dev-shm-usage','--disable-gpu']
    )
    page = await browser.newPage()
    await page.setUserAgent('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0')
    await page.setViewport({'width': 1440, 'height': 900})

    results = {}
    
    # Test 1: LeRoyMerlin.pl - szukaj Ceresit CT16
    q = urllib.parse.quote("Grunt głęboko penetrujący Ceresit CT 16")
    lrm_url = f"https://www.leroymerlin.pl/szukaj/{q}"
    
    await page.goto(lrm_url, {'waitUntil': 'networkidle0', 'timeout': 25000})
    await asyncio.sleep(2)
    
    # Zamknij cookie jeśli jest
    try:
        for btn in await page.querySelectorAll('button'):
            txt = await page.evaluate('(el) => el.textContent', btn)
            if any(w in (txt or '') for w in ['Akceptuj', 'akceptuj', 'Zgadzam', 'Zezwól', 'wszystkie']):
                await btn.click()
                await asyncio.sleep(1)
                break
    except: pass
    
    lrm_data = await page.evaluate("""
    () => {
        const items = [];
        // LeRoyMerlin product tile selectors
        const tiles = document.querySelectorAll('[data-testid*="product"], [class*="ProductTile"], [class*="product-tile"], article');
        tiles.forEach(t => {
            const nameEl = t.querySelector('h2, h3, [class*="name"], [class*="title"]');
            const urlEl = t.querySelector('a');
            if (nameEl && urlEl) {
                items.push({
                    name: nameEl.textContent.trim().substring(0, 80),
                    url: urlEl.href
                });
            }
        });
        return { count: items.length, items: items.slice(0, 3), title: document.title };
    }
    """)
    print(f"LeRoyMerlin: title={lrm_data['title'][:50]} count={lrm_data['count']}")
    for it in lrm_data['items']:
        print(f"  {it['name']}")
        print(f"  {it['url'][:80]}")
    results['lrm'] = lrm_data

    # Test 2: Ceresit.pl przez inny URL format
    for test_url in [
        'https://www.ceresit.pl/produkty/tynki-i-zaprawy',
        'https://www.ceresit.pl/produkty',
    ]:
        await page.goto(test_url, {'waitUntil': 'networkidle0', 'timeout': 20000})
        await asyncio.sleep(2)
        t = await page.title()
        prod_count = await page.evaluate("() => document.querySelectorAll('[class*=\"product\"], article').length")
        print(f"\nCeresit [{test_url[-30:]}]: {t[:50]} | produktów: {prod_count}")
        if prod_count > 0:
            break
    
    # Test 3: Atlas - atlasbudowlany.pl
    await page.goto('https://www.atlasbudowlany.pl/produkty', {'waitUntil': 'networkidle0', 'timeout': 20000})
    await asyncio.sleep(2)
    atlas_t = await page.title()
    atlas_prod = await page.evaluate("""
    () => {
        const links = Array.from(document.querySelectorAll('a[href*="produkt"]'));
        return {
            title: document.title,
            links: links.slice(0,3).map(a => ({text: a.textContent.trim().substring(0,60), href: a.href.substring(0,80)}))
        };
    }
    """)
    print(f"\nAtlas [{atlas_t[:40]}]: {len(atlas_prod['links'])} linków")
    for l in atlas_prod['links']:
        print(f"  {l['text']} → {l['href']}")

    await browser.close()
    return results

await test_public_sites()


LeRoyMerlin: title=leroymerlin.pl count=0



Ceresit [it.pl/produkty/tynki-i-zaprawy]: 404 - Ceresit | produktów: 7


PageError: net::ERR_NAME_NOT_RESOLVED at https://www.atlasbudowlany.pl/produkty

In [227]:

import requests

# Sprawdź które domeny są dostępne przez zwykłe requests (różne proxy niż pyppeteer)
import socket
domains = [
    "www.castorama.pl",
    "www.leroymerlin.pl", 
    "www.obi.pl",
    "www.knauf.pl",
    "www.weber.com.pl",
    "pl.mapei.com",
    "www.baumit.com",
    "www.swisspor.pl",
    "www.isover.pl",
]

H = {"User-Agent": "Mozilla/5.0"}
for domain in domains:
    try:
        ip = socket.gethostbyname(domain)
        r = requests.get(f"https://{domain}", headers=H, timeout=5, allow_redirects=True)
        print(f"✅ {domain:30s}  IP:{ip}  status:{r.status_code}")
    except socket.gaierror:
        print(f"❌ DNS {domain:30s}")
    except Exception as e:
        print(f"⚠️  {domain:30s}  {str(e)[:40]}")


✅ www.castorama.pl                IP:13.226.238.22  status:503
✅ www.leroymerlin.pl              IP:151.101.130.114  status:403


✅ www.obi.pl                      IP:3.167.88.119  status:404


✅ www.knauf.pl                    IP:104.18.39.203  status:200


✅ www.weber.com.pl                IP:85.128.128.104  status:200
❌ DNS pl.mapei.com                  


✅ www.baumit.com                  IP:13.248.209.114  status:202


✅ www.swisspor.pl                 IP:81.21.192.39  status:200


✅ www.isover.pl                   IP:104.18.28.123  status:403


In [231]:
import os, json, time, re, requests, traceback, sys
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
from pathlib import Path

BASE_DIR = Path('/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud')
OUT_JSONL = BASE_DIR / 'products_scraped.jsonl'
LOG_FILE = BASE_DIR / 'scrape_log.txt'
PROGRESS_FILE = BASE_DIR / 'PROGRESS.md'

SANITY_PROJECT = 'nzcwegq7'
SANITY_DATASET = 'production'
SANITY_TOKEN = 'skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J'
QUERY_URL = f'https://{SANITY_PROJECT}.api.sanity.io/v2023-08-01/data/query/{SANITY_DATASET}'
MUTATE_URL = f'https://{SANITY_PROJECT}.api.sanity.io/v2023-08-01/data/mutate/{SANITY_DATASET}'
HEADERS = {
    'Authorization': f'Bearer {SANITY_TOKEN}',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36',
    'Accept-Language': 'pl-PL,pl;q=0.9,en;q=0.8',
}
BRAND_DOMAINS = {
    'knauf': 'https://www.knauf.pl/pl/search?q={q}',
    'weber': 'https://www.weber.com.pl/pl/wyszukiwarka?search_api_fulltext={q}',
    'swisspor': 'https://www.swisspor.pl/pl/szukaj?keys={q}',
    'baumit': 'https://www.baumit.com/pl-PL/search?query={q}',
}
BASE_DIR.mkdir(parents=True, exist_ok=True)

def append_text(path: Path, text: str):
    with open(path, 'a', encoding='utf-8') as f:
        f.write(text)

def log(msg: str):
    stamp = time.strftime('%Y-%m-%d %H:%M:%S')
    line = f'[{stamp}] {msg}'
    print(line)
    append_text(LOG_FILE, line + '\n')

def progress(msg: str):
    stamp = time.strftime('%Y-%m-%d %H:%M')
    append_text(PROGRESS_FILE, f'[{stamp}] {msg}\n')

def sanity_query(groq: str):
    r = requests.get(QUERY_URL, params={'query': groq}, headers=HEADERS, timeout=60)
    r.raise_for_status()
    return r.json()['result']

def sanity_mutate(mutations):
    payload = {'mutations': mutations}
    r = requests.post(MUTATE_URL, params={'visibility': 'sync'}, json=payload, headers={**HEADERS, 'Content-Type': 'application/json'}, timeout=120)
    r.raise_for_status()
    return r.json()

def clean_text(text):
    if not text:
        return ''
    return re.sub(r'\s+', ' ', text).strip()

def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', clean_text(text)) if s.strip()]

def make_short_description(desc, name):
    sents = split_sentences(desc)
    if sents:
        return ' '.join(sents[:2])[:420].strip()
    return f'{name} to produkt budowlany do profesjonalnych zastosowań.'

def extract_specs_from_tables(soup):
    specs, seen = [], set()
    for table in soup.select('table'):
        for row in table.select('tr'):
            cells = row.find_all(['th', 'td'])
            if len(cells) >= 2:
                label = clean_text(cells[0].get_text(' ', strip=True))
                value = clean_text(cells[1].get_text(' ', strip=True))
                if label and value:
                    key = (label.lower(), value.lower())
                    if key not in seen:
                        seen.add(key)
                        specs.append({'label': label[:120], 'value': value[:500]})
    return specs[:30]

def extract_description(soup):
    selectors = ['[class*="description"]','[id*="description"]','.product-detail__description','.field--name-body','.product__description','.content-description','article','main']
    chunks = []
    for sel in selectors:
        for el in soup.select(sel):
            txt = clean_text(el.get_text(' ', strip=True))
            if len(txt) >= 120:
                chunks.append(txt)
        if chunks:
            break
    if not chunks:
        paragraphs = [clean_text(p.get_text(' ', strip=True)) for p in soup.select('p')]
        chunks = [p for p in paragraphs if len(p) >= 80]
    return ' '.join(chunks[:6])[:4000]

def fetch_url(url):
    r = requests.get(url, headers=HEADERS, timeout=25)
    r.raise_for_status()
    return r.text, r.url, r.status_code

def scrape_brand_site(product):
    brand = (product.get('brand') or '').lower()
    name = product.get('name') or ''
    for key, tmpl in BRAND_DOMAINS.items():
        if key in brand:
            search_url = tmpl.format(q=quote_plus(name))
            try:
                html, final_url, status = fetch_url(search_url)
                soup = BeautifulSoup(html, 'html.parser')
                desc = extract_description(soup)
                specs = extract_specs_from_tables(soup)
                if desc or specs:
                    return {'shortDescription': make_short_description(desc, name), 'description': desc or make_short_description('', name), 'technicalSpec': specs, 'source': 'scraped'}
            except Exception:
                return None
    return None

try:
    groq = '*[_type=="product" && !defined(shortDescription)][0...500]{_id, name, sku, "brand": brand->name, "catSlug": category->slug.current}'
    products = sanity_query(groq)
    if LOG_FILE.exists():
        LOG_FILE.unlink()
    if OUT_JSONL.exists():
        OUT_JSONL.unlink()
    log(f'Pobrano z Sanity {len(products)} produktów bez shortDescription.')
    progress(f'START: scraping opisów produktów z Sanity CMS MediaBud dla transzy {len(products)} rekordów (max 500).')
    results = []
    mutations = []
    processed = scraped = 0
    ai_needed = []
    for p in products:
        processed += 1
        rec = {'sanity_id': p['_id'], 'shortDescription': '', 'description': '', 'technicalSpec': [], 'source': ''}
        data = scrape_brand_site(p)
        if data and (data.get('description') or data.get('technicalSpec')):
            rec.update(data)
            scraped += 1
            mutations.append({'patch': {'id': p['_id'], 'set': {'shortDescription': rec['shortDescription'], 'description': rec['description'], 'technicalSpec': rec['technicalSpec']}}})
        else:
            ai_needed.append(p)
        append_text(OUT_JSONL, json.dumps(rec, ensure_ascii=False) + '\n')
        if processed % 50 == 0:
            log(f'Postęp: {processed}/{len(products)}; scraped={scraped}; ai_pending={len(ai_needed)}')
    log(f'Etap scraping zakończony: scraped={scraped}, ai_pending={len(ai_needed)}')
    ai_done = 0
    if ai_needed:
        sys.path.insert(0, '/app/skills/common/scripts')
        from batch_llm import batch_llm
        prompts = [f"Napisz profesjonalny krótki opis (2-3 zdania) produktu budowlanego: '{p['name']}', marka: '{p.get('brand') or ''}', kategoria: '{p.get('catSlug') or ''}'. Opis po polsku, bez cen, bez sztucznego upychania fraz. Tylko merytoryczne informacje o produkcie." for p in ai_needed]
        ai_results = await batch_llm(prompts, system='Jesteś ekspertem od materiałów budowlanych. Piszesz krótkie, rzeczowe opisy produktów po polsku.')
        lines = OUT_JSONL.read_text(encoding='utf-8').splitlines()
        line_map = {}
        for idx, line in enumerate(lines):
            obj = json.loads(line)
            line_map[obj['sanity_id']] = idx
        for p, text in zip(ai_needed, ai_results):
            desc = clean_text(text or '')
            short_desc = make_short_description(desc, p['name'])
            obj = {'sanity_id': p['_id'], 'shortDescription': short_desc, 'description': desc, 'technicalSpec': [], 'source': 'ai_generated'}
            lines[line_map[p['_id']]] = json.dumps(obj, ensure_ascii=False)
            mutations.append({'patch': {'id': p['_id'], 'set': {'shortDescription': short_desc, 'description': desc, 'technicalSpec': []}}})
            ai_done += 1
        OUT_JSONL.write_text('\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')
        log(f'Etap AI zakończony: ai_generated={ai_done}')
    patched_batches = 0
    for i in range(0, len(mutations), 100):
        batch = mutations[i:i+100]
        sanity_mutate(batch)
        patched_batches += 1
        log(f'PATCH batch {patched_batches}: {len(batch)} rekordów')
    progress(f'DONE: scraping opisów i parametrów produktów — przetworzono {processed}, scraped={scraped}, ai_generated={ai_done}, zapisano {OUT_JSONL.name}, wykonano PATCH do Sanity w {patched_batches} batchach.')
    print(json.dumps({'processed': processed, 'scraped': scraped, 'ai_generated': ai_done, 'mutations': len(mutations), 'jsonl_path': str(OUT_JSONL), 'log_path': str(LOG_FILE), 'progress_path': str(PROGRESS_FILE)}, ensure_ascii=False, indent=2))
except Exception as e:
    err = traceback.format_exc()
    print(err)
    progress(f'ERROR: pipeline scrapingu produktów przerwany — {type(e).__name__}: {e}')
    raise

[2026-05-29 15:11:56] Pobrano z Sanity 500 produktów bez shortDescription.


[2026-05-29 15:11:58] Postęp: 50/500; scraped=1; ai_pending=49


[2026-05-29 15:11:59] Postęp: 100/500; scraped=1; ai_pending=99


[2026-05-29 15:12:00] Postęp: 150/500; scraped=1; ai_pending=149


[2026-05-29 15:12:01] Postęp: 200/500; scraped=1; ai_pending=199


...

[2026-05-29 15:15:21] PATCH batch 1: 100 rekordów


[2026-05-29 15:15:22] PATCH batch 2: 100 rekordów


[2026-05-29 15:15:24] PATCH batch 3: 100 rekordów


[2026-05-29 15:15:25] PATCH batch 4: 100 rekordów


[2026-05-29 15:15:27] PATCH batch 5: 100 rekordów
{
  "processed": 500,
  "scraped": 3,
  "ai_generated": 497,
  "mutations": 500,
  "jsonl_path": "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/products_scraped.jsonl",
  "log_path": "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/scrape_log.txt",
  "progress_path": "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/PROGRESS.md"
}


In [4]:

import requests, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production"
H = {"Authorization": f"Bearer {TOKEN}"}

# Pobierz produkty które właśnie zaktualizowano (mają shortDescription) — różne kategorie
query = '''*[_type=="product" && defined(shortDescription) && string::length(shortDescription) > 20][0..14] | order(_updatedAt desc) {
  _id, name, brand, sku,
  "catSlug": category->slug.current,
  shortDescription,
  description,
  "specCount": count(technicalSpec),
  "specs": technicalSpec[0..2]
}'''

r = requests.get(BASE, params={"query": query}, headers=H)
products = r.json()["result"]

print(f"Próbka {len(products)} produktów zaktualizowanych AI:\n")
for p in products:
    print(f"{'='*70}")
    print(f"Nazwa:    {p['name']}")
    print(f"Marka:    {p.get('brand','?')}  |  Kategoria: {p.get('catSlug','?')}")
    print(f"Short:    {p.get('shortDescription','—')[:200]}")
    desc = p.get('description','')
    print(f"Desc:     {desc[:200] if desc else '—'}")
    specs = p.get('specs', [])
    print(f"Specs ({p.get('specCount',0)} szt.): {[f\"{s.get('label','?')}: {s.get('value','?')}\" for s in specs[:3]]}")
    print()


SyntaxError: f-string expression part cannot include a backslash (3202808228.py, line 29)

In [8]:

import requests, json

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE = "https://nzcwegq7.api.sanity.io/v2023-08-01/data/query/production"
H = {"Authorization": f"Bearer {TOKEN}"}

query = '''*[_type=="product" && defined(shortDescription) && string::length(shortDescription) > 20][0..14] | order(_updatedAt desc) {
  _id, name, brand, sku,
  "catSlug": category->slug.current,
  shortDescription,
  description,
  "specCount": count(technicalSpec),
  "specs": technicalSpec[0..2]
}'''

products = requests.get(BASE, params={"query": query}, headers=H).json()["result"]

print(f"Próbka {len(products)} produktów (AI-generated):\n")
for p in products:
    print("="*65)
    print(f"NAZWA:    {p['name']}")
    print(f"MARKA:    {p.get('brand','?')} | KAT: {p.get('catSlug','?')}")
    print(f"SHORT:    {(p.get('shortDescription') or '—')[:250]}")
    desc = (p.get('description') or '')[:200]
    print(f"DESC:     {desc or '—'}")
    specs = p.get('specs') or []
    spec_cnt = p.get('specCount', 0)
    spec_str = " | ".join(f"{s.get('label','?')}: {s.get('value','?')}" for s in specs[:3])
    print(f"SPECS({spec_cnt}): {spec_str or '—'}")
    print()


Próbka 15 produktów (AI-generated):

NAZWA:    Zaprawa klejąca weber.therm extra 25kg
MARKA:    Weber | KAT: kleje-do-styropianu-i-styroduru
SHORT:    Zaprawa klejąca do styropianu i wełny mineralnej w systemach ETICS. Wysoka przyczepność, odporność na warunki atmosferyczne.
DESC:     —
SPECS(None): —

NAZWA:    Tynk silikonowy weber.pas DR1 15kg
MARKA:    Weber | KAT: tynki-elewacyjne
SHORT:    Gotowy tynk elewacyjny silikonowy o strukturze drapany/kornik, odporny na zabrudzenia. Doskonały do systemów ociepleń ETICS.
DESC:     —
SPECS(None): —

NAZWA:    Wełna mineralna Rockwool Frontrock MAX E 15cm
MARKA:    Rockwool | KAT: welny-fasadowe
SHORT:    Wełna skalna do ociepleń fasad metodą ETICS. Wysoka odporność ogniowa A1, doskonałe parametry akustyczne i termoizolacyjne.
DESC:     —
SPECS(None): —

NAZWA:    Styropian grafitowy Swisspor EPS 70-031 Fasada 15cm
MARKA:    Swisspor | KAT: styropiany-fasadowe-eps
SHORT:    Styropian grafitowy EPS 70-031 do ociepleń fasad budynków. Niski ws

In [12]:

import sys, re, json, time, requests, math
from datetime import datetime

sys.path.insert(0, "/app/skills/common/scripts")
from batch_llm import batch_llm

# ─── Konfiguracja ───────────────────────────────────────────────────────────
SANITY_PROJECT = "nzcwegq7"
SANITY_DATASET = "production"
SANITY_TOKEN   = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"

QUERY_BASE = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-08-01/data/query/{SANITY_DATASET}"
MUTATE_URL = f"https://{SANITY_PROJECT}.api.sanity.io/v2023-08-01/data/mutate/{SANITY_DATASET}"

HDR_R = {"Authorization": f"Bearer {SANITY_TOKEN}"}
HDR_W = {**HDR_R, "Content-Type": "application/json"}

LOG_FILE      = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/scrape_log2.txt"
PROGRESS_FILE = "/data/workspace/919fac5a-210e-47ca-8b62-27ddea343c50/mediabud/PROGRESS.md"

BATCH_FETCH    = 500   # ile produktów z Sanity na raz
BATCH_PATCH    = 100   # ile patchy w jednym mutate
BATCH_LLM_SIZE = 50    # ile promptów do batch_llm naraz (max ~50 per call)

SYSTEM_PROMPT = "Jesteś ekspertem od materiałów budowlanych. Piszesz treści produktowe po polsku. Odpowiadasz WYŁĄCZNIE w JSON bez markdown."

# ─── Helpery ────────────────────────────────────────────────────────────────
def log(msg):
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def fetch_batch(offset):
    groq = f"""*[_type=="product"][{offset}...{offset+BATCH_FETCH}] {{
  _id, name, sku,
  "brandName": brand->name,
  "catName": category->name,
  "catSlug": category->slug.current,
  "catL1": category->parent->parent->name,
  "catL2": category->parent->name
}}"""
    r = requests.get(QUERY_BASE, params={"query": groq}, headers=HDR_R, timeout=60)
    r.raise_for_status()
    return r.json().get("result", [])

def is_bad_name(name):
    if not name:
        return True
    if re.match(r'^P-\d+$', name.strip()):
        return True
    if len(name.strip()) < 6:
        return True
    upper_count = sum(1 for c in name if c.isupper())
    alpha_count = sum(1 for c in name if c.isalpha())
    if alpha_count > 0 and upper_count / alpha_count > 0.60:
        return True
    return False

def build_prompt(p):
    name      = p.get("name", "")
    brand     = p.get("brandName") or "nieznana"
    cat_name  = p.get("catName") or ""
    cat_l1    = p.get("catL1") or ""
    cat_l2    = p.get("catL2") or ""
    return f"""Produkt: "{name}"
Marka: "{brand}"
Kategoria: "{cat_name}" (L1: {cat_l1}, L2: {cat_l2})

Wygeneruj JSON:
{{
  "shortDescription": "2-3 zdania. Czym jest produkt, do czego służy, kluczowa właściwość.",
  "description": "4-6 zdań. Pełny opis: zastosowanie, właściwości, korzyści, warunki aplikacji. Profesjonalnie, bez cen.",
  "technicalSpec": [
    {{"label": "Typ produktu", "value": "..."}},
    {{"label": "Zastosowanie", "value": "..."}},
    {{"label": "Opakowanie", "value": "..."}},
    {{"label": "Wydajność", "value": "..."}},
    {{"label": "Temperatura aplikacji", "value": "..."}}
  ]
}}"""

def parse_result(result_str):
    if not result_str:
        return None
    try:
        return json.loads(result_str)
    except Exception:
        m = re.search(r'\{.*\}', result_str, re.DOTALL)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                return None
    return None

def patch_batch(mutations):
    if not mutations:
        return 0
    payload = {"mutations": mutations, "returnIds": True}
    r = requests.post(MUTATE_URL, json=payload, headers=HDR_W,
                      params={"returnIds": "true", "visibility": "async"}, timeout=120)
    if not r.ok:
        log(f"  PATCH ERROR {r.status_code}: {r.text[:300]}")
        return 0
    return len(r.json().get("results", []))

log("=== START pipeline v2 ===")

total_updated = 0
total_skipped = 0
total_errors  = 0
offset        = 0
empty_streak  = 0  # ile razy z rzędu fetch zwrócił 0

while True:
    # 1. Pobierz transszę
    try:
        products = fetch_batch(offset)
    except Exception as e:
        log(f"  FETCH ERROR offset={offset}: {e}")
        offset += BATCH_FETCH
        total_errors += 1
        if empty_streak >= 3:
            break
        continue

    if not products:
        empty_streak += 1
        log(f"  Offset {offset}: brak produktów (streak={empty_streak})")
        if empty_streak >= 3:
            break
        offset += BATCH_FETCH
        continue
    empty_streak = 0

    log(f"Offset {offset}: pobrano {len(products)} produktów")

    # 2. Filtruj złe nazwy
    good = [p for p in products if not is_bad_name(p.get("name",""))]
    skipped_here = len(products) - len(good)
    total_skipped += skipped_here
    log(f"  Po filtracji: {len(good)} dobrych, {skipped_here} pominiętych")

    if not good:
        offset += BATCH_FETCH
        continue

    # 3. batch_llm — podziel na pod-paczki BATCH_LLM_SIZE
    all_parsed = []
    for i in range(0, len(good), BATCH_LLM_SIZE):
        sub = good[i:i+BATCH_LLM_SIZE]
        prompts = [build_prompt(p) for p in sub]
        try:
            results = await batch_llm(prompts, system=SYSTEM_PROMPT)
        except Exception as e:
            log(f"  batch_llm ERROR (offset={offset}, sub={i}): {e}")
            results = [None] * len(sub)

        for p, r in zip(sub, results):
            parsed = parse_result(r)
            if parsed:
                all_parsed.append((p["_id"], parsed))
            else:
                total_errors += 1

    log(f"  LLM sparsowanych: {len(all_parsed)} / {len(good)}")

    # 4. Przygotuj mutacje i wyślij w porcjach po BATCH_PATCH
    mutations = []
    for _id, data in all_parsed:
        short_desc = data.get("shortDescription", "")
        description= data.get("description", "")
        tech_spec  = data.get("technicalSpec", [])
        mutations.append({"patch": {"id": _id, "set": {
            "shortDescription": short_desc,
            "description": description,
            "technicalSpec": tech_spec
        }}})

    patched = 0
    for i in range(0, len(mutations), BATCH_PATCH):
        sub_muts = mutations[i:i+BATCH_PATCH]
        n = patch_batch(sub_muts)
        patched += n

    total_updated += patched
    log(f"  PATCH: {patched} dokumentów zapisanych")
    log(f"  Postęp: updated={total_updated}, skip={total_skipped}, err={total_errors}")

    offset += BATCH_FETCH

    # Krótka przerwa, żeby nie przeciążyć API
    time.sleep(0.5)

# ─── Finalne summary ────────────────────────────────────────────────────────
ts_end = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
summary = f"[{ts_end}] DONE: {total_updated} produktów zaktualizowanych ({total_skipped} skip, {total_errors} errors)"
log(summary)

with open(PROGRESS_FILE, "a", encoding="utf-8") as f:
    f.write(summary + "\n")

print("\n=== KONIEC ===")
print(summary)


[2026-05-29 15:25:05] === START pipeline v2 ===


[2026-05-29 15:25:06] Offset 0: pobrano 500 produktów
[2026-05-29 15:25:06]   Po filtracji: 435 dobrych, 65 pominiętych


Batch complete: 50/50 succeeded


Batch complete: 50/50 succeeded


Batch complete: 50/50 succeeded


...

[2026-05-29 17:23:52]   PATCH: 359 dokumentów zapisanych
[2026-05-29 17:23:52]   Postęp: updated=9477, skip=1738, err=285


[2026-05-29 17:23:53] Offset 11500: pobrano 500 produktów
[2026-05-29 17:23:53]   Po filtracji: 469 dobrych, 31 pominiętych


Batch complete: 50/50 succeeded


Batch complete: 50/50 succeeded


CancelledError: 